# New Venture Headstart 

**Goal:** Use predictive analytics to show how to curate an Opening Stock List for a new branch 

**Assumption:** Branch #106 is a new Tendri branch in Embu — it has no data in the system yet.  
We use dispensing and diagnosis patterns from *existing* Tendri branches to predict  
what Branch #106 should have on shelf from Day 1.

**The core logic:**
1. Build a disease burden profile for each existing Tendri branch from diagnosis data
2. Aggregate those profiles into one "Embu catchment" target profile
3. Weight each branch by how closely it matches that Embu profile (KNN)
4. Compute their steady-state dispensing (excluding ramp-up months)
5. Apply a penetration factor to size the opening stock conservatively
6. Output: Category · Opening Qty · Confidence · Dead Stock Risk

## Imports and Configuration

In [47]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

import pickle
import glob

# For PDF extraction
import fitz


# get data from MySQL database
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

print("All imports loaded successfully")

All imports loaded successfully


In [ ]:
# Establish a connection to the MySQL database 

# ── MySQL connection ──────────────────────────────────────────────────────────
DB_USER = os.getenv("DB_USER",   "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST",   "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

DATABASE = os.getenv("DB_NAME",   "tenri_raw")
TENRI= os.getenv("DB_NAME",   "tenri")
REPORTING = os.getenv("DB_NAME",   "reporting")



In [5]:
# Get the data from  Tenri Raw database

tenri_raw_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DATABASE}")
tenri = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{TENRI}")
reporting = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{REPORTING}")


monthly_diagnoses_aggregate = pd.read_sql_query("SELECT * FROM tenri_raw.monthly_diagnoses_aggregate", tenri_raw_engine)
monthly_dispensing_aggregate = pd.read_sql_query("SELECT * FROM tenri_raw.monthly_dispensing_aggregate", tenri_raw_engine)

In [19]:
# ── Load simulated dispensing aggregate ───────────────────────────────────────
_sim_agg_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_dispensing_aggregate_*.csv"))
 
if not _sim_agg_path:
    print("WARNING: No simulated_dispensing_aggregate_*.csv found.")
    print("Run dispensing_simulation.py first, then re-run this cell.")
else:
    sim_agg = pd.read_csv(_sim_agg_path[-1])

sim_agg.head()

,facility_id,months,product_id,new_category_name,parent_category_name,correct_therapeutic_class,total_qty_dispensed,unique_patients,avg_qty_per_patient
0,4,2024-06-01,91473,Vitamins & Supplements,Vitamins & Supplements,Vitamins & Supplements,6828,4088,1.67
1,4,2024-01-01,91473,Vitamins & Supplements,Vitamins & Supplements,Vitamins & Supplements,6819,3239,2.11
2,4,2025-01-01,91473,Vitamins & Supplements,Vitamins & Supplements,Vitamins & Supplements,6595,2748,2.40
3,4,2025-02-01,91473,Vitamins & Supplements,Vitamins & Supplements,Vitamins & Supplements,5956,3422,1.74
4,4,2023-12-01,91473,Vitamins & Supplements,Vitamins & Supplements,Vitamins & Supplements,5641,3050,1.85


In [20]:
# ── Validate schema before appending ──────────────────────────────────────
REQUIRED_COLS = [
    "facility_id", "months", "product_id", "new_category_name",
    "parent_category_name", "correct_therapeutic_class",
    "total_qty_dispensed", "unique_patients", "avg_qty_per_patient"
]
missing = [c for c in REQUIRED_COLS if c not in sim_agg.columns]
if missing:
    raise ValueError(f"Simulated data missing columns: {missing}")

# ── Confirm no product_id collision ───────────────────────────────────────
real_pids = set(monthly_dispensing_aggregate["product_id"].unique())
sim_pids  = set(sim_agg["product_id"].unique())
clash     = real_pids & sim_pids
if clash:
    raise ValueError(
        f"product_id collision: {len(clash)} IDs exist in both datasets. "
        f"Sample: {list(clash)[:5]}"
    )

In [22]:
# ── Align dtypes to match real data ───────────────────────────────────────
sim_agg["facility_id"]         = sim_agg["facility_id"].astype("int64")
sim_agg["product_id"]          = sim_agg["product_id"].astype("int64")
sim_agg["total_qty_dispensed"] = sim_agg["total_qty_dispensed"].astype("float64")
sim_agg["unique_patients"]     = sim_agg["unique_patients"].astype("int64")
sim_agg["avg_qty_per_patient"] = sim_agg["avg_qty_per_patient"].astype("float64")
sim_agg["months"]              = pd.to_datetime(sim_agg["months"]).dt.strftime("%Y-%m-%d")

In [23]:
 # ── Append ────────────────────────────────────────────────────────────────
monthly_dispensing_aggregate = pd.concat(
    [monthly_dispensing_aggregate[REQUIRED_COLS], sim_agg[REQUIRED_COLS]],
    ignore_index=True
)

print(f"monthly_dispensing_aggregate updated:")
print(f"  Total rows:  {len(monthly_dispensing_aggregate):,}")
print(f"  Categories:  {monthly_dispensing_aggregate['new_category_name'].nunique()}")
print(f"  Date range:  {monthly_dispensing_aggregate['months'].min()} → {monthly_dispensing_aggregate['months'].max()}")
print(f"\n  Category breakdown:")
for cat, n in monthly_dispensing_aggregate["new_category_name"].value_counts().items():
    is_new = cat in ["Beauty Products","Vitamins & Supplements","Body Building"]
    tag    = " ← NEW" if is_new else ""
    print(f"    {cat:<35} {n:>8,} rows{tag}")


monthly_dispensing_aggregate updated:
  Total rows:  213,081
  Categories:  22
  Date range:  2019-09-01 → 2026-04-01

  Category breakdown:
    Beauty Products                      108,302 rows ← NEW
    Vitamins & Supplements                46,626 rows ← NEW
    Oral Solid Forms                      33,417 rows
    Body Building                          6,354 rows ← NEW
    Oral Liquid Forms                      4,834 rows
    Topical Preparations                   3,342 rows
    Injectables                            3,174 rows
    Eyeglass Frames                        2,323 rows
    Eye Medications                        1,551 rows
    Ear & Nasal Preparations                 589 rows
    Suppositories & Pessaries                425 rows
    Wound Care                               413 rows
    IV Fluids & Infusions                    372 rows
    Surgical Supplies                        316 rows
    Inhalation Products                      285 rows
    Patient Mobility           

### Configuration

These are the tuning parameters for the model.  
You can adjust them without touching any of the logic below.

| Parameter | What it controls |
|---|---|
| `RAMP_UP_MONTHS` | How many opening months to exclude (new branches sell less while ramping up) |
| `KNN_K` | How many "most similar" branches to weight most heavily |
| `KNN_FALLBACK_THRESHOLD` | If all branches are equally similar, fall back to equal weights |
| `DEAD_STOCK_MULTIPLIER` | Flag an item as dead stock risk if opening qty > X times the monthly velocity |


In [6]:
# How many opening months to ignore when computing steady-state dispensing
# A branch opening in January may have unusual patterns for the first 1-3 months
RAMP_UP_MONTHS = 3

# How many top similar branches to weight most heavily in the prediction
KNN_K = 3

# If all branches are nearly equally similar to the Embu profile,
# the similarity score doesn't add value — fall back to equal weights
KNN_FALLBACK_THRESHOLD = 0.05

# Dead stock risk: if opening qty > this multiple of monthly velocity, flag it
# 1.5 means "you'd need more than 1.5 months to sell through this"
DEAD_STOCK_MULTIPLIER = 1.5


print("Configuration set")


Configuration set


---
## 1. Load the data

We load two queries from MySQL views:
- `monthly_diagnoses_aggregate` — one row per facility × month × diagnosis
- `monthly_dispensing_aggregate` — one row per facility × month × product category


In [7]:
# other data to pull 

fact_dispensing = pd.read_sql_query("SELECT * FROM tenri.fact_dispensing", tenri)
fact_inventory_snapshot = pd.read_sql_query("SELECT * FROM tenri.fact_inventory_snapshot", tenri)
dim_patient_profile = pd.read_sql_query("SELECT * FROM tenri.dim_patient_profile", tenri)
settings_facility = pd.read_sql_query("SELECT * FROM tenri.settings_clinics", tenri)

In [28]:
# ── Load simulated fact_dispensing ────────────────────────────────────────────
_sim_disp_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_fact_dispensing_*.csv"))
_sim_snap_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_inventory_snapshot_*.csv"))
 
if not _sim_disp_path:
    print("WARNING: No simulated_fact_dispensing_*.csv found — skipping.")
else:
    sim_disp = pd.read_csv(_sim_disp_path[-1], parse_dates=["date"])
    sim_disp = pd.read_csv(_sim_disp_path[-1], parse_dates=["date"])
 
    # Keep only columns that exist in real fact_dispensing
    # (simulated file has extra cols like 'category', 'inventory_health')
    FACT_DISP_COLS = [c for c in fact_dispensing.columns if c in sim_disp.columns]
    sim_disp_clean = sim_disp[FACT_DISP_COLS].copy()
 
    # Align dtypes
    sim_disp_clean["facility_id"] = sim_disp_clean["facility_id"].astype("int64")
    sim_disp_clean["product_id"]  = sim_disp_clean["product_id"].astype("int64")
 
    fact_dispensing = pd.concat([fact_dispensing, sim_disp_clean], ignore_index=True)
    print(f"fact_dispensing updated: {len(fact_dispensing):,} rows")

if not _sim_snap_path:
    print("WARNING: No simulated_inventory_snapshot_*.csv found — skipping.")
else:
    sim_snap = pd.read_csv(_sim_snap_path[-1], parse_dates=["snapshot_date"])
 
    # Keep only columns that exist in real fact_inventory_snapshot
    SNAP_COLS = [c for c in fact_inventory_snapshot.columns if c in sim_snap.columns]
    sim_snap_clean = sim_snap[SNAP_COLS].copy()
 
    # Align dtypes
    sim_snap_clean["facility_id"] = sim_snap_clean["facility_id"].astype("int64")
    sim_snap_clean["product_id"]  = sim_snap_clean["product_id"].astype("int64")
 
    fact_inventory_snapshot = pd.concat(
        [fact_inventory_snapshot, sim_snap_clean], ignore_index=True
    )
    print(f"fact_inventory_snapshot updated: {len(fact_inventory_snapshot):,} rows")


fact_dispensing updated: 1,919,172 rows
fact_inventory_snapshot updated: 699,698 rows


In [29]:
# ── Load product intelligence (new — for dashboard profitability tab) ─────────
_sim_intel_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_product_intelligence_*.csv"))
if _sim_intel_path:
    product_intel = pd.read_csv(_sim_intel_path[-1])
    print(f"product_intel loaded: {len(product_intel):,} products")
else:
    product_intel = pd.DataFrame()
    print("WARNING: No simulated_product_intelligence_*.csv found.")



product_intel loaded: 2,648 products


### Local Data

In [30]:
# Load diagnosis data
diag_df = monthly_diagnoses_aggregate.copy()

# Load dispensing data
disp_df = monthly_dispensing_aggregate.copy()
disp_df["months"] = pd.to_datetime(disp_df["months"])
 
print(f"disp_df rebuilt from updated monthly_dispensing_aggregate:")
print(f"  Rows:       {len(disp_df):,}")
print(f"  Categories: {disp_df['new_category_name'].nunique()}")
print(f"  Date range: {disp_df['months'].min()} → {disp_df['months'].max()}")
 
# Confirm the 3 new categories are present
new_cats = ["Beauty Products", "Vitamins & Supplements", "Body Building"]
for cat in new_cats:
    n = (disp_df["new_category_name"] == cat).sum()
    status = "✓" if n > 0 else "✗ MISSING"
    print(f"  {status} {cat}: {n:,} rows") 

# change data types 
diag_df['facility_id'] = diag_df['facility_id'].astype(int)

# Parse date columns so Python understands them as dates, not strings
diag_df['monthly'] = pd.to_datetime(diag_df['monthly'])

print(f"Diagnosis  — {diag_df['facility_id'].nunique()} facilities, "
      f"{diag_df['monthly'].nunique()} months, {len(diag_df):,} rows")
print(f"Dispensing — {disp_df['facility_id'].nunique()} facilities, "
      f"{disp_df['months'].nunique()} months, {len(disp_df):,} rows")

disp_df rebuilt from updated monthly_dispensing_aggregate:
  Rows:       213,081
  Categories: 22
  Date range: 2019-09-01 00:00:00 → 2026-04-01 00:00:00
  ✓ Beauty Products: 108,302 rows
  ✓ Vitamins & Supplements: 46,626 rows
  ✓ Body Building: 6,354 rows
Diagnosis  — 3 facilities, 46 months, 31,371 rows
Dispensing — 2 facilities, 75 months, 213,081 rows


In [31]:
print("=== DIAGNOSIS SAMPLE ===")
diag_df[['facility_id', 'monthly', 'diagnosis_disease_group',
         'unique_patient_count', 'chronic_cases', 'consultation_count']].head(8)


=== DIAGNOSIS SAMPLE ===


,facility_id,monthly,diagnosis_disease_group,unique_patient_count,chronic_cases,consultation_count
0,4,2018-08-01,MSK-Musculoskeletal,1,0.0,1
1,4,2018-10-01,Communicable-Infectious,1,0.0,1
2,4,2018-10-01,GI-Gastritis,1,1.0,1
3,4,2018-10-01,MNCH-Maternal,1,0.0,1
4,4,2018-10-01,NCD-Cardiovascular-Hypertension,1,1.0,1
5,4,2018-10-01,MSK-Musculoskeletal,1,1.0,1
6,4,2018-10-01,GI-PepticUlcer,1,1.0,1
7,4,2018-10-01,Respiratory-Pneumonia,2,0.0,2


In [33]:
print("=== DISPENSING SAMPLE ===")
disp_df[['facility_id', 'months', 'correct_therapeutic_class',
         'total_qty_dispensed', 'unique_patients']].head(10)  

=== DISPENSING SAMPLE ===


,facility_id,months,correct_therapeutic_class,total_qty_dispensed,unique_patients
0,4,2019-09-01,Ophthalmic — fluoroquinolone,2.0,2
1,4,2019-09-01,NSAID,60.0,2
2,4,2019-09-01,Antibiotic — beta-lactam combination,1.0,1
3,4,2019-09-01,Antacid,15.0,1
4,4,2019-09-01,Antiviral — topical/oral/IV,48.0,3
5,4,2019-09-01,Antiviral — topical,20.0,6
6,4,2019-09-01,Antiviral — topical/oral/IV,20.0,6
7,4,2019-09-01,None,1.0,1
8,4,2019-09-01,None,45.0,4
9,4,2019-09-01,Anthelmintic,24.0,24


### Get External Data

In [35]:
# Load Kenya health facilities GeoJSON and flatten to table
import json

geojson_path = r"C:/Users/Mercy/Documents/Tendri/Snowflake Pulls/Xana/snowflake/tendri/data/hotosm_ken_health_facilities_points_geojson.geojson"

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

rows = []
for feature in geojson["features"]:
    row = feature["properties"].copy()
    coords = feature["geometry"]["coordinates"] if feature["geometry"] else [None, None]
    row["longitude"] = coords[0]
    row["latitude"] = coords[1]
    rows.append(row)

kenya_health_facilities = pd.DataFrame(rows)

In [36]:
kenya_health_facilities[ kenya_health_facilities["name"].str.contains("Tenri", case=False, na=False) ]

,name,name:en,amenity,building,healthcare,healthcare:speciality,operator:type,capacity:persons,addr:full,addr:city,source,name:sw,osm_id,osm_type,longitude,latitude
726,Tenri Children's Hospital,None,hospital,None,hospital,None,None,None,None,None,None,None,9447253610,nodes,37.465775,-0.542050
1143,Tenri Childrens' Hospital (Maternety & Theatre),None,hospital,None,hospital,None,None,None,None,None,None,None,9447253609,nodes,37.465686,-0.542489


In [37]:
# ── Match Tenri facilities to kenya_health_facilities ─────────────────────────

# Step 1: Parse facility_id from facility_key in diag_df
diag_df['facility_id'] = diag_df['facility_key'].str.split('|').str[-1].astype(int)

# Step 2: Check pregnant_cases per facility — maternity hospital will score highest
maternity_signal = (
    diag_df.groupby('facility_id')
    .agg(
        pregnant_cases     = ('pregnant_cases', 'sum'),
        total_consultations= ('consultation_count', 'sum'),
        total_female       = ('total_female', 'sum'),
        total_male         = ('total_male', 'sum'),
    )
    .reset_index()
)
maternity_signal['pct_female'] = (
    maternity_signal['total_female'] /
    (maternity_signal['total_female'] + maternity_signal['total_male']) * 100
).round(1)
maternity_signal['pregnant_rate'] = (
    maternity_signal['pregnant_cases'] /
    maternity_signal['total_consultations'] * 100
).round(2)

print("=== Maternity signal per facility ===")
print(maternity_signal.sort_values('pregnant_cases', ascending=False).to_string())

# Step 3: Assign coordinates from kenya_health_facilities
# Facility with highest pregnant_cases = Tenri Childrens' Hospital (Maternity & Theatre)
# The other = Tenri Children's Hospital (general)

tenri_facilities = kenya_health_facilities[
    kenya_health_facilities["name"].str.contains("Tenri", case=False, na=False)
].copy()
print("\n=== Tenri facilities from OSM ===")
print(tenri_facilities[['name','latitude','longitude']].to_string())

# The maternity one has the higher pregnant_cases — assign accordingly
maternity_facility_id = maternity_signal.sort_values('pregnant_cases', ascending=False).iloc[0]['facility_id']
general_facility_id   = maternity_signal.sort_values('pregnant_cases', ascending=False).iloc[1]['facility_id']

# Map OSM entries: the one with "Maternity" in the name = maternity_facility_id
maternity_coords = tenri_facilities[tenri_facilities['name'].str.contains('Mater', case=False, na=False)].iloc[0]
general_coords   = tenri_facilities[~tenri_facilities['name'].str.contains('Mater', case=False, na=False)].iloc[0]

facility_coords = pd.DataFrame([
    {
        'facility_id': int(maternity_facility_id),
        'name':        maternity_coords['name'],
        'latitude':    maternity_coords['latitude'],
        'longitude':   maternity_coords['longitude'],
        'type':        'Maternity & Theatre',
    },
    {
        'facility_id': int(general_facility_id),
        'name':        general_coords['name'],
        'latitude':    general_coords['latitude'],
        'longitude':   general_coords['longitude'],
        'type':        'General Hospital',
    },
])

print("\n=== Facility coordinates assigned ===")
print(facility_coords.to_string())

=== Maternity signal per facility ===
   facility_id  pregnant_cases  total_consultations  total_female  total_male  pct_female  pregnant_rate
0            4           204.0                72961         35157       35961        49.4           0.28
1            5           186.0                32191         15861       15703        50.3           0.58
2            7             0.0                    2             2           0       100.0           0.00

=== Tenri facilities from OSM ===
                                                 name  latitude  longitude
726                         Tenri Children's Hospital -0.542050  37.465775
1143  Tenri Childrens' Hospital (Maternety & Theatre) -0.542489  37.465686

=== Facility coordinates assigned ===
   facility_id                                             name  latitude  longitude                 type
0            4  Tenri Childrens' Hospital (Maternety & Theatre) -0.542489  37.465686  Maternity & Theatre
1            5                 

In [38]:
from math import radians, sin, cos, sqrt, atan2

# ── Haversine distance function ───────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    """Returns distance in km between two lat/lon points."""
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── For each Tenri facility, find nearby health facilities ────────────────────
RADIUS_KM = 25  # adjust — 5km is a reasonable pharmacy catchment

nearby_results = []

for _, tenri_fac in facility_coords.iterrows():
    lat1 = tenri_fac['latitude']
    lon1 = tenri_fac['longitude']

    kdf = kenya_health_facilities.copy()
    kdf = kdf.dropna(subset=['latitude','longitude'])

    # Exclude the Tenri facilities themselves
    kdf = kdf[~kdf['name'].str.contains('Tenri', case=False, na=False)]

    kdf['distance_km'] = kdf.apply(
        lambda row: haversine_km(lat1, lon1, row['latitude'], row['longitude']),
        axis=1
    )
    

    nearby = kdf[kdf['distance_km'] <= RADIUS_KM].sort_values('distance_km').copy()
    nearby['tenri_facility_id']   = tenri_fac['facility_id']
    nearby['tenri_facility_name'] = tenri_fac['name']

    nearby_results.append(nearby)

nearby_all = pd.concat(nearby_results, ignore_index=True)

print(f"Facilities within {RADIUS_KM}km of each Tenri branch:")
print(nearby_all[['tenri_facility_name','name','amenity','healthcare',
                   'distance_km']].sort_values(['tenri_facility_name','distance_km'])
      .to_string())

Facilities within 25km of each Tenri branch:
                                tenri_facility_name                       name   amenity healthcare  distance_km
7                         Tenri Children's Hospital       Liberty Nursing home  hospital   hospital     1.685976
8                         Tenri Children's Hospital                       None  hospital       None    14.071240
9                         Tenri Children's Hospital  Mbeere subcounty Hospital  hospital   hospital    20.025875
10                        Tenri Children's Hospital      gatumbi health centre  hospital   hospital    21.078556
11                        Tenri Children's Hospital     karurumo health centre  hospital       None    22.377868
12                        Tenri Children's Hospital           Murrihs pharmacy  pharmacy   pharmacy    24.695271
13                        Tenri Children's Hospital           Thiba dispensary  hospital       None    24.837255
0   Tenri Childrens' Hospital (Maternety & Theatre)

In [39]:
# ── Proximity feature summary per Tenri facility ──────────────────────────────
proximity_features = []

for facility_id in facility_coords['facility_id']:
    nearby = nearby_all[nearby_all['tenri_facility_id'] == facility_id]

    proximity_features.append({
        'facility_id':          facility_id,
        'n_facilities_25km':     len(nearby),
        'n_hospitals_25km':      (nearby['amenity'] == 'hospital').sum(),
        'n_clinics_25km':        (nearby['amenity'].isin(['clinic','doctors'])).sum(),
        'n_pharmacies_25km':     (nearby['amenity'] == 'pharmacy').sum(),
        'n_health_centres_25km': (nearby['healthcare'].str.contains('health_centre|health centre',
                                  case=False, na=False)).sum(),
        'has_hospital_nearby':  int((nearby['amenity'] == 'hospital').any()),
    })

proximity_df = pd.DataFrame(proximity_features)
print(proximity_df.to_string())

   facility_id  n_facilities_25km  n_hospitals_25km  n_clinics_25km  n_pharmacies_25km  n_health_centres_25km  has_hospital_nearby
0            4                  7                 6               0                  1                      0                    1
1            5                  7                 6               0                  1                      0                    1


In [40]:
# ── Build category_proximity_driver ──────────────────────────────────────────
# Maps each product category to nearby facilities that drive demand for it
# Used in model_reasoning column in Step 8e

FACILITY_DEMAND_MAP = {
    'hospital':      ['Injectables', 'IV Fluids & Infusions', 'Wound Care',
                      'Oral Solid Forms', 'Infection Control'],
    'clinic':        ['Oral Solid Forms', 'Oral Liquid Forms', 'Injectables'],
    'doctors':       ['Oral Solid Forms', 'Oral Liquid Forms'],
    'pharmacy':      ['Oral Solid Forms', 'Topical Preparations'],
    'maternity':     ['Injectables', 'Oral Solid Forms', 'Vaccines & Biologicals',
                      'IV Fluids & Infusions'],
    'health_centre': ['Oral Solid Forms', 'Oral Liquid Forms', 'Injectables',
                      'Vaccines & Biologicals'],
}

category_proximity_driver = {}

for _, fac in nearby_all.iterrows():
    amenity    = str(fac.get('amenity',    '')).lower()
    healthcare = str(fac.get('healthcare', '')).lower()
    name       = str(fac.get('name', 'Unknown facility'))
    dist       = round(fac.get('distance_km', 0), 1)

    # Determine facility type
    if 'maternity' in name.lower():
        fac_type = 'maternity'
    elif amenity in FACILITY_DEMAND_MAP:
        fac_type = amenity
    elif 'health_centre' in healthcare or 'health centre' in healthcare:
        fac_type = 'health_centre'
    else:
        continue

    for category in FACILITY_DEMAND_MAP[fac_type]:
        if category not in category_proximity_driver:
            category_proximity_driver[category] = []
        entry = f"{name} ({dist}km)"
        if entry not in category_proximity_driver[category]:
            category_proximity_driver[category].append(entry)

# Cap at 2 facilities per category for readability
for cat in category_proximity_driver:
    category_proximity_driver[cat] = category_proximity_driver[cat][:2]

print("=== category_proximity_driver ===")
for cat, drivers in sorted(category_proximity_driver.items()):
    print("  {:<35} {}".format(cat, drivers))

'''
#The full order for the cells that feed into Step 8e is then:
proximity_df        ← built from nearby_all
category_proximity_driver  ← NEW, built from nearby_all + FACILITY_DEMAND_MAP
CATEGORY_CONTEXT + BURDEN_SIGNALS (Step 3f)

Step 8e             ← uses all three: category_proximity_driver, CATEGORY_CONTEXT, BURDEN_SIGNALS
'''


=== category_proximity_driver ===
  IV Fluids & Infusions               ['Liberty Nursing home (1.7km)', 'None (14.0km)']
  Infection Control                   ['Liberty Nursing home (1.7km)', 'None (14.0km)']
  Injectables                         ['Liberty Nursing home (1.7km)', 'None (14.0km)']
  Oral Solid Forms                    ['Liberty Nursing home (1.7km)', 'None (14.0km)']
  Topical Preparations                ['Murrihs pharmacy (24.7km)']
  Wound Care                          ['Liberty Nursing home (1.7km)', 'None (14.0km)']


'\n#The full order for the cells that feed into Step 8e is then:\nproximity_df        ← built from nearby_all\ncategory_proximity_driver  ← NEW, built from nearby_all + FACILITY_DEMAND_MAP\nCATEGORY_CONTEXT + BURDEN_SIGNALS (Step 3f)\n\nStep 8e             ← uses all three: category_proximity_driver, CATEGORY_CONTEXT, BURDEN_SIGNALS\n'

In [42]:
# ── Load Embu DHS external data ───────────────────────────────────────────────
embu_dhs = pd.read_csv(r'C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\embu_dhs_2022.csv')
embu_dhs = embu_dhs.rename(columns={'Unnamed: 0': 'geography'})
embu_dhs = embu_dhs.set_index('geography')

In [43]:


# Helper to safely parse values — some have brackets e.g. (91)
def safe_float(val):
    if pd.isna(val): return None
    s = str(val).replace('(','').replace(')','').replace('<','').strip()
    try: return float(s)
    except: return None

# ── Extract features relevant to pharmacy demand ──────────────────────────────
embu_row = embu_dhs.loc['embu']

embu_external = {
    # Maternal & reproductive health — drives ANC, FP, postnatal dispensing
    'fertility_rate':             safe_float(embu_row['Total fertility rate (number of children per woman)']),
    'teen_pregnancy_pct':         safe_float(embu_row['Teenage pregnancy (% age 15-19 who have ever been pregnant)']),
    'modern_fp_use_pct':          safe_float(embu_row['Use of modern method of FP (% of married women age 15-49)']),
    'antenatal_4plus_visits_pct': safe_float(embu_row['Women age 15-49 who had a live birth and had 4+ antenatal visits (%)']),
    'skilled_birth_pct':          safe_float(embu_row['Births delivered by a skilled provider2 (%)']),

    # Child health — drives paediatric dispensing (ORS, vaccines, supplements)
    'u5_stunting_pct':            safe_float(embu_row['Children under 5 who are stunted (%) (too short for their age)']),
    'u5_underweight_pct':         safe_float(embu_row['Children under 5 who are underweight (%) (too thin for their age)']),
    'vaccination_pct':            safe_float(embu_row['Children age 12-23 months fully vaccinated (basic antigens)3 (%)']),

    # Malaria burden — drives artemether, ITN, SP/Fansidar dispensing
    'itn_access_pct':             safe_float(embu_row['Household population with access to an insecticide-treated net (ITN) (%)']),
    'itn_use_pct':                safe_float(embu_row['Household population who slept under an ITN the night before the survey (%)']),

    # Socioeconomic — proxy for spend tier, OTC vs Rx mix
    'clean_fuel_access_pct':      safe_float(embu_row['Household population relying on clean fuels and technologies for cooking, space heating, & lighting (%)']),
    'water_access_pct':           safe_float(embu_row['Household population with access to at least basic drinking water service (%)']),
    'sanitation_access_pct':      safe_float(embu_row['Household population with at least basic sanitation service (%)']),
    'women_no_education_pct':     safe_float(embu_row['Women age 15-49 with no formal education (%)']),

    # Urban assumption
    'is_urban_branch':            1,
}

print("=== Embu external features ===")
for k, v in embu_external.items():
    print(f"  {k:<35} {v}")

=== Embu external features ===
  fertility_rate                      3.1
  teen_pregnancy_pct                  14.0
  modern_fp_use_pct                   75.0
  antenatal_4plus_visits_pct          62.0
  skilled_birth_pct                   96.0
  u5_stunting_pct                     20.0
  u5_underweight_pct                  11.0
  vaccination_pct                     91.0
  itn_access_pct                      36.0
  itn_use_pct                         33.0
  clean_fuel_access_pct               15.0
  water_access_pct                    73.0
  sanitation_access_pct               48.0
  women_no_education_pct              1.0
  is_urban_branch                     1


In [44]:
# ── PDF paths — update if different on your machine ──────────────────────────
_PROJ_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\2019-Kenya-population-and-Housing-Census-Summary-Report-on-Kenyas-Population-Projections.pdf"
_VOL1_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\2019-Kenya-population-and-Housing-Census-Volume-1-Population-By-County-And-Sub-County.pdf"
_CIDP_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\EMBU_CIDP_2023-2027.pdf"

In [45]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _parse_num(s):
    try: return int(str(s).replace(",","").strip())
    except: return None
 
def _parse_float(s):
    try: return float(str(s).replace(",","").strip())
    except: return None
 
def _get_lines(doc, pg):
    return [l.strip() for l in doc[pg].get_text().split("\n") if l.strip()]
 
def _county_table(doc, page_idx, n_cols, skip):
    lines = _get_lines(doc, page_idx)
    rows, i = [], 0
    while i < len(lines):
        line = lines[i]
        if re.match(r"^[\d,]+$", line) or line in skip or len(line) < 2:
            i += 1; continue
        nums, j = [], i + 1
        while j < len(lines) and len(nums) < n_cols:
            if re.match(r"^[\d,]+$", lines[j]):
                nums.append(_parse_num(lines[j])); j += 1
            else: break
        if len(nums) == n_cols:
            rows.append({"county": line, "_nums": nums}); i = j
        else: i += 1
    return rows
 

In [48]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — KNBS Official Projections
# ══════════════════════════════════════════════════════════════════════════════
_doc_proj = fitz.open(_PROJ_PDF)
_PROJ_YEARS = ["2020","2021","2022","2023","2024","2025","2030","2035","2040","2045"]
_LF_YEARS   = ["2020","2021","2022","2023","2024","2025","2030","2035"]
_HH_YEARS   = ["2020","2021","2022","2023","2024","2025","2026","2027","2028","2029","2030"]
 
# Population projections (page 2)
_raw = _county_table(_doc_proj, 1, 10,
    {"County","Kenya"} | set(_PROJ_YEARS) | {"Population Projections by County, 2020-2045"})
_proj_embu = next(r for r in _raw if r["county"] == "Embu")
_proj = dict(zip(_PROJ_YEARS, _proj_embu["_nums"]))
 
# Labour force (page 4)
_raw_lf = _county_table(_doc_proj, 3, 8,
    {"County","Kenya"} | set(_LF_YEARS) | {
        "Population in the Labour Force, age 15-64 by County, 2020-2035",
        "Labour Force Projections",
        "The population in the labour force (age 15-64) is expected to increase by 40.7 percent from 28.8 million",
        "in 2020 to 40.5 million by 2035."})
_lf_embu = next(r for r in _raw_lf if r["county"] == "Embu")
_lf = dict(zip(_LF_YEARS, _lf_embu["_nums"]))
 
# Households (page 5)
_raw_hh = _county_table(_doc_proj, 4, 11,
    {"County","Kenya"} | set(_HH_YEARS) | {
        "Projected Number of Households by County, 2020-2030",
        "Households Projections",
        "Data on Household projections show that by 2030, there will be approximately 15.9 million households.",
        "Nairobi City, which is entirely urban, will require nearly 2 million houses to host its population by 2030."})
_hh_embu = next(r for r in _raw_hh if r["county"] == "Embu")
_hh = dict(zip(_HH_YEARS, _hh_embu["_nums"]))
 
# 2019 census baseline from Volume I (sex breakdown)
_doc_vol1 = fitz.open(_VOL1_PDF)
_census_rows = []
for _pg in range(len(_doc_vol1)):
    _text  = _doc_vol1[_pg].get_text()
    _lines = [l.strip() for l in _text.split("\n") if l.strip()]
    for _i, _l in enumerate(_lines):
        if not re.match(r"^[\d,]+$", _l) and len(_l) > 2 and _i + 4 < len(_lines):
            _cands = []
            for _k in range(1, 5):
                if re.match(r"^[\d,]+$", _lines[_i+_k]): _cands.append(_parse_num(_lines[_i+_k]))
                else: break
            if len(_cands) == 4 and _cands[3] == _cands[0] + _cands[1] + _cands[2]:
                _census_rows.append({"county": _l, "male": _cands[0], "female": _cands[1],
                                     "intersex": _cands[2], "total": _cands[3]})
_census_df  = pd.DataFrame(_census_rows).drop_duplicates("county")
_embu_census = _census_df[_census_df["county"].str.contains("Embu", na=False)].iloc[0]
 
_TOTAL_2019 = int(_embu_census["total"])    # 608,599
_TOTAL_2025 = int(_proj["2025"])            # 661,690
_LABOUR_2025 = int(_lf["2025"])             # 423,316
_HH_2025     = int(_hh["2025"])             # 208,991
_ANNUAL_RATE = (_TOTAL_2025 / _TOTAL_2019) ** (1/6) - 1
 
 


In [49]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — CIDP Age Cohort Data (Table 5)
# ══════════════════════════════════════════════════════════════════════════════
_doc_cidp = fitz.open(_CIDP_PDF)
_CIDP_YEARS = [2019, 2022, 2025, 2027]
 
# Table 5: 5-year age cohorts
_lines_45 = _get_lines(_doc_cidp, 44)
_rows5 = []
_i5 = 0
while _i5 < len(_lines_45):
    _line = _lines_45[_i5]
    if re.match(r"^\s*\d+[-+]\d*\s*$", _line) or _line in ("Age NS","All Ages"):
        _nums, _j = [], _i5 + 1
        while _j < len(_lines_45) and len(_nums) < 12:
            _v = _lines_45[_j]
            if re.match(r"^[\d,]+$", _v): _nums.append(_parse_num(_v)); _j += 1
            elif _v == "-": _nums.append(None); _j += 1
            else: break
        if len(_nums) == 12:
            for _yr_idx, _year in enumerate(_CIDP_YEARS):
                _o = _yr_idx * 3
                _rows5.append({"age_cohort": _line.strip(), "year": _year,
                               "male": _nums[_o], "female": _nums[_o+1], "total": _nums[_o+2]})
            _i5 = _j; continue
    _i5 += 1
 
_t5 = pd.DataFrame(_rows5)
_t5_2025 = _t5[_t5["year"] == 2025].set_index("age_cohort")
 
# Table 8: broad age groups
_lines_49  = _get_lines(_doc_cidp, 48)
_lines_50  = _get_lines(_doc_cidp, 49)
_t8_start  = next(i for i,l in enumerate(_lines_49) if "1.5.3" in l)
_lines_t8  = _lines_49[_t8_start:] + _lines_50
 
_BROAD = [
    ("Infant",       "infant_under1"),
    ("Under 5",      "under5"),
    ("Pre-School",   "preschool_3to5"),
    ("Primary",      "primary_6to13"),
    ("Secondary",    "secondary_13to19"),
    ("Youth",        "youth_15to29"),
    ("Economically", "economically_active_15to64"),
    ("Aged",         "aged_65plus"),
]
_rows8, _seen8 = [], set()
for _trigger, _label in _BROAD:
    if _label in _seen8: continue
    _idx = next((i for i,l in enumerate(_lines_t8) if _trigger in l), None)
    if _idx is None: continue
    _nums8, _j = [], _idx + 1
    while _j < len(_lines_t8) and len(_nums8) < 12:
        _v = _lines_t8[_j]
        if re.match(r"^[\d,]+$", _v): _nums8.append(_parse_num(_v)); _j += 1
        elif _v == "-": _nums8.append(None); _j += 1
        else: _j += 1
        if len(_nums8) == 12: break
    if len(_nums8) == 12:
        _seen8.add(_label)
        for _yr_idx, _year in enumerate(_CIDP_YEARS):
            _o = _yr_idx * 3
            _rows8.append({"age_group": _label, "year": _year,
                           "male": _nums8[_o], "female": _nums8[_o+1], "total": _nums8[_o+2]})
 
_t8 = pd.DataFrame(_rows8)
_t8_2025 = _t8[_t8["year"] == 2025].set_index("age_group")
 

In [50]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Compute derived features (all scaled 0-1)
# ══════════════════════════════════════════════════════════════════════════════
def _cohort_total(*groups):
    return sum(_t5_2025.loc[g, "total"] for g in groups if g in _t5_2025.index)
 
def _cohort_male(*groups):
    return sum(_t5_2025.loc[g, "male"] for g in groups if g in _t5_2025.index)
 
def _cohort_female(*groups):
    return sum(_t5_2025.loc[g, "female"] for g in groups if g in _t5_2025.index)
 
def _broad(group):
    return int(_t8_2025.loc[group, "total"]) if group in _t8_2025.index else None
 
# Age band shares (2025 CIDP cohorts / official KNBS 2025 total)
_pct_under_15    = round(_cohort_total("0-4","5-9","10-14") / _TOTAL_2025, 4)
_pct_youth_15_29 = round(_cohort_total("15-19","20-24","25-29") / _TOTAL_2025, 4)
_pct_adult_30_64 = round(_cohort_total("30-34","35-39","40-44","45-49","50-54","55-59","60-64") / _TOTAL_2025, 4)
_pct_aged_65plus = round(_cohort_total("65-69","70-74","75-79","80+") / _TOTAL_2025, 4)
 
# Category demand proxies (target buyer population / total population)
_beauty_women    = _cohort_female("15-19","20-24","25-29","30-34","35-39","40-44","45-49")
_bb_men          = _cohort_male("15-19","20-24","25-29","30-34")
_supp_adults     = _cohort_total("25-29","30-34","35-39","40-44","45-49","50-54","55-59","60-64")
 
_beauty_target_pct      = round(_beauty_women / _TOTAL_2025, 4)
_bb_target_pct          = round(_bb_men       / _TOTAL_2025, 4)
_supplements_target_pct = round(_supp_adults  / _TOTAL_2025, 4)
 
# Labour force pct and gender split
_labour_force_pct = round(_LABOUR_2025 / _TOTAL_2025, 4)
_pct_female_2019  = round(float(_embu_census["female"]) / _TOTAL_2019, 4)
 
 

In [51]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Update embu_external
# This extends the dict already defined above in the notebook.
# Keys are grouped by source so they're easy to audit.
# ══════════════════════════════════════════════════════════════════════════════
embu_external.update({
 
    # ── KNBS Official Projections ─────────────────────────────────────────────
    # Source: 2019 KNBS Population Projections Summary Report
    # These are exact official figures — not modelled here
    "embu_total_population_2025": _TOTAL_2025,      # 661,690
    "embu_labour_force_2025":     _LABOUR_2025,     # 423,316  (age 15-64)
    "embu_households_2025":       _HH_2025,         # 208,991
    "embu_annual_growth_rate":    round(_ANNUAL_RATE, 5),  # 0.01404 (1.4%/yr)
 
    # Year-by-year for trend analysis
    "embu_pop_2020": int(_proj["2020"]),  # 628,527
    "embu_pop_2021": int(_proj["2021"]),  # 635,160
    "embu_pop_2022": int(_proj["2022"]),  # 641,792
    "embu_pop_2023": int(_proj["2023"]),  # 648,425
    "embu_pop_2024": int(_proj["2024"]),  # 655,057
    "embu_pop_2025": int(_proj["2025"]),  # 661,690
 
    # ── Derived ratios (0-1 scale) ────────────────────────────────────────────
    # These match the scale of the DHS indicators already in embu_external
    "embu_labour_force_pct": _labour_force_pct,   # 0.6397 — working-age share
    "embu_pct_female_2019":  _pct_female_2019,    # 0.5001 — near 50/50
 
    # ── CIDP Age Band Shares (2025, derived from KNBS base) ───────────────────
    # Source: CIDP Table 5 age cohorts / KNBS 2025 total
    # Used by KNN to infer disease burden and consumption patterns
    "pct_under_15":     _pct_under_15,    # 0.2969 — paediatric pharma, vaccines
    "pct_youth_15_29":  _pct_youth_15_29, # 0.2675 — BB (male), beauty exploration (female)
    "pct_adult_30_64":  _pct_adult_30_64, # 0.3723 — core supplement + beauty buyer
    "pct_aged_65_plus": _pct_aged_65plus, # 0.0634 — chronic disease pharma, supplements
 
    # ── Category Demand Proxies (0-1 scale) ───────────────────────────────────
    # Derived from CIDP Table 5 cohorts, 2025 projection
    # Represents the proportion of the population that is the PRIMARY buyer
    # for each of the 3 new product categories
    "beauty_target_pct":      _beauty_target_pct,      # 0.2660 — women aged 15-49
    "bb_target_pct":          _bb_target_pct,          # 0.1742 — men aged 15-34
    "supplements_target_pct": _supplements_target_pct, # 0.4569 — adults aged 25-64
 
})

In [52]:
# ── Validation print ──────────────────────────────────────────────────────────
print("=== embu_external updated ===")
print(f"  Total keys now: {len(embu_external)}")
print(f"  2025 population:      {embu_external['embu_total_population_2025']:,}")
print(f"  Labour force 2025:    {embu_external['embu_labour_force_2025']:,}")
print(f"  Households 2025:      {embu_external['embu_households_2025']:,}")
print(f"  Annual growth rate:   {embu_external['embu_annual_growth_rate']*100:.3f}%")
print(f"  Age bands sum:        {_pct_under_15 + _pct_youth_15_29 + _pct_adult_30_64 + _pct_aged_65plus:.4f}  (expected ~1.0)")
print(f"  Beauty target:        {embu_external['beauty_target_pct']*100:.1f}% of population  ({_beauty_women:,} women 15-49)")
print(f"  BB target:            {embu_external['bb_target_pct']*100:.1f}% of population  ({_bb_men:,} men 15-34)")
print(f"  Supplements target:   {embu_external['supplements_target_pct']*100:.1f}% of population  ({_supp_adults:,} adults 25-64)")
 
print("\n  All embu_external keys:")
for k, v in sorted(embu_external.items()):
    print(f"    {k:<40} {v}")

=== embu_external updated ===
  Total keys now: 34
  2025 population:      661,690
  Labour force 2025:    423,316
  Households 2025:      208,991
  Annual growth rate:   1.404%
  Age bands sum:        1.0001  (expected ~1.0)
  Beauty target:        26.6% of population  (176,024 women 15-49)
  BB target:            17.4% of population  (115,274 men 15-34)
  Supplements target:   45.7% of population  (302,331 adults 25-64)

  All embu_external keys:
    antenatal_4plus_visits_pct               62.0
    bb_target_pct                            0.1742
    beauty_target_pct                        0.266
    clean_fuel_access_pct                    15.0
    embu_annual_growth_rate                  0.01404
    embu_households_2025                     208991
    embu_labour_force_2025                   423316
    embu_labour_force_pct                    0.6397
    embu_pct_female_2019                     0.5001
    embu_pop_2020                            628527
    embu_pop_2021              

---
## 2. Define the disease burden mapping

Your diagnosis view stores disease groups as text like `"NCD-Cardiovascular-Hypertension"`.  
We want to convert those into numeric scores per dimension (malaria burden, diabetes burden, etc.)  
so we can compare facilities mathematically.

This dictionary maps each burden dimension to the disease group text fragments that belong to it.  
If any of those fragments appear in a row's `diagnosis_disease_group`, that row scores 1 for that dimension.


In [70]:
# Each key is a burden dimension we care about.
# Each value is a list of text fragments to look for in diagnosis_disease_group.
# If any fragment matches, the row gets a score of 1 for that dimension.
BURDEN_MAP = {
    'malaria':         ['Communicable-Malaria', 'Malaria'],
    'communicable':    ['Communicable-Infectious', 'Communicable'],
    'hypertension':    ['NCD-Cardiovascular-Hypertension'],
    'cardiovascular':  ['NCD-Cardiovascular-CoronaryHeart',
                        'NCD-Cardiovascular-HeartFailure',
                        'NCD-Cardiovascular'],
    'diabetes':        ['NCD-Endocrine-Diabetes', 'NCD-Endocrine-Other',
                        'NCD-Endocrine'],
    'respiratory':     ['Respiratory-URTI', 'Respiratory-Pneumonia',
                        'Respiratory-Rhinitis-Sinusitis',
                        'NCD-Respiratory-Asthma', 'Respiratory'],
    'gi':              ['GI-Gastritis', 'GI-PepticUlcer', 'GI-GERD-Oesophageal'],
    'maternal':        ['MNCH-Maternal', 'MNCH'],
    'gynaecological':  ['GU-GynaeUrological'],
    'musculoskeletal': ['MSK-Musculoskeletal'],
    'dermatological':  ['Dermatological'],
    'mental_health':   ['NCD-Mental'],
}

print(f"Defined {len(BURDEN_MAP)} burden dimensions:")
print(list(BURDEN_MAP.keys()))


Defined 12 burden dimensions:
['malaria', 'communicable', 'hypertension', 'cardiovascular', 'diabetes', 'respiratory', 'gi', 'maternal', 'gynaecological', 'musculoskeletal', 'dermatological', 'mental_health']


---
## 3. Build facility profiles

For each existing Tendri branch, we build a **feature vector** — a row of numbers  
that describes what kind of patients that branch sees.

We use ratios (percentages) not raw counts, so a small branch and a large branch  
can be compared fairly.

**Features:**
- Disease burden ratios (e.g. 17% of diagnoses are hypertension)
- Case type ratios (e.g. 12% of visits are chronic disease follow-ups)
- Age mix ratios (e.g. 23% of patients are paediatric)
- Gender ratio
- Average monthly consultations (facility size)

In [ ]:


# Step 3a: Score each diagnosis row for each burden dimension
# We create one new column per burden dimension
# The value is 1 if the disease group matches, 0 if not

df = diag_df.copy()   # always work on a copy, don't modify the original

for burden_name, keywords in BURDEN_MAP.items():
    # Wrap each keyword so it only matches a full segment, not a prefix.
    # combined_diagnosis uses '|' as segment separator, so a valid match is:
    #   - at start of string OR after a '|'     → (?:^|\|)
    #   - at end of string OR before a '|'      → (?=\||$)
    # This prevents 'NCD-Cardiovascular' matching inside
    # 'NCD-Cardiovascular-Hypertension'.
    segmented = [r'(?:^|\|)' + re.escape(kw) + r'(?=\||$)' for kw in keywords]
    pattern = '|'.join(segmented)
    
    # str.contains checks if the pattern appears anywhere in the text
    # case=False means it's not case-sensitive
    # na=False means NaN values score 0 instead of NaN
    df[f'burden_{burden_name}'] = (
        df['combined_diagnosis']
        .str.contains(pattern, case=False, na=False)
        .astype(int)   # convert True/False to 1/0
    )

print("Burden columns created:")
burden_cols = [f'burden_{k}' for k in BURDEN_MAP]
print(burden_cols)
print()

# Show a sample: diagnosis row → burden scores
df[['combined_diagnosis'] + burden_cols[:5]].head(5)

Burden columns created:
['burden_malaria', 'burden_communicable', 'burden_hypertension', 'burden_cardiovascular', 'burden_diabetes', 'burden_respiratory', 'burden_gi', 'burden_maternal', 'burden_gynaecological', 'burden_musculoskeletal', 'burden_dermatological', 'burden_mental_health']



,combined_diagnosis,burden_malaria,burden_communicable,burden_hypertension,burden_cardiovascular,burden_diabetes
0,MSK-Musculoskeletal,0,0,0,0,0
1,Communicable-Infectious,0,1,0,0,0
2,GI-Gastritis|Digestive: Gastritis & Duodenitis,0,0,0,0,0
3,MNCH-Maternal,0,0,0,0,0
4,NCD-Cardiovascular-Hypertension,0,0,1,0,0


In [ ]:
# Step 3b: Aggregate age bands into three groups
# Instead of 10 separate age columns, we summarise as paediatric / working age / chronic age

# Paediatric = under 13
df['age_paediatric'] = (
    df['total_age_less_than_1'] +
    df['total_age_1_4'] +
    df['total_age_5_12']
)

# Working age = 18-44
df['age_working'] = (
    df['total_age_18_24'] +
    df['total_age_25_34'] +
    df['total_age_35_44']
)

# Chronic age = 45+ (more likely to have NCDs like diabetes, hypertension)
df['age_chronic'] = (
    df['total_age_45_54'] +
    df['total_age_55_64'] +
    df['total_age_over_65']
)

print("Age band columns created: age_paediatric, age_working, age_chronic")


Age band columns created: age_paediatric, age_working, age_chronic


In [ ]:
# Step 3c: Aggregate everything to facility level
# Currently: one row per facility × month × diagnosis
# We want: one row per facility (sum across all months and diagnoses)

case_cols  = ['chronic_cases', 'pregnant_cases', 'follow_up_cases',
              'immunisation_cases', 'medication_pickup_cases']

count_cols = ['unique_patient_count', 'total_male', 'total_female',
              'age_paediatric', 'age_working', 'age_chronic',
              'consultation_count']

# .groupby('facility_id') groups all rows for the same facility together
# .sum() adds up all the values
# .reset_index() turns the group key back into a regular column
facility_agg = (
    df.groupby('facility_id')[burden_cols + case_cols + count_cols]
    .sum()
    .reset_index()
)

print(f"Aggregated to {len(facility_agg)} facilities")
facility_agg.head()


Aggregated to 3 facilities


,facility_id,burden_malaria,burden_communicable,burden_hypertension,burden_cardiovascular,burden_diabetes,burden_respiratory,burden_gi,burden_maternal,burden_gynaecological,...,follow_up_cases,immunisation_cases,medication_pickup_cases,unique_patient_count,total_male,total_female,age_paediatric,age_working,age_chronic,consultation_count
0,4,0,5281,1997,187,1313,4131,1977,594,1727,...,107.0,89.0,1.0,71118,35961,35157,9241,22299,35898,72961
1,5,0,2943,502,59,368,1862,760,326,763,...,11.0,30.0,1.0,31564,15703,15861,4256,9678,15835,32191
2,7,0,0,0,0,0,0,1,0,0,...,0.0,0.0,0.0,2,0,2,0,0,1,2


In [ ]:
# Step 3d: Convert counts → ratios
# Ratios allow fair comparison between large and small facilities

# Total diagnoses across all burden dimensions (denominator for burden ratios)
total_diagnoses = facility_agg[burden_cols].sum(axis=1).replace(0, np.nan)
# axis=1 means sum across columns (row-wise)
# replace(0, np.nan) avoids division by zero

total_consults = facility_agg['consultation_count'].replace(0, np.nan)
total_patients = facility_agg['unique_patient_count'].replace(0, np.nan)

# Burden ratios: what fraction of diagnoses fall in each disease group
for col in burden_cols:
    facility_agg[f'{col}_ratio'] = facility_agg[col] / total_diagnoses

# Case type ratios: what fraction of consultations are each case type
for col in case_cols:
    facility_agg[f'{col}_ratio'] = facility_agg[col] / total_consults

# Age mix ratios
facility_agg['pct_paediatric']  = facility_agg['age_paediatric'] / total_patients
facility_agg['pct_working']     = facility_agg['age_working']    / total_patients
facility_agg['pct_chronic_age'] = facility_agg['age_chronic']    / total_patients
facility_agg['pct_female']      = facility_agg['total_female']   / total_patients

print("Ratios computed. Sample burden ratios:")
ratio_cols = (
    [f'{c}_ratio' for c in burden_cols + case_cols]
    + ['pct_paediatric', 'pct_working', 'pct_chronic_age', 'pct_female']
)

facility_agg[['facility_id'] + [c for c in ratio_cols if 'burden_' in c]].round(3)


Ratios computed. Sample burden ratios:


,facility_id,burden_malaria_ratio,burden_communicable_ratio,burden_hypertension_ratio,burden_cardiovascular_ratio,burden_diabetes_ratio,burden_respiratory_ratio,burden_gi_ratio,burden_maternal_ratio,burden_gynaecological_ratio,burden_musculoskeletal_ratio,burden_dermatological_ratio,burden_mental_health_ratio
0,4,0.0,0.239,0.090,0.008,0.059,0.187,0.089,0.027,0.078,0.155,0.066,0.0
1,5,0.0,0.330,0.056,0.007,0.041,0.209,0.085,0.037,0.085,0.094,0.057,0.0
2,7,0.0,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.0


In [ ]:
# Step 3e: Add average monthly consultations as a facility size feature
# This tells us how busy each branch is on average per month

monthly_size = (
    diag_df
    .groupby(['facility_id', 'monthly'])['consultation_count']   # group by facility AND month
    .sum()                                                         # total consultations per month
    .groupby('facility_id')                                        # then group by facility only
    .mean()                                                        # average across months
    .rename('avg_monthly_consultations')
    .reset_index()
)

# Merge this back onto our facility profiles
facility_profiles = facility_agg.merge(monthly_size, on='facility_id', how='left')
# ── ADD: Dispensing behaviour features (category mix) ────────────────────────
# NEW — was discussed but missing from notebook
disp_behaviour = (
    disp_df.groupby('facility_id')
    .agg(
        total_qty       = ('total_qty_dispensed', 'sum'),
        unique_products = ('product_id',          'nunique'),
        avg_patients    = ('unique_patients',      'mean'),
    ).reset_index()
)
cat_mix = (
    disp_df.groupby(['facility_id','new_category_name'])['total_qty_dispensed']
    .sum().unstack(fill_value=0)
)
cat_mix = cat_mix.div(cat_mix.sum(axis=1), axis=0)
cat_mix.columns = [f'cat_ratio_{c.lower().replace(" ","_").replace("/","_")}'
                   for c in cat_mix.columns]
cat_mix = cat_mix.reset_index()
cat_ratio_cols = [c for c in cat_mix.columns if c.startswith('cat_ratio_')]

facility_profiles = facility_profiles.merge(disp_behaviour, on='facility_id', how='left')
facility_profiles = facility_profiles.merge(cat_mix, on='facility_id', how='left')

# ── FIX: Merge proximity ONCE only ───────────────────────────────────────────
# REMOVED duplicate merge — was appearing twice
facility_profiles = facility_profiles.merge(proximity_df, on='facility_id', how='left')

proximity_cols = ['n_facilities_25km','n_hospitals_25km','n_clinics_25km',
                  'n_pharmacies_25km','has_hospital_nearby']
for col in proximity_cols:
    facility_profiles[col] = facility_profiles[col].fillna(0)

# Build ratio_cols — ONCE, in one place
ratio_cols = (
    [f'{c}_ratio' for c in burden_cols + case_cols]
    + ['pct_paediatric', 'pct_working', 'pct_chronic_age', 'pct_female']
    + ['avg_monthly_consultations']
    + proximity_cols
    + cat_ratio_cols
)

print(f"Final facility profiles: {len(facility_profiles)} facilities × {len(ratio_cols)} features")
print(f"  — {len([c for c in ratio_cols if 'burden_' in c])} burden ratio features")
print(f"  — {len(proximity_cols)} proximity features")
print(f"  — {len(cat_ratio_cols)} dispensing category features")

Final facility profiles: 3 facilities × 46 features
  — 12 burden ratio features
  — 5 proximity features
  — 19 dispensing category features


In [ ]:
# Verify all ratio_cols exist in facility_profiles before Step 4
missing = [c for c in ratio_cols if c not in facility_profiles.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print(f"✓ All {len(ratio_cols)} feature columns present in facility_profiles")

print(f"\nfacility_profiles shape: {facility_profiles.shape}")
print(facility_profiles[proximity_cols].head())  # confirm proximity values are not all 0

✓ All 46 feature columns present in facility_profiles

facility_profiles shape: (3, 75)
   n_facilities_25km  n_hospitals_25km  n_clinics_25km  n_pharmacies_25km  \
0                7.0               6.0             0.0                1.0   
1                7.0               6.0             0.0                1.0   
2                0.0               0.0             0.0                0.0   

   has_hospital_nearby  
0                  1.0  
1                  1.0  
2                  0.0  


In [ ]:
# ── Paths — update if different ───────────────────────────────────────────────
GT_DIR   = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\google_trends_output"    # folder where trend scripts saved outputs
   # folder where trend scripts saved outputs
PROX_CSV = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\pharmaplus_proximity_data.csv"  # from pharmaplus_proximity_scraper.py

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — GOOGLE TRENDS FEATURES
# Loads embu_google_trends_features_{DATE}.csv
# Each row: feature name | value (already 0-1 scaled by the trends script)
# ══════════════════════════════════════════════════════════════════════════════
print("Loading Google Trends features...")
 
_gt_paths = sorted(glob.glob(os.path.join(GT_DIR, "embu_google_trends_features_*.csv")))
 
if not _gt_paths:
    print("  WARNING: No embu_google_trends_features_*.csv found in google_trends_output/")
    print("  Run the Google Trends pipeline first, then re-run this cell.")
    embu_google_trends = {}
else:
    _gt_df = pd.read_csv(_gt_paths[-1])  # most recent run
 
    # Validate expected columns
    if not {"feature", "value"}.issubset(_gt_df.columns):
        raise ValueError(f"Expected columns 'feature' and 'value', got: {_gt_df.columns.tolist()}")
 
    embu_google_trends = dict(zip(_gt_df["feature"], _gt_df["value"]))
 
    print(f"  Loaded {len(embu_google_trends)} features from: {_gt_paths[-1]}")
    print(f"  Features:")
    for feat, val in embu_google_trends.items():
        print(f"    {feat:<45} {val}")
 

In [ ]:
# 3f. ══════════════════════════════════════════════════════════════════════════════
# PART 2 — PROXIMITY FEATURES
# Loads pharmaplus_proximity_data.csv from the SerpAPI scraper
# Derives count-based features per category within 3km of each facility
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading proximity data...")
 
if not os.path.exists(PROX_CSV):
    print(f"  WARNING: {PROX_CSV} not found.")
    print("  Run pharmaplus_proximity_scraper.py first, then re-run this cell.")
    embu_proximity = {}
else:
    prox = pd.read_csv(PROX_CSV)
 
    # Validate
    required_cols = {"facility_id","category","distance_km","is_chain","rating"}
    if not required_cols.issubset(prox.columns):
        raise ValueError(f"Missing columns in proximity CSV: {required_cols - set(prox.columns)}")
 
    # Coerce distance to numeric (some rows may be empty string)
    prox["distance_km"] = pd.to_numeric(prox["distance_km"], errors="coerce")
 
    print(f"  Loaded {len(prox):,} rows")
    print(f"  Facilities: {sorted(prox['facility_id'].unique())}")
    print(f"  Categories: {sorted(prox['category'].unique())}")
 
    # ── Aggregate across both facilities ─────────────────────────────────────
    # Branch 106 sits in Embu town near both Tendri facilities.
    # We use the union of both facility catchments (3km radius) as the
    # demand signal for Branch 106 — deduplicated by place_name.
    prox_dedup = prox.drop_duplicates(subset=["place_name","category"])
 
    RADIUS_KM = 3.0
    prox_near = prox_dedup[prox_dedup["distance_km"] <= RADIUS_KM].copy()
 
    # ── Compute features per search category ─────────────────────────────────
    CAT_MAP = {
        # search category in CSV          → feature prefix
        "gym_fitness":           "gym",
        "beauty_salon_spa":      "beauty_salon",
        "beauty_shop_cosmetics": "beauty_shop",
        "supplements_vitamins":  "supplement_store",
        "bodybuilding_shop":     "bb_shop",
        "pharmacy_competitor":   "pharmacy",
        "supermarket":           "supermarket",
    }
 
    # Max possible counts for normalisation (sensible ceiling per category)
    # Based on what a dense Kenyan town centre might realistically have in 3km
    NORMALISE_CAP = {
        "gym":              10,
        "beauty_salon":     20,
        "beauty_shop":      15,
        "supplement_store": 8,
        "bb_shop":          5,
        "pharmacy":         15,
        "supermarket":      10,
    }
 
    embu_proximity = {}
 
    for csv_cat, feat_prefix in CAT_MAP.items():
        subset = prox_near[prox_near["category"] == csv_cat]
        cap    = NORMALISE_CAP[feat_prefix]
 
        # Count of places (normalised 0-1)
        count        = len(subset)
        count_norm   = round(min(count / cap, 1.0), 4)
 
        # Count of chains (normalised to count)
        chain_count  = int(subset["is_chain"].sum()) if "is_chain" in subset.columns else 0
        chain_norm   = round(min(chain_count / max(count, 1), 1.0), 4)
 
        # Average rating (normalised to 5.0 max)
        ratings      = pd.to_numeric(subset["rating"], errors="coerce").dropna()
        avg_rating   = round(ratings.mean() / 5.0, 4) if len(ratings) > 0 else 0.0
 
        # Nearest distance (normalised: 0 = at door, 1 = at 3km edge)
        distances    = subset["distance_km"].dropna()
        nearest_norm = round(distances.min() / RADIUS_KM, 4) if len(distances) > 0 else 1.0
 
        embu_proximity[f"n_{feat_prefix}_3km"]         = count_norm    # 0-1: how many nearby
        embu_proximity[f"n_{feat_prefix}_chain_3km"]   = chain_norm    # 0-1: chain share
        embu_proximity[f"avg_{feat_prefix}_rating"]    = avg_rating    # 0-1: quality signal
        embu_proximity[f"nearest_{feat_prefix}_km"]    = round(1 - nearest_norm, 4)  # 0-1: inverse distance (1=very close)
 
    # ── Composite demand signals ──────────────────────────────────────────────
    # These combine multiple proximity signals into single demand proxies
    # for each new product category
 
    # Beauty demand signal: beauty salons + beauty shops nearby
    embu_proximity["beauty_demand_proximity"] = round(
        (embu_proximity.get("n_beauty_salon_3km", 0) +
         embu_proximity.get("n_beauty_shop_3km", 0)) / 2, 4
    )
 
    # BB demand signal: gyms + BB shops nearby
    embu_proximity["bb_demand_proximity"] = round(
        (embu_proximity.get("n_gym_3km", 0) +
         embu_proximity.get("n_bb_shop_3km", 0)) / 2, 4
    )
 
    # Supplement demand signal: supplement stores + gyms (gyms drive supplement demand)
    embu_proximity["supplement_demand_proximity"] = round(
        (embu_proximity.get("n_supplement_store_3km", 0) +
         embu_proximity.get("n_gym_3km", 0)) / 2, 4
    )
 
    # Competitive pressure: pharmacies + supermarkets (OTC competition)
    embu_proximity["pharmacy_competition_index"] = round(
        (embu_proximity.get("n_pharmacy_3km", 0) +
         embu_proximity.get("n_supermarket_3km", 0)) / 2, 4
    )
 
    print(f"\n  Proximity features computed ({len(embu_proximity)} features):")
    for feat, val in embu_proximity.items():
        raw_count_key = feat.replace("n_","").replace("_3km","")
        print(f"    {feat:<40} {val}")
 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — UPDATE embu_external
# ══════════════════════════════════════════════════════════════════════════════
print("\nUpdating embu_external...")
 
before = len(embu_external)
 
# Google Trends features
embu_external.update(embu_google_trends)
 
# Proximity features
embu_external.update(embu_proximity)
 
added = len(embu_external) - before
print(f"  embu_external: {before} → {len(embu_external)} keys (+{added} added)")
print(f"  Google Trends: {len(embu_google_trends)} features")
print(f"  Proximity:     {len(embu_proximity)} features")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 FIX — Extend ratio_cols to include GT + proximity features
# Run this BEFORE Step 4 (Build the Embu target profile)
# ══════════════════════════════════════════════════════════════════════════════
 
# Collect all new feature keys that were just added to embu_external
# These come from embu_google_trends and embu_proximity defined in the previous cell
_new_external_features = (
    list(embu_google_trends.keys()) +
    list(embu_proximity.keys())
)
 
# Append to ratio_cols — deduplicated in case any overlap
_before = len(ratio_cols)
ratio_cols = list(dict.fromkeys(ratio_cols + _new_external_features))
_added = len(ratio_cols) - _before
 
print(f"ratio_cols updated: {_before} → {len(ratio_cols)} features (+{_added})")
print(f"  New GT features:        {len(embu_google_trends)}")
print(f"  New proximity features: {len(embu_proximity)}")
print(f"\n  Step 4 will now include all {len(ratio_cols)} features in embu_target.")
print(f"  KNN will use all {len(ratio_cols)} features for similarity scoring.")
 
# Confirm facility_profiles has all new columns (should already be zero-filled)
_missing_in_profiles = [f for f in _new_external_features if f not in facility_profiles.columns]
if _missing_in_profiles:
    print(f"\n  WARNING: {len(_missing_in_profiles)} features missing from facility_profiles — zero-filling now")
    for feat in _missing_in_profiles:
        facility_profiles[feat] = 0.0
else:
    print(f"\n  ✓ All new features present in facility_profiles")
 
 

In [ ]:
# Step 3f:── Data-driven category ↔ burden mapping ─────────────────────────────────────
# Replaces both CATEGORY_CONTEXT and BURDEN_SIGNALS
# Derived from actual co-occurrence of diagnosis groups and dispensing categories
# in the same facility in the same month

# Step 1 — Monthly diagnosis groups per facility
diag_monthly = (
    diag_df.groupby(['facility_id', 'monthly'])['diagnosis_disease_group']
    .apply(list).reset_index()
)

# Step 2 — Monthly dispensing categories per facility
disp_monthly = (
    disp_df.groupby(['facility_id', 'months'])['correct_therapeutic_class']
    .apply(lambda x: list(x.dropna())).reset_index()
    .rename(columns={'months': 'monthly'})
)

# Step 3 — Join on facility + month
paired = diag_monthly.merge(disp_monthly, on=['facility_id', 'monthly'], how='inner')

# Step 4 — For each category, find top co-occurring diagnosis groups
from collections import Counter

CATEGORY_CONTEXT = {}
BURDEN_SIGNALS   = {}

all_categories = disp_df['correct_therapeutic_class'].dropna().unique()
all_burdens    = list(BURDEN_MAP.keys())  # your existing burden dimensions

for category in all_categories:
    # Months where this category was dispensed
    months_with_cat = paired[
        paired['correct_therapeutic_class'].apply(lambda cats: category in cats)
    ]
    if months_with_cat.empty:
        CATEGORY_CONTEXT[category] = 'Insufficient co-occurrence data'
        continue

    # Flatten all diagnosis groups from those months
    all_diags = [
        d for row in months_with_cat['diagnosis_disease_group']
        for d in row if pd.notna(d)
    ]

    # Count and take top 3
    top_diags = Counter(all_diags).most_common(3)
    top_labels = [d[0] for d in top_diags]

    # Map to burden dimensions using BURDEN_MAP keywords
    matched_burdens = []
    for burden_name, keywords in BURDEN_MAP.items():
        if any(any(kw.lower() in diag.lower() for kw in keywords)
               for diag in top_labels):
            matched_burdens.append(burden_name.replace('_',' ').title())

    CATEGORY_CONTEXT[category] = (
        ' / '.join(matched_burdens) if matched_burdens
        else top_labels[0] if top_labels
        else 'Review therapeutic class data'
    )

# Step 5 — Reverse: for each burden, find top co-occurring categories
for burden_name, keywords in BURDEN_MAP.items():
    # Months where this burden was present in diagnoses
    months_with_burden = paired[
        paired['diagnosis_disease_group'].apply(
            lambda diags: any(
                any(kw.lower() in d.lower() for kw in keywords)
                for d in diags if pd.notna(d)
            )
        )
    ]
    if months_with_burden.empty:
        BURDEN_SIGNALS[burden_name] = []
        continue

    # Flatten all categories dispensed in those months
    all_cats = [
        c for row in months_with_burden['correct_therapeutic_class']
        for c in row if pd.notna(c)
    ]

    # Top 4 categories
    top_cats = [c for c, _ in Counter(all_cats).most_common(4)]
    BURDEN_SIGNALS[burden_name] = top_cats

print("=== Data-derived CATEGORY_CONTEXT ===")
for cat, context in sorted(CATEGORY_CONTEXT.items()):
    print(f"  {cat:<35} {context}")

print("\n=== Data-derived BURDEN_SIGNALS ===")
for burden, cats in sorted(BURDEN_SIGNALS.items()):
    print(f"  {burden:<20} {cats}")

=== Data-derived CATEGORY_CONTEXT ===
  Acetylcholinesterase inhibitor      Communicable / Musculoskeletal / Dermatological
  Adsorbent / antidiarrhoeal          Communicable / Musculoskeletal / Dermatological
  Alkylating agent                    Communicable / Respiratory / Musculoskeletal
  Alpha-blocker + 5-alpha-reductase   Communicable / Respiratory / Musculoskeletal
  Alpha-blocker combination           Communicable / Musculoskeletal / Dermatological
  Alpha-blocker — uroselective        Communicable / Musculoskeletal
  Analgesic / antipyretic             Communicable / Musculoskeletal
  Antacid                             Communicable / Musculoskeletal / Dermatological
  Antacid / GI preparation            Communicable / Respiratory / Musculoskeletal
  Antacid / alginate                  Communicable / Musculoskeletal
  Antacid suspension                  Communicable / Musculoskeletal
  Anthelmintic                        Communicable / Musculoskeletal
  Anthelmintic / immunom

---
## 4. Build the Embu target profile

Branch #106 is a **new branch** — it has no data in the system.  
We can't query it. We can't compare it to anything.

Instead, we build the **Embu catchment profile** from what we already know:  
all existing Tendri branches are in Embu, so the average of their profiles  
is our best estimate of what the Embu patient population looks like.

This is the target vector we use in KNN — not Branch #106 itself.


In [ ]:
# ── Step 1: Add external DHS features to facility_profiles (existing = 0) ────
external_feature_names = list(embu_external.keys())
for feat in external_feature_names:
    facility_profiles[feat] = 0

# ── FIX: Scale DHS % values to 0-1 to match ratio scale ─────────────────────
# NEW — without this, values like 75.0 dominate KNN distances
for feat in external_feature_names:
    val = embu_external[feat]
    if val is not None and val > 1:
        embu_external[feat] = val / 100

# ── Step 2: Add external + cat_ratio cols to ratio_cols ──────────────────────
ratio_cols = ratio_cols + external_feature_names
ratio_cols = list(dict.fromkeys(ratio_cols))  # remove any accidental duplicates

# ── FIX: Weighted mean — larger branches contribute more ─────────────────────
# CHANGED from facility_profiles[ratio_cols].mean()
weights = facility_profiles['avg_monthly_consultations'].fillna(1)
embu_target = facility_profiles[ratio_cols].apply(
    lambda col: (col * weights).sum() / weights.sum()
)

# ── Step 3: Override with actual Embu values for Branch 106 ──────────────────
for feat, val in embu_external.items():
    embu_target[feat] = val if val is not None else 0

# Set Branch 106 proximity from nearby_all results
embu_target['n_hospitals_25km']    = (nearby_all['amenity'] == 'hospital').sum()
embu_target['n_clinics_25km']      = nearby_all['amenity'].isin(['clinic','doctors']).sum()
embu_target['n_pharmacies_25km']   = (nearby_all['amenity'] == 'pharmacy').sum()
embu_target['n_facilities_25km']   = len(nearby_all)
embu_target['has_hospital_nearby'] = 1

# Set category mix for Branch 106 = mean of existing branches
for col in cat_ratio_cols:
    embu_target[col] = facility_profiles[col].mean()

print("Embu catchment profile — top disease burden dimensions:")
print()
burden_ratios = {
    k.replace('burden_', '').replace('_ratio', ''): round(v, 3)
    for k, v in embu_target.items()
    if 'burden_' in k and v > 0
}
for name, score in sorted(burden_ratios.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(score * 40)
    print(f"  {name:<20} {score:.3f}  {bar}")

Embu catchment profile — top disease burden dimensions:

  communicable         0.267  ██████████
  respiratory          0.194  ███████
  musculoskeletal      0.136  █████
  gi                   0.089  ███
  gynaecological       0.080  ███
  hypertension         0.079  ███
  dermatological       0.063  ██
  diabetes             0.054  ██
  maternal             0.030  █
  cardiovascular       0.008  


In [ ]:
facility_profiles.head()

,facility_id,burden_malaria,burden_communicable,burden_hypertension,burden_cardiovascular,burden_diabetes,burden_respiratory,burden_gi,burden_maternal,burden_gynaecological,...,u5_stunting_pct,u5_underweight_pct,vaccination_pct,itn_access_pct,itn_use_pct,clean_fuel_access_pct,water_access_pct,sanitation_access_pct,women_no_education_pct,is_urban_branch
0,4,0,5281,1997,187,1313,4131,1977,594,1727,...,0,0,0,0,0,0,0,0,0,0
1,5,0,2943,502,59,368,1862,760,326,763,...,0,0,0,0,0,0,0,0,0,0
2,7,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


---
## 5. Compute steady-state dispensing

We want to know what a mature Tendri branch dispenses per month on average —  
not what it dispensed in its early months when it was still finding its feet.

So we:
1. Rank each facility's months from 1 (first month) to N (latest month)
2. Exclude the first `RAMP_UP_MONTHS` months
3. Average what remains → this is the **steady-state** dispensing

We also separately pull Month 1 data for each branch, which we'll use  
in the next step to compute the penetration factor.


In [ ]:
# Step 5a: Rank months per facility chronologically
# This tells us which month was the "1st month", "2nd month", etc. for each facility

disp_work = disp_df.copy()

# .groupby('facility_id')['months'] groups months by facility
# .rank(method='dense') assigns rank 1 to the earliest month, 2 to next, etc.
# 'dense' means no gaps in ranking (unlike 'average' which handles ties differently)
disp_work['month_rank'] = (
    disp_work
    .groupby('facility_id')['months']
    .rank(method='dense')
    .astype(int)
)

print("Month ranks assigned. Sample:")
disp_work[['facility_id', 'months', 'month_rank', 
            'correct_therapeutic_class', 'total_qty_dispensed']].head(10)


Month ranks assigned. Sample:


,facility_id,months,month_rank,correct_therapeutic_class,total_qty_dispensed
0,4,2019-09-01,1,Ophthalmic — fluoroquinolone,2.0
1,4,2019-09-01,1,NSAID,60.0
2,4,2019-09-01,1,Antibiotic — beta-lactam combination,1.0
3,4,2019-09-01,1,Antacid,15.0
4,4,2019-09-01,1,Antiviral — topical/oral/IV,48.0
5,4,2019-09-01,1,Antiviral — topical,20.0
6,4,2019-09-01,1,Antiviral — topical/oral/IV,20.0
7,4,2019-09-01,1,None,1.0
8,4,2019-09-01,1,None,45.0
9,4,2019-09-01,1,Anthelmintic,24.0


In [ ]:
# Step 5b: Pull Month 1 data separately
# We need this to calculate the penetration factor later
# (how does Month 1 volume compare to steady-state?)

month1 = (
    disp_work[disp_work['month_rank'] == 1]    # filter to first month only
    .groupby(['facility_id', 'correct_therapeutic_class'])
    .agg(month1_qty=('total_qty_dispensed', 'sum'))
    .reset_index()
)

print(f"Month 1 data: {len(month1)} facility-category combinations")
month1.head(8)


Month 1 data: 316 facility-category combinations


,facility_id,correct_therapeutic_class,month1_qty
0,4,Analgesic / antipyretic,3939.0
1,4,Antacid,17.0
2,4,Antacid / alginate,5.0
3,4,Anthelmintic,275.0
4,4,Anti-anginal — ranolazine,210.0
5,4,Antibiotic / antiprotozoal,6180.0
6,4,Antibiotic — aminoglycoside,309.0
7,4,Antibiotic — aminopenicillin,4011.0


In [ ]:
# Step 5c: Compute steady-state (exclude ramp-up months)

steady_df = disp_work[disp_work['month_rank'] > RAMP_UP_MONTHS]

if steady_df.empty:
    print(f"WARNING: No data beyond month {RAMP_UP_MONTHS}.")
    print("Not enough history to exclude ramp-up months.")
    print("Using all months as steady state. Penetration factor will use default.")
    steady_df = disp_work.copy()
else:
    print(f"Using months {RAMP_UP_MONTHS+1}+ as steady state.")

# Average dispensing per facility per category across all steady-state months
steady_agg = (
    steady_df
    .groupby(['facility_id', 'correct_therapeutic_class'])
    .agg(
        avg_monthly_qty     = ('total_qty_dispensed', 'mean'),   # avg quantity per month
        avg_unique_patients = ('unique_patients',     'mean'),   # avg patients per month
        months_of_data      = ('months',              'nunique') # how many months went in
    )
    .reset_index()
)

print(f"Steady-state table: {len(steady_agg)} rows")
print(f"  {steady_agg['facility_id'].nunique()} facilities")
print(f"  {steady_agg['correct_therapeutic_class'].nunique()} product categories")
steady_agg.head(10)


Using months 4+ as steady state.
Steady-state table: 581 rows
  2 facilities
  324 product categories


,facility_id,correct_therapeutic_class,avg_monthly_qty,avg_unique_patients,months_of_data
0,4,Acetylcholinesterase inhibitor,83.461538,1.153846,13
1,4,Adsorbent / antidiarrhoeal,15.500000,2.000000,12
2,4,Alkylating agent,2.500000,2.125000,7
3,4,Alpha-blocker + 5-alpha-reductase,40.000000,2.117647,17
4,4,Alpha-blocker combination,30.166667,1.333333,6
5,4,Alpha-blocker — uroselective,38.490196,2.000000,30
6,4,Analgesic / antipyretic,1018.871728,72.015707,32
7,4,Antacid,75.285714,64.040816,31
8,4,Antacid / GI preparation,487.645161,12.000000,12
9,4,Antacid / alginate,17.717391,12.021739,28


---
## 6. Compute the penetration factor

A new branch doesn't immediately sell at full capacity.  
In Month 1, patients are still discovering the branch, staff are settling in,  
referral patterns haven't been established yet.

The **penetration factor** answers: *what fraction of mature monthly volume  
does a branch typically sell in its very first month?*

We compute it from existing branches that have both Month 1 data and  
steady-state data (4+ months of history).

> If we don't have enough history, we fall back to **0.35** —  
> meaning we assume Branch #106 will sell about 35% of its  
> eventual mature monthly volume in Month 1. This is conservative  
> and protects against dead stock.


In [ ]:
# Join Month 1 qty to steady-state avg for the same facility-category
merged = steady_agg.merge(
    month1, 
    on=['facility_id', 'correct_therapeutic_class'], 
    how='inner'   # only keep rows that exist in BOTH tables
)

# Penetration ratio = Month 1 qty / steady-state avg
# If a branch sold 30 units in Month 1 and averages 100 units in steady state,
# the penetration ratio is 0.30 (30%)
merged['penetration_ratio'] = (
    merged['month1_qty'] / merged['avg_monthly_qty'].replace(0, np.nan)
)

# Use the median ratio across all facility-category combinations
# Median is more robust than mean — less affected by outliers
penetration_factor = merged['penetration_ratio'].dropna().median()
pf_is_default = False

# Sanity check: if factor > 2, something is wrong with the data
# (usually means we only have 1 month, so month1 == steady state)
if pd.isna(penetration_factor) or penetration_factor <= 0 or penetration_factor > 2:
    print("WARNING: Cannot compute a reliable penetration factor.")
    print(f"  Computed value: {penetration_factor}")
    print("  Reason: insufficient history (need 4+ months per facility)")
    print("  Falling back to 0.35 (conservative default)")
    penetration_factor = 0.35
    pf_is_default = True
else:
    print(f"Penetration factor: {penetration_factor:.2f}")
    print(f"  → Branch #106 is expected to sell {penetration_factor*100:.0f}% "
          f"of mature monthly volume in Month 1")
    pf_is_default = False


Penetration factor: 1.39
  → Branch #106 is expected to sell 139% of mature monthly volume in Month 1


---
## 7. Rank facilities by similarity to the Embu profile (KNN)

We have the Embu target profile (what Branch #106's catchment looks like)  
and we have a profile for each existing Tendri branch.

**KNN (K-Nearest Neighbours)** finds which existing branches are most similar  
to the Embu target. We then weight their dispensing patterns accordingly —  
a branch whose patient mix closely matches the Embu profile gets more weight  
than a branch that's quite different.

**Steps:**
1. Standardise all feature values (so a large number like "500 consultations"  
   doesn't dominate over a small ratio like "0.17 hypertension rate")
2. Find the K branches with the smallest cosine distance to the Embu target
3. Weight them by inverse distance (closer = higher weight)
4. Check if the spread is meaningful — if all branches score almost the same,  
   equal weights are just as good as KNN weights


In [ ]:
facility_profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 90 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   facility_id                          3 non-null      int64  
 1   burden_malaria                       3 non-null      int64  
 2   burden_communicable                  3 non-null      int64  
 3   burden_hypertension                  3 non-null      int64  
 4   burden_cardiovascular                3 non-null      int64  
 5   burden_diabetes                      3 non-null      int64  
 6   burden_respiratory                   3 non-null      int64  
 7   burden_gi                            3 non-null      int64  
 8   burden_maternal                      3 non-null      int64  
 9   burden_gynaecological                3 non-null      int64  
 10  burden_musculoskeletal               3 non-null      int64  
 11  burden_dermatological               

In [ ]:
# Step 7a: Prepare the feature matrix
# Rows = facilities, Columns = ratio features

profiles_indexed = facility_profiles.set_index('facility_id')
X = profiles_indexed[ratio_cols].fillna(0)   # fill NaN with 0 (missing = no signal)

print(f"Feature matrix: {X.shape[0]} facilities × {X.shape[1]} features")
print()
print("Sample (first 5 features):")
X.iloc[:, :5].round(3)


Feature matrix: 3 facilities × 61 features

Sample (first 5 features):


,burden_malaria_ratio,burden_communicable_ratio,burden_hypertension_ratio,burden_cardiovascular_ratio,burden_diabetes_ratio
facility_id,,,,,
4,0.0,0.239,0.090,0.008,0.059
5,0.0,0.330,0.056,0.007,0.041
7,0.0,0.000,0.000,0.000,0.000


In [ ]:
# Step 7b: Standardise the features
# StandardScaler transforms each feature so it has mean=0 and std=1
# This prevents features with large absolute values from dominating the distance calculation
# e.g. "avg_monthly_consultations = 500" vs "burden_hypertension_ratio = 0.17"
# Without scaling, the consultation count would dominate every distance calculation

scaler = StandardScaler()

# fit_transform: learn the mean and std of each column, then apply the transformation
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

print("Features standardised (mean=0, std=1 per column)")
print()
print("Before scaling — consultation count range:")
print(f"  min: {X['avg_monthly_consultations'].min():.1f}, "
      f"  max: {X['avg_monthly_consultations'].max():.1f}")
print()
print("After scaling — consultation count range:")
print(f"  min: {X_scaled['avg_monthly_consultations'].min():.2f}, "
      f"  max: {X_scaled['avg_monthly_consultations'].max():.2f}")


Features standardised (mean=0, std=1 per column)

Before scaling — consultation count range:
  min: 2.0,   max: 1658.2

After scaling — consultation count range:
  min: -1.19,   max: 1.25


In [ ]:
# Step 7c: Standardise the Embu target vector using the SAME scaler
# Important: we use transform() not fit_transform() here
# fit_transform() would learn new mean/std from the target — wrong
# transform() applies the mean/std already learned from the facility data — correct

embu_vector = embu_target.reindex(ratio_cols).fillna(0).values.reshape(1, -1)
embu_scaled = scaler.transform(embu_vector)[0]

print(f"Embu target vector standardised ({len(embu_scaled)} dimensions)")


Embu target vector standardised (61 dimensions)


In [ ]:
# Step 7d: Fit KNN and find most similar facilities

n_facilities = len(X_scaled)
k = min(KNN_K, n_facilities)   # can't have more neighbours than facilities

if n_facilities < 2:
    print("Only 1 facility in the data — KNN not applicable, using it with full weight")
    weight_map = {X_scaled.index[0]: 1.0}
    used_fallback = False
else:
    # NearestNeighbors finds the k closest points in the feature space
    # metric='cosine' measures the angle between vectors (good for ratio data)
    # Alternative: metric='euclidean' measures straight-line distance
    knn = NearestNeighbors(n_neighbors=k, metric='cosine')
    knn.fit(X_scaled)   # learn the positions of all facilities in feature space

    # kneighbors returns:
    #   distances — how far each neighbour is from the target (0 = identical, 1 = opposite)
    #   indices   — which row numbers in X_scaled those neighbours are
    distances, indices = knn.kneighbors([embu_scaled])
    distances = distances[0]   # unwrap from array-of-arrays
    indices   = indices[0]

    # Convert row indices back to facility IDs
    top_facility_ids = [X_scaled.index[i] for i in indices]

    print(f"Top {k} facilities most similar to the Embu profile:")
    print(f"{'Facility ID':<15} {'Cosine Distance':<20} {'Interpretation'}")
    print("-" * 55)
    for fid, dist in zip(top_facility_ids, distances):
        interp = "Very similar" if dist < 0.1 else "Similar" if dist < 0.3 else "Moderately similar"
        print(f"{fid:<15} {dist:<20.4f} {interp}")


Top 3 facilities most similar to the Embu profile:
Facility ID     Cosine Distance      Interpretation
-------------------------------------------------------
4               0.3171               Moderately similar
5               0.4025               Moderately similar
7               1.7862               Moderately similar


In [ ]:
# Step 7e: Assign weights — closer to Embu profile = higher weight
# Check first whether the spread is meaningful

if n_facilities >= 2:
    spread = distances.max() - distances.min()
    print(f"Similarity spread: {spread:.4f}  (threshold: {KNN_FALLBACK_THRESHOLD})")
    print()

    if spread < KNN_FALLBACK_THRESHOLD:
        # All facilities are almost equally similar to Embu
        # KNN weighting adds no value here — use equal weights instead
        print("Spread too narrow → all facilities equally representative of Embu.")
        print("Using equal weights (fallback).")
        weight_map    = {fid: 1.0 / n_facilities for fid in X_scaled.index}
        used_fallback = True
    else:
        # Inverse distance weighting:
        # weight = 1 / distance
        # A facility with distance 0.1 gets weight 10, one with distance 0.5 gets weight 2
        # We then normalise so all weights sum to 1
        weights = 1 / (distances + 1e-6)   # 1e-6 avoids division by zero
        weights = weights / weights.sum()   # normalise to sum=1

        weight_map    = dict(zip(top_facility_ids, weights))
        used_fallback = False

        print("Weights assigned:")
        for fid, w in weight_map.items():
            print(f"  facility_id={fid}  weight={w:.3f}  ({w*100:.1f}% influence)")


Similarity spread: 1.4691  (threshold: 0.05)

Weights assigned:
  facility_id=4  weight=0.509  (50.9% influence)
  facility_id=5  weight=0.401  (40.1% influence)
  facility_id=7  weight=0.090  (9.0% influence)


---
## 8. Predict opening stock for Branch #106

Now we combine everything:
1. Take the weighted average of steady-state dispensing from the most similar facilities
2. Multiply by the penetration factor to size the opening stock conservatively
3. Flag any categories where we're recommending more than 1.5x monthly velocity (dead stock risk)
4. Assign confidence levels based on data quality


In [ ]:
steady_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 581 entries, 0 to 580
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   facility_id                581 non-null    int64  
 1   correct_therapeutic_class  581 non-null    object 
 2   avg_monthly_qty            581 non-null    float64
 3   avg_unique_patients        581 non-null    float64
 4   months_of_data             581 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 22.8+ KB


In [ ]:
# Step 8a: Weighted average of steady-state dispensing

# Filter to only the facilities that have weights assigned
# (if fallback was used, this is all facilities)
df_pred = steady_agg[
    steady_agg['facility_id'].isin(weight_map.keys())
].copy()

# Map each facility's weight onto the rows
df_pred['weight'] = df_pred['facility_id'].map(weight_map).fillna(0)

# Weighted quantity = each facility's avg qty × its weight
# Summing these per category gives us the weighted average
df_pred['weighted_qty'] = df_pred['avg_monthly_qty'] * df_pred['weight']

print("Weighted quantities per facility per category:")
df_pred[['facility_id', 'correct_therapeutic_class', 
          'avg_monthly_qty', 'weight', 'weighted_qty']].head(10).round(3)


Weighted quantities per facility per category:


,facility_id,correct_therapeutic_class,avg_monthly_qty,weight,weighted_qty
0,4,Acetylcholinesterase inhibitor,83.462,0.509,42.467
1,4,Adsorbent / antidiarrhoeal,15.500,0.509,7.887
2,4,Alkylating agent,2.500,0.509,1.272
3,4,Alpha-blocker + 5-alpha-reductase,40.000,0.509,20.353
4,4,Alpha-blocker combination,30.167,0.509,15.349
5,4,Alpha-blocker — uroselective,38.490,0.509,19.585
6,4,Analgesic / antipyretic,1018.872,0.509,518.425
7,4,Antacid,75.286,0.509,38.307
8,4,Antacid / GI preparation,487.645,0.509,248.125
9,4,Antacid / alginate,17.717,0.509,9.015


In [ ]:
# Step 8b: Aggregate to category level

prediction = (
    df_pred
    .groupby('correct_therapeutic_class')
    .agg(
        # Sum of weighted qtys = weighted average across facilities
        predicted_steady_state    = ('weighted_qty',        'sum'),
        avg_unique_patients       = ('avg_unique_patients', 'mean'),
        n_facilities_contributing = ('facility_id',         'nunique')
    )
    .reset_index()
)

print("Predicted steady-state monthly volume per category:")
prediction[['correct_therapeutic_class', 'predicted_steady_state',
             'n_facilities_contributing']].round(1)


Predicted steady-state monthly volume per category:


,correct_therapeutic_class,predicted_steady_state,n_facilities_contributing
0,Acetylcholinesterase inhibitor,83.0,2
1,Adsorbent / antidiarrhoeal,13.9,2
2,Alkylating agent,1.3,1
3,Alpha-blocker + 5-alpha-reductase,32.4,2
4,Alpha-blocker combination,24.2,2
...,...,...,...
319,Vitamin supplement,516.8,2
320,Wound debriding agent,1.9,1
321,Wound healing — topical,0.4,1
322,Wound hydrogel,0.5,1


In [ ]:
# Step 8c: Apply penetration factor → opening stock quantity

# Opening stock = what a mature branch would sell in a month × how much a new branch sells
# We round to whole units (you can't stock half a tablet box)
prediction['opening_stock_qty'] = (
    prediction['predicted_steady_state'] * penetration_factor
).round().astype(int)

print(f"Penetration factor applied: {penetration_factor:.2f}")
print()
prediction[['correct_therapeutic_class', 'predicted_steady_state', 
             'opening_stock_qty']].round(1)


Penetration factor applied: 1.39



,correct_therapeutic_class,predicted_steady_state,opening_stock_qty
0,Acetylcholinesterase inhibitor,83.0,115
1,Adsorbent / antidiarrhoeal,13.9,19
2,Alkylating agent,1.3,2
3,Alpha-blocker + 5-alpha-reductase,32.4,45
4,Alpha-blocker combination,24.2,34
...,...,...,...
319,Vitamin supplement,516.8,717
320,Wound debriding agent,1.9,3
321,Wound healing — topical,0.4,1
322,Wound hydrogel,0.5,1


In [ ]:
# Step 8d: Dead stock risk flag

# If opening stock qty > DEAD_STOCK_MULTIPLIER × monthly velocity,
# it means you'd need more than 1.5 months to sell through just the opening stock
# That's a dead stock risk — you might be over-ordering
prediction['dead_stock_risk'] = (
    prediction['opening_stock_qty'] >
    prediction['predicted_steady_state'] * DEAD_STOCK_MULTIPLIER
)

risky = prediction[prediction['dead_stock_risk']]
if len(risky) > 0:
    print(f"⚠️  {len(risky)} categories flagged as dead stock risk:")
    print(risky[['correct_therapeutic_class', 'opening_stock_qty', 
                  'predicted_steady_state']].round(1).to_string(index=False))
else:
    print("✓ No dead stock risk flags")


⚠️  29 categories flagged as dead stock risk:
                  correct_therapeutic_class  opening_stock_qty  predicted_steady_state
                           Alkylating agent                  2                     1.3
             Anthelmintic / immunomodulator                  1                     0.4
                     Antibiotic — rifamycin                  2                     1.2
                       Antifungal — topical                  3                     1.9
                              Antihistamine                  2                     1.2
                              Antipsychotic                  1                     0.5
           Antipsychotic — injectable depot                  1                     0.6
                Bisphosphonate — injectable                  1                     0.6
Bronchodilator + corticosteroid combination                  5                     3.2
                      Bronchodilator — LABA                  4                     2

In [ ]:
# Step 8e: Confidence levels + diagnosis context + model reasoning
# ─────────────────────────────────────────────────────────────────
# Confidence logic (updated):
#   High   — 2+ facilities, real penetration factor, 6+ months of data, KNN not fallback
#   Medium — KNN fallback used OR fewer than 6 months of data
#   Low    — penetration factor is default 0.35 OR fewer than 2 facilities


# Merge average months of data per category
months_avg = (
    steady_agg.groupby('correct_therapeutic_class')['months_of_data']
    .mean().reset_index()
    .rename(columns={'months_of_data': 'months_avg'})
)
prediction = prediction.merge(months_avg, on='correct_therapeutic_class', how='left')

prediction['confidence'] = prediction.apply(
    lambda row: (
        'Low'    if pf_is_default or row['n_facilities_contributing'] < 2
        else 'High'   if not used_fallback and row.get('months_avg', 0) >= 6
        else 'Medium'
    ), axis=1
)

# Diagnosis context — which burden drives demand for each category
# CATEGORY_CONTEXT = {
#     'Oral Solid Forms':         'Mixed — review therapeutic class breakdown',
#     'Injectables':              'Communicable / Maternal / Cardiovascular',
#     'Oral Liquid Forms':        'Paediatric / Respiratory / GI',
#     'Eye Medications':          'Communicable / NCD',
#     'Topical Preparations':     'Dermatological / MSK',
#     'IV Fluids & Infusions':    'Communicable / Maternal / Surgical',
#     'Vaccines & Biologicals':   'Immunisation / Maternal',
#     'Ear & Nasal Preparations': 'Respiratory / ENT',
#     'Inhalation Products':      'Respiratory / Asthma',
#     'Wound Care':               'Surgical / Communicable',
#     'Infection Control':        'Facility operations',
# }

prediction['diagnosis_driver'] = (
    prediction['correct_therapeutic_class']
    .map(CATEGORY_CONTEXT)
    .fillna('Review therapeutic class data')
)

# Model reasoning — human-readable explanation per category
# BURDEN_SIGNALS = {
#     'malaria':      ['Injectables', 'Oral Solid Forms', 'Oral Liquid Forms'],
#     'maternal':     ['Injectables', 'IV Fluids & Infusions', 'Oral Solid Forms'],
#     'hypertension': ['Oral Solid Forms'],
#     'diabetes':     ['Oral Solid Forms'],
#     'respiratory':  ['Oral Liquid Forms', 'Inhalation Products', 'Oral Solid Forms'],
#     'communicable': ['Injectables', 'Oral Solid Forms', 'Oral Liquid Forms'],
# }

top_burden = sorted(
    [(k.replace('burden_','').replace('_ratio',''), v)
     for k, v in embu_target.items() if 'burden_' in k and v > 0],
    key=lambda x: x[1], reverse=True
)[:3]

def build_reasoning(row):
    reasons = []
    category = row['correct_therapeutic_class']

    # Signal 1 — Proximity
    drivers = category_proximity_driver.get(category, [])
    if drivers:
        n = len(drivers)
        nearest = drivers[0]
        reasons.append(f"{n} nearby {'facility' if n==1 else 'facilities'} — {nearest}")

    # Signal 2 — Disease burden
    for burden_name, burden_score in top_burden:
        if category in BURDEN_SIGNALS.get(burden_name, []) and burden_score > 0.1:
            reasons.append(f"High {burden_name} catchment burden ({burden_score:.0%})")
            break

    # Signal 3 — Age mix
    if embu_target.get('pct_chronic_age', 0) > 0.25 and category == 'Oral Solid Forms':
        reasons.append(f"High adult 45+ population ({embu_target['pct_chronic_age']:.0%})")
    if embu_target.get('pct_paediatric', 0) > 0.20 and category in ['Oral Liquid Forms', 'Vaccines & Biologicals']:
        reasons.append(f"High paediatric population ({embu_target['pct_paediatric']:.0%})")

    # Signal 4 — Comparable branch
    if not used_fallback and row['n_facilities_contributing'] >= 2:
        top_fac = max(weight_map, key=weight_map.get)
        reasons.append(f"Comparable to Branch {top_fac} Month 1 pattern")

    # Signal 5 — Conservative flag
    if row['dead_stock_risk']:
        return "Low cross-branch demand — order conservatively"
    if row['confidence'] == 'Low':
        reasons.append("Insufficient history — conservative estimate")

    if not reasons:
        return "Based on Embu catchment profile"
    return ' — '.join(reasons[:2])

prediction['model_reasoning'] = prediction.apply(build_reasoning, axis=1)

print("Confidence levels, diagnosis context and model reasoning added")
print(prediction[['correct_therapeutic_class','confidence',
                   'months_avg','model_reasoning']].to_string())

Confidence levels, diagnosis context and model reasoning added
                       correct_therapeutic_class confidence  months_avg                                                                    model_reasoning
0                 Acetylcholinesterase inhibitor       High         7.0                                             Comparable to Branch 4 Month 1 pattern
1                     Adsorbent / antidiarrhoeal       High         6.5                                             Comparable to Branch 4 Month 1 pattern
2                               Alkylating agent        Low         7.0                                     Low cross-branch demand — order conservatively
3              Alpha-blocker + 5-alpha-reductase       High         9.0                                             Comparable to Branch 4 Month 1 pattern
4                      Alpha-blocker combination     Medium         4.0                                             Comparable to Branch 4 Month 1 pattern
5      

##  9.Use Bayesian Confidence Intervals

In [ ]:
# Map from new_category_name to GT category index key
# These must match the 'category' values in embu_trends_category_index
GT_CATEGORY_MAP = {
    "Beauty Products":         "beauty",
    "Vitamins & Supplements":  "supplements",
    "Body Building":           "bodybuilding",
}
 
def compute_gt_weights(category_name, steady_df_subset):
    """
    Returns a weight array aligned to steady_df_subset rows.
    For GT-linked categories: weight = GT index for that month (floored at 0.15).
    For all other categories: uniform weights (all 1.0).
    """
    if embu_trends_category_index.empty:
        return np.ones(len(steady_df_subset))
 
    gt_key = GT_CATEGORY_MAP.get(category_name)
    if gt_key is None:
        return np.ones(len(steady_df_subset))
 
    # Get GT index for this category
    gt_cat = embu_trends_category_index[
        embu_trends_category_index["category"] == gt_key
    ][["month", "category_index"]].copy()
 
    if gt_cat.empty:
        return np.ones(len(steady_df_subset))
 
    gt_cat = gt_cat.set_index("month")["category_index"]
 
    # Align weights to steady_df months
    weights = []
    for _, row in steady_df_subset.iterrows():
        month = pd.to_datetime(row["months"]).to_period("M").to_timestamp()
        w = gt_cat.get(month, None)
        if w is None:
            # Try nearest month fallback
            w = gt_cat.iloc[-1] if len(gt_cat) > 0 else 1.0
        weights.append(max(float(w), 0.15))  # floor at 15%
 
    return np.array(weights)
 
 
def compute_credible_intervals_gt(prediction, steady_df, penetration_factor,
                                   confidence_level=0.90):
    """
    GT-weighted Bayesian credible intervals.
    Drop-in replacement for the original Bayesian CI block in Step 9.
 
    For new categories (Beauty/Supplements/BB):
      - Weights observations by GT monthly index
      - Fits a weighted normal distribution
      - Computes CI from weighted mean/std
    For pharma categories:
      - Identical to the original unweighted logic
    """
    credible_intervals = []
 
    for _, row in prediction.iterrows():
        category = row["correct_therapeutic_class"]
        point    = row["opening_stock_qty"]
 
        # Get observations from steady_df for this category
        obs_df = steady_df[
            steady_df["correct_therapeutic_class"] == category
        ][["months", "total_qty_dispensed"]].dropna()
 
        obs = obs_df["total_qty_dispensed"].values
 
        if len(obs) < 2:
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            credible_intervals.append({
                "correct_therapeutic_class": category,
                "ci_lower":       lo,
                "ci_upper":       hi,
                "ci_method":      "fallback ±40%",
                "n_observations": len(obs),
            })
            continue
 
        # ── GT weighting for new categories ──────────────────────────────────
        weights = compute_gt_weights(category, obs_df)
 
        # Weighted mean and std
        w_sum    = weights.sum()
        w_mean   = (weights * obs).sum() / w_sum
        w_var    = (weights * (obs - w_mean) ** 2).sum() / w_sum
        w_std    = np.sqrt(w_var)
 
        # Guard against degenerate cases
        if w_std <= 0 or np.isnan(w_std) or np.isnan(w_mean):
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            credible_intervals.append({
                "correct_therapeutic_class": category,
                "ci_lower":       lo,
                "ci_upper":       hi,
                "ci_method":      "fallback ±40% (zero variance)",
                "n_observations": len(obs),
            })
            continue
 
        # Apply penetration factor
        mu_m1  = w_mean * penetration_factor
        std_m1 = w_std  * penetration_factor
 
        alpha  = (1 - confidence_level) / 2
        lo_f   = stats.norm.ppf(alpha,     loc=mu_m1, scale=std_m1)
        hi_f   = stats.norm.ppf(1 - alpha, loc=mu_m1, scale=std_m1)
 
        if np.isnan(lo_f) or np.isnan(hi_f):
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            method = "fallback ±40% (ppf NaN)"
        else:
            lo = max(0, round(lo_f))
            hi = round(hi_f)
            gt_key = GT_CATEGORY_MAP.get(category)
            method = (
                f"GT-weighted normal (n={len(obs)})" if gt_key
                else f"normal fit (n={len(obs)} months)"
            )
 
        credible_intervals.append({
            "correct_therapeutic_class": category,
            "ci_lower":       lo,
            "ci_upper":       hi,
            "ci_method":      method,
            "n_observations": len(obs),
        })
 
    return pd.DataFrame(credible_intervals)
 
 
print("\n✓ compute_credible_intervals_gt() defined")
print("  Use this in Step 9 instead of the original CI block:")
print("  ci_df = compute_credible_intervals_gt(prediction, steady_df, penetration_factor)")
 

KeyError: 'ci_lower'

In [ ]:
def external_multiplier(product_name, embu_target):
    """
    Adjusts product share within its category based on external signals.
    Higher multiplier = boost this product's share in the opening stock allocation.
 
    Sources:
      Original:  hospital proximity, malaria/maternal/paediatric/NCD burden
      New:       beauty/BB/supplement demand proximity, GT momentum per category
    """
    n = str(product_name).lower()
    multiplier = 1.0
 
    # ── ORIGINAL SIGNALS (pharma — unchanged) ────────────────────────────────
 
    # Hospital proximity boosts injectables and IV products
    has_hospital = embu_target.get("has_hospital_nearby", 0)
    if has_hospital and any(k in n for k in ["injection","injectable","iv ","infusion","vial","ampoule"]):
        multiplier *= 1.3
 
    # Malaria burden boosts ACTs and antimalarials
    malaria_burden = embu_target.get("burden_malaria_ratio", 0)
    if malaria_burden > 0.1 and any(k in n for k in ["artemether","coartem","lumefantrine","quinine","fansidar","sp "]):
        multiplier *= (1 + malaria_burden * 2)
 
    # Maternal burden boosts ANC products
    maternal_burden = embu_target.get("burden_maternal_ratio", 0)
    antenatal_pct   = embu_target.get("antenatal_4plus_visits_pct", 0)
    if (maternal_burden > 0.05 or antenatal_pct > 0.5) and \
       any(k in n for k in ["folic","ferrous","antenatal","oxytocin","magnesium","prenatal"]):
        multiplier *= 1.25
 
    # Paediatric mix boosts liquid formulations
    pct_paediatric = embu_target.get("pct_paediatric", 0)
    if pct_paediatric > 0.2 and any(k in n for k in ["syrup","suspension","paediatric","paed","infant","drops","125mg","250mg/5"]):
        multiplier *= (1 + pct_paediatric)
 
    # NCD burden boosts chronic disease products
    hypert_burden = embu_target.get("burden_hypertension_ratio", 0)
    diab_burden   = embu_target.get("burden_diabetes_ratio", 0)
    if hypert_burden > 0.05 and any(k in n for k in ["amlodipine","atenolol","losartan","lisinopril","ramipril","nifedipine"]):
        multiplier *= (1 + hypert_burden * 1.5)
    if diab_burden > 0.05 and any(k in n for k in ["metformin","glibenclamide","insulin","glucophage"]):
        multiplier *= (1 + diab_burden * 1.5)
 
    # ── NEW SIGNALS — BEAUTY PRODUCTS ────────────────────────────────────────
 
    # Is this a beauty product? Check via product name keywords
    _is_beauty = any(k in n for k in [
        "serum","moisturiser","moisturizer","sunscreen","spf","cleanser","toner",
        "mask","scrub","lotion","cream","gel","lip balm","lip gloss","body lotion",
        "body butter","face wash","micellar","retinol","vitamin c","niacinamide",
        "hyaluronic","collagen","brightening","whitening","glow","anti-aging",
        "anti-ageing","eye cream","body wash","shower gel","body mist","perfume",
        "cologne","foundation","concealer","mascara","eyeliner","blush","highlighter"
    ])
 
    if _is_beauty:
        # Proximity: nearby beauty salons and shops drive demand
        beauty_prox = embu_target.get("beauty_demand_proximity", 0)
        if beauty_prox > 0.3:
            multiplier *= (1 + beauty_prox * 0.5)   # up to +50% boost
 
        # GT momentum: if beauty category is trending upward, boost
        beauty_momentum = embu_target.get("beauty_momentum", 0)
        if beauty_momentum > 0.6:                    # accelerating demand
            multiplier *= (1 + (beauty_momentum - 0.5) * 0.4)
 
        # GT recent signal: high recent interest boosts established products
        beauty_recent = embu_target.get("beauty_recent4w", 0)
        if beauty_recent > 0.7:
            multiplier *= 1.15
 
        # Women 15-49 share — core beauty buyer
        beauty_target_pct = embu_target.get("beauty_target_pct", 0)
        if beauty_target_pct > 0.25:
            multiplier *= (1 + (beauty_target_pct - 0.25) * 0.8)
 
    # ── NEW SIGNALS — BODY BUILDING ───────────────────────────────────────────
 
    _is_bb = any(k in n for k in [
        "whey","protein","creatine","pre-workout","preworkout","amino","bcaa",
        "mass gainer","isolate","concentrate","casein","glutamine","beta-alanine",
        "l-carnitine","caffeine","shaker","gym","workout","muscle","bulk",
        "lean mass","fat burner","thermogenic"
    ])
 
    if _is_bb:
        # Proximity: nearby gyms and BB shops are the primary demand driver
        bb_prox = embu_target.get("bb_demand_proximity", 0)
        if bb_prox > 0.2:
            multiplier *= (1 + bb_prox * 0.8)       # gyms nearby = strong signal
 
        # GT momentum for bodybuilding category
        bb_momentum = embu_target.get("bodybuilding_momentum", 0)
        if bb_momentum > 0.5:
            multiplier *= (1 + (bb_momentum - 0.5) * 0.5)
 
        # Men 15-34 share — core BB buyer
        bb_target_pct = embu_target.get("bb_target_pct", 0)
        if bb_target_pct > 0.15:
            multiplier *= (1 + (bb_target_pct - 0.15) * 1.0)
 
    # ── NEW SIGNALS — VITAMINS & SUPPLEMENTS ──────────────────────────────────
 
    _is_supplement = any(k in n for k in [
        "vitamin","supplement","zinc","magnesium","calcium","iron","omega",
        "fish oil","multivitamin","probiotic","prebiotic","collagen","biotin",
        "folic acid","vitamin d","vitamin c","vitamin b","immune","antioxidant",
        "melatonin","turmeric","curcumin","glucosamine","chondroitin","coq10",
        "evening primrose","spirulina","moringa","ashwagandha","echinacea"
    ])
 
    if _is_supplement and not _is_bb:   # BB already handled above
        # Proximity: supplement stores and gyms nearby
        supp_prox = embu_target.get("supplement_demand_proximity", 0)
        if supp_prox > 0.2:
            multiplier *= (1 + supp_prox * 0.5)
 
        # GT momentum for supplements
        supp_momentum = embu_target.get("supplements_momentum", 0)
        if supp_momentum > 0.5:
            multiplier *= (1 + (supp_momentum - 0.5) * 0.4)
 
        # Adults 25-64 share — core supplement buyer
        supp_target_pct = embu_target.get("supplements_target_pct", 0)
        if supp_target_pct > 0.40:
            multiplier *= (1 + (supp_target_pct - 0.40) * 0.6)
 
        # Aged 65+ — bone/joint/heart supplements specifically
        pct_aged = embu_target.get("pct_aged_65_plus", 0)
        if pct_aged > 0.05 and any(k in n for k in ["calcium","vitamin d","glucosamine","chondroitin","coq10","omega"]):
            multiplier *= (1 + pct_aged * 1.5)
 
    return round(multiplier, 3)
 
 
print("\n✓ external_multiplier() redefined with GT + proximity signals")
print("  New signals added:")
print("    Beauty:      beauty_demand_proximity, beauty_momentum, beauty_recent4w, beauty_target_pct")
print("    BB:          bb_demand_proximity, bodybuilding_momentum, bb_target_pct")
print("    Supplements: supplement_demand_proximity, supplements_momentum, supplements_target_pct, pct_aged_65_plus")
print("    Pharma:      unchanged")
 

---
## 10. Final output — Branch #106 Opening Stock List


In [ ]:
# Final formatted output table

output = prediction[[
    'correct_therapeutic_class',
    'opening_stock_qty',
    'stock_range',
    'predicted_steady_state',
    'confidence',
    'dead_stock_risk',
    'model_reasoning',
]].sort_values('opening_stock_qty', ascending=False).copy()

output.columns = [
    'Category',
    'Opening Stock Qty',
    'Stock Range (80% CI)',
    'Predicted Monthly (Steady State)',
    'Confidence',
    'Dead Stock Risk',
    'Model Reasoning',
]

print("=" * 70)
print("BRANCH #106 — RECOMMENDED OPENING STOCK")
method = "KNN-weighted" if not used_fallback else "equal-weight average"
print(f"Method:             {method}")
print(f"Penetration factor: {penetration_factor:.2f}"
      + (" (default)" if pf_is_default else " (data-derived)"))
print(f"Facilities used:    {len(weight_map)}")
print("=" * 70)

output


BRANCH #106 — RECOMMENDED OPENING STOCK
Method:             KNN-weighted
Penetration factor: 1.39 (data-derived)
Facilities used:    3


,Category,Opening Stock Qty,Stock Range (80% CI),Predicted Monthly (Steady State),Confidence,Dead Stock Risk,Model Reasoning
153,Diuretic — loop,1881,0 – 6681 (80% CI),1355.919240,High,False,Comparable to Branch 4 Month 1 pattern
71,Antihypertensive — CCB,1577,0 – 5423 (80% CI),1136.197152,High,False,Comparable to Branch 4 Month 1 pattern
77,Antihypertensive — injectable,1438,17 – 3656 (80% CI),1036.087806,High,False,Comparable to Branch 4 Month 1 pattern
68,Antihypertensive — ARB,1308,0 – 7251 (80% CI),942.971251,High,False,High communicable catchment burden (27%) — Com...
29,Antibiotic — penicillinase-resistant,1135,0 – 4262 (80% CI),818.033117,High,False,Comparable to Branch 4 Month 1 pattern
...,...,...,...,...,...,...,...
78,Antihypertensive — vasodilator,1,1 – 4 (80% CI),0.763234,Low,False,Insufficient history — conservative estimate
278,Scar treatment,1,1 – 1 (80% CI),0.909669,Medium,False,Comparable to Branch 4 Month 1 pattern
284,Taxane — antineoplastic,1,0 – 5 (80% CI),0.890440,Low,False,Insufficient history — conservative estimate
321,Wound healing — topical,1,1 – 1 (80% CI),0.400846,Low,True,Low cross-branch demand — order conservatively


In [ ]:
# ── Step 9b: Product-level allocation ────────────────────────────────────────
# Logic:
# 1. Category totals come from the KNN model (reliable, data-rich)
# 2. Within each category, products are ranked by their actual dispensing share
#    across similar branches (buying patterns)
# 3. Shares are then adjusted by external signals — proximity and DHS burden
# 4. Adjusted shares are normalised and applied to the category total

# ── A. Product dispensing share per category from similar branches ────────────
# Use steady_df (ramp-up excluded) filtered to weighted branches only
prod_steady = (
    disp_work[
        (disp_work["month_rank"] > RAMP_UP_MONTHS) &
        (disp_work["facility_id"].isin(weight_map))
    ]
    .copy()
)
prod_steady["weight"] = prod_steady["facility_id"].map(weight_map)
prod_steady["weighted_qty"] = prod_steady["total_qty_dispensed"] * prod_steady["weight"]

# Need product_name — join from fact_dispensing
prod_names = (fact_dispensing[["product_id","product_name"]]
              .dropna()
              .drop_duplicates("product_id"))

prod_agg = (prod_steady
            .merge(prod_names, on="product_id", how="left")
            .groupby(["correct_therapeutic_class","product_id","product_name"])
            .agg(weighted_qty=("weighted_qty","sum"))
            .reset_index())

# Drop products with no name
prod_agg = prod_agg[prod_agg["product_name"].notna()]

# Share within each category
cat_total_qty = prod_agg.groupby("correct_therapeutic_class")["weighted_qty"].transform("sum")
prod_agg["base_share"] = prod_agg["weighted_qty"] / cat_total_qty.replace(0, 1)

# ── B. External adjustment multipliers ───────────────────────────────────────
# Map product to clinical signal using product name keywords
# Higher multiplier = boost this product's share in the allocation

# New Venture Headstart 

**Goal:** Use predictive analytics to show how to curate an Opening Stock List for a new branch 

**Assumption:** Branch #106 is a new Tendri branch in Embu — it has no data in the system yet.  
We use dispensing and diagnosis patterns from *existing* Tendri branches to predict  
what Branch #106 should have on shelf from Day 1.

**The core logic:**
1. Build a disease burden profile for each existing Tendri branch from diagnosis data
2. Aggregate those profiles into one "Embu catchment" target profile
3. Weight each branch by how closely it matches that Embu profile (KNN)
4. Compute their steady-state dispensing (excluding ramp-up months)
5. Apply a penetration factor to size the opening stock conservatively
6. Output: Category · Opening Qty · Confidence · Dead Stock Risk
## Imports and Configuration
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

import pickle
import glob

# For PDF extraction
import fitz


# get data from MySQL database
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

print("All imports loaded successfully")
# Establish a connection to the MySQL database 

# ── MySQL connection ──────────────────────────────────────────────────────────
DB_USER = os.getenv("DB_USER",   "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST",   "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

DATABASE = os.getenv("DB_NAME",   "tenri_raw")
TENRI= os.getenv("DB_NAME",   "tenri")
REPORTING = os.getenv("DB_NAME",   "reporting")


# Get the data from  Tenri Raw database

tenri_raw_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DATABASE}")
tenri = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{TENRI}")
reporting = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{REPORTING}")


monthly_diagnoses_aggregate = pd.read_sql_query("SELECT * FROM tenri_raw.monthly_diagnoses_aggregate", tenri_raw_engine)
monthly_dispensing_aggregate = pd.read_sql_query("SELECT * FROM tenri_raw.monthly_dispensing_aggregate", tenri_raw_engine)
# ── Load simulated dispensing aggregate ───────────────────────────────────────
_sim_agg_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_dispensing_aggregate_*.csv"))
 
if not _sim_agg_path:
    print("WARNING: No simulated_dispensing_aggregate_*.csv found.")
    print("Run dispensing_simulation.py first, then re-run this cell.")
else:
    sim_agg = pd.read_csv(_sim_agg_path[-1])

sim_agg.head()
# ── Validate schema before appending ──────────────────────────────────────
REQUIRED_COLS = [
    "facility_id", "months", "product_id", "new_category_name",
    "parent_category_name", "correct_therapeutic_class",
    "total_qty_dispensed", "unique_patients", "avg_qty_per_patient"
]
missing = [c for c in REQUIRED_COLS if c not in sim_agg.columns]
if missing:
    raise ValueError(f"Simulated data missing columns: {missing}")

# ── Confirm no product_id collision ───────────────────────────────────────
real_pids = set(monthly_dispensing_aggregate["product_id"].unique())
sim_pids  = set(sim_agg["product_id"].unique())
clash     = real_pids & sim_pids
if clash:
    raise ValueError(
        f"product_id collision: {len(clash)} IDs exist in both datasets. "
        f"Sample: {list(clash)[:5]}"
    )
# ── Align dtypes to match real data ───────────────────────────────────────
sim_agg["facility_id"]         = sim_agg["facility_id"].astype("int64")
sim_agg["product_id"]          = sim_agg["product_id"].astype("int64")
sim_agg["total_qty_dispensed"] = sim_agg["total_qty_dispensed"].astype("float64")
sim_agg["unique_patients"]     = sim_agg["unique_patients"].astype("int64")
sim_agg["avg_qty_per_patient"] = sim_agg["avg_qty_per_patient"].astype("float64")
sim_agg["months"]              = pd.to_datetime(sim_agg["months"]).dt.strftime("%Y-%m-%d")
 # ── Append ────────────────────────────────────────────────────────────────
monthly_dispensing_aggregate = pd.concat(
    [monthly_dispensing_aggregate[REQUIRED_COLS], sim_agg[REQUIRED_COLS]],
    ignore_index=True
)

print(f"monthly_dispensing_aggregate updated:")
print(f"  Total rows:  {len(monthly_dispensing_aggregate):,}")
print(f"  Categories:  {monthly_dispensing_aggregate['new_category_name'].nunique()}")
print(f"  Date range:  {monthly_dispensing_aggregate['months'].min()} → {monthly_dispensing_aggregate['months'].max()}")
print(f"\n  Category breakdown:")
for cat, n in monthly_dispensing_aggregate["new_category_name"].value_counts().items():
    is_new = cat in ["Beauty Products","Vitamins & Supplements","Body Building"]
    tag    = " ← NEW" if is_new else ""
    print(f"    {cat:<35} {n:>8,} rows{tag}")

### Configuration

These are the tuning parameters for the model.  
You can adjust them without touching any of the logic below.

| Parameter | What it controls |
|---|---|
| `RAMP_UP_MONTHS` | How many opening months to exclude (new branches sell less while ramping up) |
| `KNN_K` | How many "most similar" branches to weight most heavily |
| `KNN_FALLBACK_THRESHOLD` | If all branches are equally similar, fall back to equal weights |
| `DEAD_STOCK_MULTIPLIER` | Flag an item as dead stock risk if opening qty > X times the monthly velocity |

# How many opening months to ignore when computing steady-state dispensing
# A branch opening in January may have unusual patterns for the first 1-3 months
RAMP_UP_MONTHS = 3

# How many top similar branches to weight most heavily in the prediction
KNN_K = 3

# If all branches are nearly equally similar to the Embu profile,
# the similarity score doesn't add value — fall back to equal weights
KNN_FALLBACK_THRESHOLD = 0.05

# Dead stock risk: if opening qty > this multiple of monthly velocity, flag it
# 1.5 means "you'd need more than 1.5 months to sell through this"
DEAD_STOCK_MULTIPLIER = 1.5


print("Configuration set")

---
## 1. Load the data

We load two queries from MySQL views:
- `monthly_diagnoses_aggregate` — one row per facility × month × diagnosis
- `monthly_dispensing_aggregate` — one row per facility × month × product category

# other data to pull 

fact_dispensing = pd.read_sql_query("SELECT * FROM tenri.fact_dispensing", tenri)
fact_inventory_snapshot = pd.read_sql_query("SELECT * FROM tenri.fact_inventory_snapshot", tenri)
dim_patient_profile = pd.read_sql_query("SELECT * FROM tenri.dim_patient_profile", tenri)
settings_facility = pd.read_sql_query("SELECT * FROM tenri.settings_clinics", tenri)
# ── Load simulated fact_dispensing ────────────────────────────────────────────
_sim_disp_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_fact_dispensing_*.csv"))
_sim_snap_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_inventory_snapshot_*.csv"))
 
if not _sim_disp_path:
    print("WARNING: No simulated_fact_dispensing_*.csv found — skipping.")
else:
    sim_disp = pd.read_csv(_sim_disp_path[-1], parse_dates=["date"])
    sim_disp = pd.read_csv(_sim_disp_path[-1], parse_dates=["date"])
 
    # Keep only columns that exist in real fact_dispensing
    # (simulated file has extra cols like 'category', 'inventory_health')
    FACT_DISP_COLS = [c for c in fact_dispensing.columns if c in sim_disp.columns]
    sim_disp_clean = sim_disp[FACT_DISP_COLS].copy()
 
    # Align dtypes
    sim_disp_clean["facility_id"] = sim_disp_clean["facility_id"].astype("int64")
    sim_disp_clean["product_id"]  = sim_disp_clean["product_id"].astype("int64")
 
    fact_dispensing = pd.concat([fact_dispensing, sim_disp_clean], ignore_index=True)
    print(f"fact_dispensing updated: {len(fact_dispensing):,} rows")

if not _sim_snap_path:
    print("WARNING: No simulated_inventory_snapshot_*.csv found — skipping.")
else:
    sim_snap = pd.read_csv(_sim_snap_path[-1], parse_dates=["snapshot_date"])
 
    # Keep only columns that exist in real fact_inventory_snapshot
    SNAP_COLS = [c for c in fact_inventory_snapshot.columns if c in sim_snap.columns]
    sim_snap_clean = sim_snap[SNAP_COLS].copy()
 
    # Align dtypes
    sim_snap_clean["facility_id"] = sim_snap_clean["facility_id"].astype("int64")
    sim_snap_clean["product_id"]  = sim_snap_clean["product_id"].astype("int64")
 
    fact_inventory_snapshot = pd.concat(
        [fact_inventory_snapshot, sim_snap_clean], ignore_index=True
    )
    print(f"fact_inventory_snapshot updated: {len(fact_inventory_snapshot):,} rows")

# ── Load product intelligence (new — for dashboard profitability tab) ─────────
_sim_intel_path = sorted(glob.glob(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\simulated_product_intelligence_*.csv"))
if _sim_intel_path:
    product_intel = pd.read_csv(_sim_intel_path[-1])
    print(f"product_intel loaded: {len(product_intel):,} products")
else:
    product_intel = pd.DataFrame()
    print("WARNING: No simulated_product_intelligence_*.csv found.")


### Local Data
# Load diagnosis data
diag_df = monthly_diagnoses_aggregate.copy()

# Load dispensing data
disp_df = monthly_dispensing_aggregate.copy()
disp_df["months"] = pd.to_datetime(disp_df["months"])
 
print(f"disp_df rebuilt from updated monthly_dispensing_aggregate:")
print(f"  Rows:       {len(disp_df):,}")
print(f"  Categories: {disp_df['new_category_name'].nunique()}")
print(f"  Date range: {disp_df['months'].min()} → {disp_df['months'].max()}")
 
# Confirm the 3 new categories are present
new_cats = ["Beauty Products", "Vitamins & Supplements", "Body Building"]
for cat in new_cats:
    n = (disp_df["new_category_name"] == cat).sum()
    status = "✓" if n > 0 else "✗ MISSING"
    print(f"  {status} {cat}: {n:,} rows") 

# change data types 
diag_df['facility_id'] = diag_df['facility_id'].astype(int)

# Parse date columns so Python understands them as dates, not strings
diag_df['monthly'] = pd.to_datetime(diag_df['monthly'])

print(f"Diagnosis  — {diag_df['facility_id'].nunique()} facilities, "
      f"{diag_df['monthly'].nunique()} months, {len(diag_df):,} rows")
print(f"Dispensing — {disp_df['facility_id'].nunique()} facilities, "
      f"{disp_df['months'].nunique()} months, {len(disp_df):,} rows")
print("=== DIAGNOSIS SAMPLE ===")
diag_df[['facility_id', 'monthly', 'diagnosis_disease_group',
         'unique_patient_count', 'chronic_cases', 'consultation_count']].head(8)

print("=== DISPENSING SAMPLE ===")
disp_df[['facility_id', 'months', 'correct_therapeutic_class',
         'total_qty_dispensed', 'unique_patients']].head(10)  
### Get External Data
# Load Kenya health facilities GeoJSON and flatten to table
import json

geojson_path = r"C:/Users/Mercy/Documents/Tendri/Snowflake Pulls/Xana/snowflake/tendri/data/hotosm_ken_health_facilities_points_geojson.geojson"

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

rows = []
for feature in geojson["features"]:
    row = feature["properties"].copy()
    coords = feature["geometry"]["coordinates"] if feature["geometry"] else [None, None]
    row["longitude"] = coords[0]
    row["latitude"] = coords[1]
    rows.append(row)

kenya_health_facilities = pd.DataFrame(rows)
kenya_health_facilities[ kenya_health_facilities["name"].str.contains("Tenri", case=False, na=False) ]
# ── Match Tenri facilities to kenya_health_facilities ─────────────────────────

# Step 1: Parse facility_id from facility_key in diag_df
diag_df['facility_id'] = diag_df['facility_key'].str.split('|').str[-1].astype(int)

# Step 2: Check pregnant_cases per facility — maternity hospital will score highest
maternity_signal = (
    diag_df.groupby('facility_id')
    .agg(
        pregnant_cases     = ('pregnant_cases', 'sum'),
        total_consultations= ('consultation_count', 'sum'),
        total_female       = ('total_female', 'sum'),
        total_male         = ('total_male', 'sum'),
    )
    .reset_index()
)
maternity_signal['pct_female'] = (
    maternity_signal['total_female'] /
    (maternity_signal['total_female'] + maternity_signal['total_male']) * 100
).round(1)
maternity_signal['pregnant_rate'] = (
    maternity_signal['pregnant_cases'] /
    maternity_signal['total_consultations'] * 100
).round(2)

print("=== Maternity signal per facility ===")
print(maternity_signal.sort_values('pregnant_cases', ascending=False).to_string())

# Step 3: Assign coordinates from kenya_health_facilities
# Facility with highest pregnant_cases = Tenri Childrens' Hospital (Maternity & Theatre)
# The other = Tenri Children's Hospital (general)

tenri_facilities = kenya_health_facilities[
    kenya_health_facilities["name"].str.contains("Tenri", case=False, na=False)
].copy()
print("\n=== Tenri facilities from OSM ===")
print(tenri_facilities[['name','latitude','longitude']].to_string())

# The maternity one has the higher pregnant_cases — assign accordingly
maternity_facility_id = maternity_signal.sort_values('pregnant_cases', ascending=False).iloc[0]['facility_id']
general_facility_id   = maternity_signal.sort_values('pregnant_cases', ascending=False).iloc[1]['facility_id']

# Map OSM entries: the one with "Maternity" in the name = maternity_facility_id
maternity_coords = tenri_facilities[tenri_facilities['name'].str.contains('Mater', case=False, na=False)].iloc[0]
general_coords   = tenri_facilities[~tenri_facilities['name'].str.contains('Mater', case=False, na=False)].iloc[0]

facility_coords = pd.DataFrame([
    {
        'facility_id': int(maternity_facility_id),
        'name':        maternity_coords['name'],
        'latitude':    maternity_coords['latitude'],
        'longitude':   maternity_coords['longitude'],
        'type':        'Maternity & Theatre',
    },
    {
        'facility_id': int(general_facility_id),
        'name':        general_coords['name'],
        'latitude':    general_coords['latitude'],
        'longitude':   general_coords['longitude'],
        'type':        'General Hospital',
    },
])

print("\n=== Facility coordinates assigned ===")
print(facility_coords.to_string())
from math import radians, sin, cos, sqrt, atan2

# ── Haversine distance function ───────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    """Returns distance in km between two lat/lon points."""
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── For each Tenri facility, find nearby health facilities ────────────────────
RADIUS_KM = 25  # adjust — 5km is a reasonable pharmacy catchment

nearby_results = []

for _, tenri_fac in facility_coords.iterrows():
    lat1 = tenri_fac['latitude']
    lon1 = tenri_fac['longitude']

    kdf = kenya_health_facilities.copy()
    kdf = kdf.dropna(subset=['latitude','longitude'])

    # Exclude the Tenri facilities themselves
    kdf = kdf[~kdf['name'].str.contains('Tenri', case=False, na=False)]

    kdf['distance_km'] = kdf.apply(
        lambda row: haversine_km(lat1, lon1, row['latitude'], row['longitude']),
        axis=1
    )
    

    nearby = kdf[kdf['distance_km'] <= RADIUS_KM].sort_values('distance_km').copy()
    nearby['tenri_facility_id']   = tenri_fac['facility_id']
    nearby['tenri_facility_name'] = tenri_fac['name']

    nearby_results.append(nearby)

nearby_all = pd.concat(nearby_results, ignore_index=True)

print(f"Facilities within {RADIUS_KM}km of each Tenri branch:")
print(nearby_all[['tenri_facility_name','name','amenity','healthcare',
                   'distance_km']].sort_values(['tenri_facility_name','distance_km'])
      .to_string())
# ── Proximity feature summary per Tenri facility ──────────────────────────────
proximity_features = []

for facility_id in facility_coords['facility_id']:
    nearby = nearby_all[nearby_all['tenri_facility_id'] == facility_id]

    proximity_features.append({
        'facility_id':          facility_id,
        'n_facilities_25km':     len(nearby),
        'n_hospitals_25km':      (nearby['amenity'] == 'hospital').sum(),
        'n_clinics_25km':        (nearby['amenity'].isin(['clinic','doctors'])).sum(),
        'n_pharmacies_25km':     (nearby['amenity'] == 'pharmacy').sum(),
        'n_health_centres_25km': (nearby['healthcare'].str.contains('health_centre|health centre',
                                  case=False, na=False)).sum(),
        'has_hospital_nearby':  int((nearby['amenity'] == 'hospital').any()),
    })

proximity_df = pd.DataFrame(proximity_features)
print(proximity_df.to_string())
# ── Build category_proximity_driver ──────────────────────────────────────────
# Maps each product category to nearby facilities that drive demand for it
# Used in model_reasoning column in Step 8e

FACILITY_DEMAND_MAP = {
    'hospital':      ['Injectables', 'IV Fluids & Infusions', 'Wound Care',
                      'Oral Solid Forms', 'Infection Control'],
    'clinic':        ['Oral Solid Forms', 'Oral Liquid Forms', 'Injectables'],
    'doctors':       ['Oral Solid Forms', 'Oral Liquid Forms'],
    'pharmacy':      ['Oral Solid Forms', 'Topical Preparations'],
    'maternity':     ['Injectables', 'Oral Solid Forms', 'Vaccines & Biologicals',
                      'IV Fluids & Infusions'],
    'health_centre': ['Oral Solid Forms', 'Oral Liquid Forms', 'Injectables',
                      'Vaccines & Biologicals'],
}

category_proximity_driver = {}

for _, fac in nearby_all.iterrows():
    amenity    = str(fac.get('amenity',    '')).lower()
    healthcare = str(fac.get('healthcare', '')).lower()
    name       = str(fac.get('name', 'Unknown facility'))
    dist       = round(fac.get('distance_km', 0), 1)

    # Determine facility type
    if 'maternity' in name.lower():
        fac_type = 'maternity'
    elif amenity in FACILITY_DEMAND_MAP:
        fac_type = amenity
    elif 'health_centre' in healthcare or 'health centre' in healthcare:
        fac_type = 'health_centre'
    else:
        continue

    for category in FACILITY_DEMAND_MAP[fac_type]:
        if category not in category_proximity_driver:
            category_proximity_driver[category] = []
        entry = f"{name} ({dist}km)"
        if entry not in category_proximity_driver[category]:
            category_proximity_driver[category].append(entry)

# Cap at 2 facilities per category for readability
for cat in category_proximity_driver:
    category_proximity_driver[cat] = category_proximity_driver[cat][:2]

print("=== category_proximity_driver ===")
for cat, drivers in sorted(category_proximity_driver.items()):
    print("  {:<35} {}".format(cat, drivers))

'''
#The full order for the cells that feed into Step 8e is then:
proximity_df        ← built from nearby_all
category_proximity_driver  ← NEW, built from nearby_all + FACILITY_DEMAND_MAP
CATEGORY_CONTEXT + BURDEN_SIGNALS (Step 3f)

Step 8e             ← uses all three: category_proximity_driver, CATEGORY_CONTEXT, BURDEN_SIGNALS
'''

# ── Load Embu DHS external data ───────────────────────────────────────────────
embu_dhs = pd.read_csv(r'C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\embu_dhs_2022.csv')
embu_dhs = embu_dhs.rename(columns={'Unnamed: 0': 'geography'})
embu_dhs = embu_dhs.set_index('geography')


# Helper to safely parse values — some have brackets e.g. (91)
def safe_float(val):
    if pd.isna(val): return None
    s = str(val).replace('(','').replace(')','').replace('<','').strip()
    try: return float(s)
    except: return None

# ── Extract features relevant to pharmacy demand ──────────────────────────────
embu_row = embu_dhs.loc['embu']

embu_external = {
    # Maternal & reproductive health — drives ANC, FP, postnatal dispensing
    'fertility_rate':             safe_float(embu_row['Total fertility rate (number of children per woman)']),
    'teen_pregnancy_pct':         safe_float(embu_row['Teenage pregnancy (% age 15-19 who have ever been pregnant)']),
    'modern_fp_use_pct':          safe_float(embu_row['Use of modern method of FP (% of married women age 15-49)']),
    'antenatal_4plus_visits_pct': safe_float(embu_row['Women age 15-49 who had a live birth and had 4+ antenatal visits (%)']),
    'skilled_birth_pct':          safe_float(embu_row['Births delivered by a skilled provider2 (%)']),

    # Child health — drives paediatric dispensing (ORS, vaccines, supplements)
    'u5_stunting_pct':            safe_float(embu_row['Children under 5 who are stunted (%) (too short for their age)']),
    'u5_underweight_pct':         safe_float(embu_row['Children under 5 who are underweight (%) (too thin for their age)']),
    'vaccination_pct':            safe_float(embu_row['Children age 12-23 months fully vaccinated (basic antigens)3 (%)']),

    # Malaria burden — drives artemether, ITN, SP/Fansidar dispensing
    'itn_access_pct':             safe_float(embu_row['Household population with access to an insecticide-treated net (ITN) (%)']),
    'itn_use_pct':                safe_float(embu_row['Household population who slept under an ITN the night before the survey (%)']),

    # Socioeconomic — proxy for spend tier, OTC vs Rx mix
    'clean_fuel_access_pct':      safe_float(embu_row['Household population relying on clean fuels and technologies for cooking, space heating, & lighting (%)']),
    'water_access_pct':           safe_float(embu_row['Household population with access to at least basic drinking water service (%)']),
    'sanitation_access_pct':      safe_float(embu_row['Household population with at least basic sanitation service (%)']),
    'women_no_education_pct':     safe_float(embu_row['Women age 15-49 with no formal education (%)']),

    # Urban assumption
    'is_urban_branch':            1,
}

print("=== Embu external features ===")
for k, v in embu_external.items():
    print(f"  {k:<35} {v}")
# ── PDF paths — update if different on your machine ──────────────────────────
_PROJ_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\2019-Kenya-population-and-Housing-Census-Summary-Report-on-Kenyas-Population-Projections.pdf"
_VOL1_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\2019-Kenya-population-and-Housing-Census-Volume-1-Population-By-County-And-Sub-County.pdf"
_CIDP_PDF = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\EMBU_CIDP_2023-2027.pdf"
# ── Helpers ───────────────────────────────────────────────────────────────────
def _parse_num(s):
    try: return int(str(s).replace(",","").strip())
    except: return None
 
def _parse_float(s):
    try: return float(str(s).replace(",","").strip())
    except: return None
 
def _get_lines(doc, pg):
    return [l.strip() for l in doc[pg].get_text().split("\n") if l.strip()]
 
def _county_table(doc, page_idx, n_cols, skip):
    lines = _get_lines(doc, page_idx)
    rows, i = [], 0
    while i < len(lines):
        line = lines[i]
        if re.match(r"^[\d,]+$", line) or line in skip or len(line) < 2:
            i += 1; continue
        nums, j = [], i + 1
        while j < len(lines) and len(nums) < n_cols:
            if re.match(r"^[\d,]+$", lines[j]):
                nums.append(_parse_num(lines[j])); j += 1
            else: break
        if len(nums) == n_cols:
            rows.append({"county": line, "_nums": nums}); i = j
        else: i += 1
    return rows
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — KNBS Official Projections
# ══════════════════════════════════════════════════════════════════════════════
_doc_proj = fitz.open(_PROJ_PDF)
_PROJ_YEARS = ["2020","2021","2022","2023","2024","2025","2030","2035","2040","2045"]
_LF_YEARS   = ["2020","2021","2022","2023","2024","2025","2030","2035"]
_HH_YEARS   = ["2020","2021","2022","2023","2024","2025","2026","2027","2028","2029","2030"]
 
# Population projections (page 2)
_raw = _county_table(_doc_proj, 1, 10,
    {"County","Kenya"} | set(_PROJ_YEARS) | {"Population Projections by County, 2020-2045"})
_proj_embu = next(r for r in _raw if r["county"] == "Embu")
_proj = dict(zip(_PROJ_YEARS, _proj_embu["_nums"]))
 
# Labour force (page 4)
_raw_lf = _county_table(_doc_proj, 3, 8,
    {"County","Kenya"} | set(_LF_YEARS) | {
        "Population in the Labour Force, age 15-64 by County, 2020-2035",
        "Labour Force Projections",
        "The population in the labour force (age 15-64) is expected to increase by 40.7 percent from 28.8 million",
        "in 2020 to 40.5 million by 2035."})
_lf_embu = next(r for r in _raw_lf if r["county"] == "Embu")
_lf = dict(zip(_LF_YEARS, _lf_embu["_nums"]))
 
# Households (page 5)
_raw_hh = _county_table(_doc_proj, 4, 11,
    {"County","Kenya"} | set(_HH_YEARS) | {
        "Projected Number of Households by County, 2020-2030",
        "Households Projections",
        "Data on Household projections show that by 2030, there will be approximately 15.9 million households.",
        "Nairobi City, which is entirely urban, will require nearly 2 million houses to host its population by 2030."})
_hh_embu = next(r for r in _raw_hh if r["county"] == "Embu")
_hh = dict(zip(_HH_YEARS, _hh_embu["_nums"]))
 
# 2019 census baseline from Volume I (sex breakdown)
_doc_vol1 = fitz.open(_VOL1_PDF)
_census_rows = []
for _pg in range(len(_doc_vol1)):
    _text  = _doc_vol1[_pg].get_text()
    _lines = [l.strip() for l in _text.split("\n") if l.strip()]
    for _i, _l in enumerate(_lines):
        if not re.match(r"^[\d,]+$", _l) and len(_l) > 2 and _i + 4 < len(_lines):
            _cands = []
            for _k in range(1, 5):
                if re.match(r"^[\d,]+$", _lines[_i+_k]): _cands.append(_parse_num(_lines[_i+_k]))
                else: break
            if len(_cands) == 4 and _cands[3] == _cands[0] + _cands[1] + _cands[2]:
                _census_rows.append({"county": _l, "male": _cands[0], "female": _cands[1],
                                     "intersex": _cands[2], "total": _cands[3]})
_census_df  = pd.DataFrame(_census_rows).drop_duplicates("county")
_embu_census = _census_df[_census_df["county"].str.contains("Embu", na=False)].iloc[0]
 
_TOTAL_2019 = int(_embu_census["total"])    # 608,599
_TOTAL_2025 = int(_proj["2025"])            # 661,690
_LABOUR_2025 = int(_lf["2025"])             # 423,316
_HH_2025     = int(_hh["2025"])             # 208,991
_ANNUAL_RATE = (_TOTAL_2025 / _TOTAL_2019) ** (1/6) - 1
 
 

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — CIDP Age Cohort Data (Table 5)
# ══════════════════════════════════════════════════════════════════════════════
_doc_cidp = fitz.open(_CIDP_PDF)
_CIDP_YEARS = [2019, 2022, 2025, 2027]
 
# Table 5: 5-year age cohorts
_lines_45 = _get_lines(_doc_cidp, 44)
_rows5 = []
_i5 = 0
while _i5 < len(_lines_45):
    _line = _lines_45[_i5]
    if re.match(r"^\s*\d+[-+]\d*\s*$", _line) or _line in ("Age NS","All Ages"):
        _nums, _j = [], _i5 + 1
        while _j < len(_lines_45) and len(_nums) < 12:
            _v = _lines_45[_j]
            if re.match(r"^[\d,]+$", _v): _nums.append(_parse_num(_v)); _j += 1
            elif _v == "-": _nums.append(None); _j += 1
            else: break
        if len(_nums) == 12:
            for _yr_idx, _year in enumerate(_CIDP_YEARS):
                _o = _yr_idx * 3
                _rows5.append({"age_cohort": _line.strip(), "year": _year,
                               "male": _nums[_o], "female": _nums[_o+1], "total": _nums[_o+2]})
            _i5 = _j; continue
    _i5 += 1
 
_t5 = pd.DataFrame(_rows5)
_t5_2025 = _t5[_t5["year"] == 2025].set_index("age_cohort")
 
# Table 8: broad age groups
_lines_49  = _get_lines(_doc_cidp, 48)
_lines_50  = _get_lines(_doc_cidp, 49)
_t8_start  = next(i for i,l in enumerate(_lines_49) if "1.5.3" in l)
_lines_t8  = _lines_49[_t8_start:] + _lines_50
 
_BROAD = [
    ("Infant",       "infant_under1"),
    ("Under 5",      "under5"),
    ("Pre-School",   "preschool_3to5"),
    ("Primary",      "primary_6to13"),
    ("Secondary",    "secondary_13to19"),
    ("Youth",        "youth_15to29"),
    ("Economically", "economically_active_15to64"),
    ("Aged",         "aged_65plus"),
]
_rows8, _seen8 = [], set()
for _trigger, _label in _BROAD:
    if _label in _seen8: continue
    _idx = next((i for i,l in enumerate(_lines_t8) if _trigger in l), None)
    if _idx is None: continue
    _nums8, _j = [], _idx + 1
    while _j < len(_lines_t8) and len(_nums8) < 12:
        _v = _lines_t8[_j]
        if re.match(r"^[\d,]+$", _v): _nums8.append(_parse_num(_v)); _j += 1
        elif _v == "-": _nums8.append(None); _j += 1
        else: _j += 1
        if len(_nums8) == 12: break
    if len(_nums8) == 12:
        _seen8.add(_label)
        for _yr_idx, _year in enumerate(_CIDP_YEARS):
            _o = _yr_idx * 3
            _rows8.append({"age_group": _label, "year": _year,
                           "male": _nums8[_o], "female": _nums8[_o+1], "total": _nums8[_o+2]})
 
_t8 = pd.DataFrame(_rows8)
_t8_2025 = _t8[_t8["year"] == 2025].set_index("age_group")
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Compute derived features (all scaled 0-1)
# ══════════════════════════════════════════════════════════════════════════════
def _cohort_total(*groups):
    return sum(_t5_2025.loc[g, "total"] for g in groups if g in _t5_2025.index)
 
def _cohort_male(*groups):
    return sum(_t5_2025.loc[g, "male"] for g in groups if g in _t5_2025.index)
 
def _cohort_female(*groups):
    return sum(_t5_2025.loc[g, "female"] for g in groups if g in _t5_2025.index)
 
def _broad(group):
    return int(_t8_2025.loc[group, "total"]) if group in _t8_2025.index else None
 
# Age band shares (2025 CIDP cohorts / official KNBS 2025 total)
_pct_under_15    = round(_cohort_total("0-4","5-9","10-14") / _TOTAL_2025, 4)
_pct_youth_15_29 = round(_cohort_total("15-19","20-24","25-29") / _TOTAL_2025, 4)
_pct_adult_30_64 = round(_cohort_total("30-34","35-39","40-44","45-49","50-54","55-59","60-64") / _TOTAL_2025, 4)
_pct_aged_65plus = round(_cohort_total("65-69","70-74","75-79","80+") / _TOTAL_2025, 4)
 
# Category demand proxies (target buyer population / total population)
_beauty_women    = _cohort_female("15-19","20-24","25-29","30-34","35-39","40-44","45-49")
_bb_men          = _cohort_male("15-19","20-24","25-29","30-34")
_supp_adults     = _cohort_total("25-29","30-34","35-39","40-44","45-49","50-54","55-59","60-64")
 
_beauty_target_pct      = round(_beauty_women / _TOTAL_2025, 4)
_bb_target_pct          = round(_bb_men       / _TOTAL_2025, 4)
_supplements_target_pct = round(_supp_adults  / _TOTAL_2025, 4)
 
# Labour force pct and gender split
_labour_force_pct = round(_LABOUR_2025 / _TOTAL_2025, 4)
_pct_female_2019  = round(float(_embu_census["female"]) / _TOTAL_2019, 4)
 
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Update embu_external
# This extends the dict already defined above in the notebook.
# Keys are grouped by source so they're easy to audit.
# ══════════════════════════════════════════════════════════════════════════════
embu_external.update({
 
    # ── KNBS Official Projections ─────────────────────────────────────────────
    # Source: 2019 KNBS Population Projections Summary Report
    # These are exact official figures — not modelled here
    "embu_total_population_2025": _TOTAL_2025,      # 661,690
    "embu_labour_force_2025":     _LABOUR_2025,     # 423,316  (age 15-64)
    "embu_households_2025":       _HH_2025,         # 208,991
    "embu_annual_growth_rate":    round(_ANNUAL_RATE, 5),  # 0.01404 (1.4%/yr)
 
    # Year-by-year for trend analysis
    "embu_pop_2020": int(_proj["2020"]),  # 628,527
    "embu_pop_2021": int(_proj["2021"]),  # 635,160
    "embu_pop_2022": int(_proj["2022"]),  # 641,792
    "embu_pop_2023": int(_proj["2023"]),  # 648,425
    "embu_pop_2024": int(_proj["2024"]),  # 655,057
    "embu_pop_2025": int(_proj["2025"]),  # 661,690
 
    # ── Derived ratios (0-1 scale) ────────────────────────────────────────────
    # These match the scale of the DHS indicators already in embu_external
    "embu_labour_force_pct": _labour_force_pct,   # 0.6397 — working-age share
    "embu_pct_female_2019":  _pct_female_2019,    # 0.5001 — near 50/50
 
    # ── CIDP Age Band Shares (2025, derived from KNBS base) ───────────────────
    # Source: CIDP Table 5 age cohorts / KNBS 2025 total
    # Used by KNN to infer disease burden and consumption patterns
    "pct_under_15":     _pct_under_15,    # 0.2969 — paediatric pharma, vaccines
    "pct_youth_15_29":  _pct_youth_15_29, # 0.2675 — BB (male), beauty exploration (female)
    "pct_adult_30_64":  _pct_adult_30_64, # 0.3723 — core supplement + beauty buyer
    "pct_aged_65_plus": _pct_aged_65plus, # 0.0634 — chronic disease pharma, supplements
 
    # ── Category Demand Proxies (0-1 scale) ───────────────────────────────────
    # Derived from CIDP Table 5 cohorts, 2025 projection
    # Represents the proportion of the population that is the PRIMARY buyer
    # for each of the 3 new product categories
    "beauty_target_pct":      _beauty_target_pct,      # 0.2660 — women aged 15-49
    "bb_target_pct":          _bb_target_pct,          # 0.1742 — men aged 15-34
    "supplements_target_pct": _supplements_target_pct, # 0.4569 — adults aged 25-64
 
})
# ── Validation print ──────────────────────────────────────────────────────────
print("=== embu_external updated ===")
print(f"  Total keys now: {len(embu_external)}")
print(f"  2025 population:      {embu_external['embu_total_population_2025']:,}")
print(f"  Labour force 2025:    {embu_external['embu_labour_force_2025']:,}")
print(f"  Households 2025:      {embu_external['embu_households_2025']:,}")
print(f"  Annual growth rate:   {embu_external['embu_annual_growth_rate']*100:.3f}%")
print(f"  Age bands sum:        {_pct_under_15 + _pct_youth_15_29 + _pct_adult_30_64 + _pct_aged_65plus:.4f}  (expected ~1.0)")
print(f"  Beauty target:        {embu_external['beauty_target_pct']*100:.1f}% of population  ({_beauty_women:,} women 15-49)")
print(f"  BB target:            {embu_external['bb_target_pct']*100:.1f}% of population  ({_bb_men:,} men 15-34)")
print(f"  Supplements target:   {embu_external['supplements_target_pct']*100:.1f}% of population  ({_supp_adults:,} adults 25-64)")
 
print("\n  All embu_external keys:")
for k, v in sorted(embu_external.items()):
    print(f"    {k:<40} {v}")
---
## 2. Define the disease burden mapping

Your diagnosis view stores disease groups as text like `"NCD-Cardiovascular-Hypertension"`.  
We want to convert those into numeric scores per dimension (malaria burden, diabetes burden, etc.)  
so we can compare facilities mathematically.

This dictionary maps each burden dimension to the disease group text fragments that belong to it.  
If any of those fragments appear in a row's `diagnosis_disease_group`, that row scores 1 for that dimension.

# Each key is a burden dimension we care about.
# Each value is a list of text fragments to look for in diagnosis_disease_group.
# If any fragment matches, the row gets a score of 1 for that dimension.
BURDEN_MAP = {
    'malaria':         ['Communicable-Malaria', 'Malaria'],
    'communicable':    ['Communicable-Infectious', 'Communicable'],
    'hypertension':    ['NCD-Cardiovascular-Hypertension'],
    'cardiovascular':  ['NCD-Cardiovascular-CoronaryHeart',
                        'NCD-Cardiovascular-HeartFailure',
                        'NCD-Cardiovascular'],
    'diabetes':        ['NCD-Endocrine-Diabetes', 'NCD-Endocrine-Other',
                        'NCD-Endocrine'],
    'respiratory':     ['Respiratory-URTI', 'Respiratory-Pneumonia',
                        'Respiratory-Rhinitis-Sinusitis',
                        'NCD-Respiratory-Asthma', 'Respiratory'],
    'gi':              ['GI-Gastritis', 'GI-PepticUlcer', 'GI-GERD-Oesophageal'],
    'maternal':        ['MNCH-Maternal', 'MNCH'],
    'gynaecological':  ['GU-GynaeUrological'],
    'musculoskeletal': ['MSK-Musculoskeletal'],
    'dermatological':  ['Dermatological'],
    'mental_health':   ['NCD-Mental'],
}

print(f"Defined {len(BURDEN_MAP)} burden dimensions:")
print(list(BURDEN_MAP.keys()))

---
## 3. Build facility profiles

For each existing Tendri branch, we build a **feature vector** — a row of numbers  
that describes what kind of patients that branch sees.

We use ratios (percentages) not raw counts, so a small branch and a large branch  
can be compared fairly.

**Features:**
- Disease burden ratios (e.g. 17% of diagnoses are hypertension)
- Case type ratios (e.g. 12% of visits are chronic disease follow-ups)
- Age mix ratios (e.g. 23% of patients are paediatric)
- Gender ratio
- Average monthly consultations (facility size)


# Step 3a: Score each diagnosis row for each burden dimension
# We create one new column per burden dimension
# The value is 1 if the disease group matches, 0 if not

df = diag_df.copy()   # always work on a copy, don't modify the original

for burden_name, keywords in BURDEN_MAP.items():
    # Wrap each keyword so it only matches a full segment, not a prefix.
    # combined_diagnosis uses '|' as segment separator, so a valid match is:
    #   - at start of string OR after a '|'     → (?:^|\|)
    #   - at end of string OR before a '|'      → (?=\||$)
    # This prevents 'NCD-Cardiovascular' matching inside
    # 'NCD-Cardiovascular-Hypertension'.
    segmented = [r'(?:^|\|)' + re.escape(kw) + r'(?=\||$)' for kw in keywords]
    pattern = '|'.join(segmented)
    
    # str.contains checks if the pattern appears anywhere in the text
    # case=False means it's not case-sensitive
    # na=False means NaN values score 0 instead of NaN
    df[f'burden_{burden_name}'] = (
        df['combined_diagnosis']
        .str.contains(pattern, case=False, na=False)
        .astype(int)   # convert True/False to 1/0
    )

print("Burden columns created:")
burden_cols = [f'burden_{k}' for k in BURDEN_MAP]
print(burden_cols)
print()

# Show a sample: diagnosis row → burden scores
df[['combined_diagnosis'] + burden_cols[:5]].head(5)
# Step 3b: Aggregate age bands into three groups
# Instead of 10 separate age columns, we summarise as paediatric / working age / chronic age

# Paediatric = under 13
df['age_paediatric'] = (
    df['total_age_less_than_1'] +
    df['total_age_1_4'] +
    df['total_age_5_12']
)

# Working age = 18-44
df['age_working'] = (
    df['total_age_18_24'] +
    df['total_age_25_34'] +
    df['total_age_35_44']
)

# Chronic age = 45+ (more likely to have NCDs like diabetes, hypertension)
df['age_chronic'] = (
    df['total_age_45_54'] +
    df['total_age_55_64'] +
    df['total_age_over_65']
)

print("Age band columns created: age_paediatric, age_working, age_chronic")

# Step 3c: Aggregate everything to facility level
# Currently: one row per facility × month × diagnosis
# We want: one row per facility (sum across all months and diagnoses)

case_cols  = ['chronic_cases', 'pregnant_cases', 'follow_up_cases',
              'immunisation_cases', 'medication_pickup_cases']

count_cols = ['unique_patient_count', 'total_male', 'total_female',
              'age_paediatric', 'age_working', 'age_chronic',
              'consultation_count']

# .groupby('facility_id') groups all rows for the same facility together
# .sum() adds up all the values
# .reset_index() turns the group key back into a regular column
facility_agg = (
    df.groupby('facility_id')[burden_cols + case_cols + count_cols]
    .sum()
    .reset_index()
)

print(f"Aggregated to {len(facility_agg)} facilities")
facility_agg.head()

# Step 3d: Convert counts → ratios
# Ratios allow fair comparison between large and small facilities

# Total diagnoses across all burden dimensions (denominator for burden ratios)
total_diagnoses = facility_agg[burden_cols].sum(axis=1).replace(0, np.nan)
# axis=1 means sum across columns (row-wise)
# replace(0, np.nan) avoids division by zero

total_consults = facility_agg['consultation_count'].replace(0, np.nan)
total_patients = facility_agg['unique_patient_count'].replace(0, np.nan)

# Burden ratios: what fraction of diagnoses fall in each disease group
for col in burden_cols:
    facility_agg[f'{col}_ratio'] = facility_agg[col] / total_diagnoses

# Case type ratios: what fraction of consultations are each case type
for col in case_cols:
    facility_agg[f'{col}_ratio'] = facility_agg[col] / total_consults

# Age mix ratios
facility_agg['pct_paediatric']  = facility_agg['age_paediatric'] / total_patients
facility_agg['pct_working']     = facility_agg['age_working']    / total_patients
facility_agg['pct_chronic_age'] = facility_agg['age_chronic']    / total_patients
facility_agg['pct_female']      = facility_agg['total_female']   / total_patients

print("Ratios computed. Sample burden ratios:")
ratio_cols = (
    [f'{c}_ratio' for c in burden_cols + case_cols]
    + ['pct_paediatric', 'pct_working', 'pct_chronic_age', 'pct_female']
)

facility_agg[['facility_id'] + [c for c in ratio_cols if 'burden_' in c]].round(3)

# Step 3e: Add average monthly consultations as a facility size feature
# This tells us how busy each branch is on average per month

monthly_size = (
    diag_df
    .groupby(['facility_id', 'monthly'])['consultation_count']   # group by facility AND month
    .sum()                                                         # total consultations per month
    .groupby('facility_id')                                        # then group by facility only
    .mean()                                                        # average across months
    .rename('avg_monthly_consultations')
    .reset_index()
)

# Merge this back onto our facility profiles
facility_profiles = facility_agg.merge(monthly_size, on='facility_id', how='left')
# ── ADD: Dispensing behaviour features (category mix) ────────────────────────
# NEW — was discussed but missing from notebook
disp_behaviour = (
    disp_df.groupby('facility_id')
    .agg(
        total_qty       = ('total_qty_dispensed', 'sum'),
        unique_products = ('product_id',          'nunique'),
        avg_patients    = ('unique_patients',      'mean'),
    ).reset_index()
)
cat_mix = (
    disp_df.groupby(['facility_id','new_category_name'])['total_qty_dispensed']
    .sum().unstack(fill_value=0)
)
cat_mix = cat_mix.div(cat_mix.sum(axis=1), axis=0)
cat_mix.columns = [f'cat_ratio_{c.lower().replace(" ","_").replace("/","_")}'
                   for c in cat_mix.columns]
cat_mix = cat_mix.reset_index()
cat_ratio_cols = [c for c in cat_mix.columns if c.startswith('cat_ratio_')]

facility_profiles = facility_profiles.merge(disp_behaviour, on='facility_id', how='left')
facility_profiles = facility_profiles.merge(cat_mix, on='facility_id', how='left')

# ── FIX: Merge proximity ONCE only ───────────────────────────────────────────
# REMOVED duplicate merge — was appearing twice
facility_profiles = facility_profiles.merge(proximity_df, on='facility_id', how='left')

proximity_cols = ['n_facilities_25km','n_hospitals_25km','n_clinics_25km',
                  'n_pharmacies_25km','has_hospital_nearby']
for col in proximity_cols:
    facility_profiles[col] = facility_profiles[col].fillna(0)

# Build ratio_cols — ONCE, in one place
ratio_cols = (
    [f'{c}_ratio' for c in burden_cols + case_cols]
    + ['pct_paediatric', 'pct_working', 'pct_chronic_age', 'pct_female']
    + ['avg_monthly_consultations']
    + proximity_cols
    + cat_ratio_cols
)

print(f"Final facility profiles: {len(facility_profiles)} facilities × {len(ratio_cols)} features")
print(f"  — {len([c for c in ratio_cols if 'burden_' in c])} burden ratio features")
print(f"  — {len(proximity_cols)} proximity features")
print(f"  — {len(cat_ratio_cols)} dispensing category features")
# Verify all ratio_cols exist in facility_profiles before Step 4
missing = [c for c in ratio_cols if c not in facility_profiles.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print(f"✓ All {len(ratio_cols)} feature columns present in facility_profiles")

print(f"\nfacility_profiles shape: {facility_profiles.shape}")
print(facility_profiles[proximity_cols].head())  # confirm proximity values are not all 0
# ── Paths — update if different ───────────────────────────────────────────────
GT_DIR   = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\google_trends_output"    # folder where trend scripts saved outputs
   # folder where trend scripts saved outputs
PROX_CSV = r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\pharmaplus_proximity_data.csv"  # from pharmaplus_proximity_scraper.py
# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — GOOGLE TRENDS FEATURES
# Loads embu_google_trends_features_{DATE}.csv
# Each row: feature name | value (already 0-1 scaled by the trends script)
# ══════════════════════════════════════════════════════════════════════════════
print("Loading Google Trends features...")
 
_gt_paths = sorted(glob.glob(os.path.join(GT_DIR, "embu_google_trends_features_*.csv")))
 
if not _gt_paths:
    print("  WARNING: No embu_google_trends_features_*.csv found in google_trends_output/")
    print("  Run the Google Trends pipeline first, then re-run this cell.")
    embu_google_trends = {}
else:
    _gt_df = pd.read_csv(_gt_paths[-1])  # most recent run
 
    # Validate expected columns
    if not {"feature", "value"}.issubset(_gt_df.columns):
        raise ValueError(f"Expected columns 'feature' and 'value', got: {_gt_df.columns.tolist()}")
 
    embu_google_trends = dict(zip(_gt_df["feature"], _gt_df["value"]))
 
    print(f"  Loaded {len(embu_google_trends)} features from: {_gt_paths[-1]}")
    print(f"  Features:")
    for feat, val in embu_google_trends.items():
        print(f"    {feat:<45} {val}")
 
# 3f. ══════════════════════════════════════════════════════════════════════════════
# PART 2 — PROXIMITY FEATURES
# Loads pharmaplus_proximity_data.csv from the SerpAPI scraper
# Derives count-based features per category within 3km of each facility
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading proximity data...")
 
if not os.path.exists(PROX_CSV):
    print(f"  WARNING: {PROX_CSV} not found.")
    print("  Run pharmaplus_proximity_scraper.py first, then re-run this cell.")
    embu_proximity = {}
else:
    prox = pd.read_csv(PROX_CSV)
 
    # Validate
    required_cols = {"facility_id","category","distance_km","is_chain","rating"}
    if not required_cols.issubset(prox.columns):
        raise ValueError(f"Missing columns in proximity CSV: {required_cols - set(prox.columns)}")
 
    # Coerce distance to numeric (some rows may be empty string)
    prox["distance_km"] = pd.to_numeric(prox["distance_km"], errors="coerce")
 
    print(f"  Loaded {len(prox):,} rows")
    print(f"  Facilities: {sorted(prox['facility_id'].unique())}")
    print(f"  Categories: {sorted(prox['category'].unique())}")
 
    # ── Aggregate across both facilities ─────────────────────────────────────
    # Branch 106 sits in Embu town near both Tendri facilities.
    # We use the union of both facility catchments (3km radius) as the
    # demand signal for Branch 106 — deduplicated by place_name.
    prox_dedup = prox.drop_duplicates(subset=["place_name","category"])
 
    RADIUS_KM = 3.0
    prox_near = prox_dedup[prox_dedup["distance_km"] <= RADIUS_KM].copy()
 
    # ── Compute features per search category ─────────────────────────────────
    CAT_MAP = {
        # search category in CSV          → feature prefix
        "gym_fitness":           "gym",
        "beauty_salon_spa":      "beauty_salon",
        "beauty_shop_cosmetics": "beauty_shop",
        "supplements_vitamins":  "supplement_store",
        "bodybuilding_shop":     "bb_shop",
        "pharmacy_competitor":   "pharmacy",
        "supermarket":           "supermarket",
    }
 
    # Max possible counts for normalisation (sensible ceiling per category)
    # Based on what a dense Kenyan town centre might realistically have in 3km
    NORMALISE_CAP = {
        "gym":              10,
        "beauty_salon":     20,
        "beauty_shop":      15,
        "supplement_store": 8,
        "bb_shop":          5,
        "pharmacy":         15,
        "supermarket":      10,
    }
 
    embu_proximity = {}
 
    for csv_cat, feat_prefix in CAT_MAP.items():
        subset = prox_near[prox_near["category"] == csv_cat]
        cap    = NORMALISE_CAP[feat_prefix]
 
        # Count of places (normalised 0-1)
        count        = len(subset)
        count_norm   = round(min(count / cap, 1.0), 4)
 
        # Count of chains (normalised to count)
        chain_count  = int(subset["is_chain"].sum()) if "is_chain" in subset.columns else 0
        chain_norm   = round(min(chain_count / max(count, 1), 1.0), 4)
 
        # Average rating (normalised to 5.0 max)
        ratings      = pd.to_numeric(subset["rating"], errors="coerce").dropna()
        avg_rating   = round(ratings.mean() / 5.0, 4) if len(ratings) > 0 else 0.0
 
        # Nearest distance (normalised: 0 = at door, 1 = at 3km edge)
        distances    = subset["distance_km"].dropna()
        nearest_norm = round(distances.min() / RADIUS_KM, 4) if len(distances) > 0 else 1.0
 
        embu_proximity[f"n_{feat_prefix}_3km"]         = count_norm    # 0-1: how many nearby
        embu_proximity[f"n_{feat_prefix}_chain_3km"]   = chain_norm    # 0-1: chain share
        embu_proximity[f"avg_{feat_prefix}_rating"]    = avg_rating    # 0-1: quality signal
        embu_proximity[f"nearest_{feat_prefix}_km"]    = round(1 - nearest_norm, 4)  # 0-1: inverse distance (1=very close)
 
    # ── Composite demand signals ──────────────────────────────────────────────
    # These combine multiple proximity signals into single demand proxies
    # for each new product category
 
    # Beauty demand signal: beauty salons + beauty shops nearby
    embu_proximity["beauty_demand_proximity"] = round(
        (embu_proximity.get("n_beauty_salon_3km", 0) +
         embu_proximity.get("n_beauty_shop_3km", 0)) / 2, 4
    )
 
    # BB demand signal: gyms + BB shops nearby
    embu_proximity["bb_demand_proximity"] = round(
        (embu_proximity.get("n_gym_3km", 0) +
         embu_proximity.get("n_bb_shop_3km", 0)) / 2, 4
    )
 
    # Supplement demand signal: supplement stores + gyms (gyms drive supplement demand)
    embu_proximity["supplement_demand_proximity"] = round(
        (embu_proximity.get("n_supplement_store_3km", 0) +
         embu_proximity.get("n_gym_3km", 0)) / 2, 4
    )
 
    # Competitive pressure: pharmacies + supermarkets (OTC competition)
    embu_proximity["pharmacy_competition_index"] = round(
        (embu_proximity.get("n_pharmacy_3km", 0) +
         embu_proximity.get("n_supermarket_3km", 0)) / 2, 4
    )
 
    print(f"\n  Proximity features computed ({len(embu_proximity)} features):")
    for feat, val in embu_proximity.items():
        raw_count_key = feat.replace("n_","").replace("_3km","")
        print(f"    {feat:<40} {val}")
 
# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — UPDATE embu_external
# ══════════════════════════════════════════════════════════════════════════════
print("\nUpdating embu_external...")
 
before = len(embu_external)
 
# Google Trends features
embu_external.update(embu_google_trends)
 
# Proximity features
embu_external.update(embu_proximity)
 
added = len(embu_external) - before
print(f"  embu_external: {before} → {len(embu_external)} keys (+{added} added)")
print(f"  Google Trends: {len(embu_google_trends)} features")
print(f"  Proximity:     {len(embu_proximity)} features")
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 FIX — Extend ratio_cols to include GT + proximity features
# Run this BEFORE Step 4 (Build the Embu target profile)
# ══════════════════════════════════════════════════════════════════════════════
 
# Collect all new feature keys that were just added to embu_external
# These come from embu_google_trends and embu_proximity defined in the previous cell
_new_external_features = (
    list(embu_google_trends.keys()) +
    list(embu_proximity.keys())
)
 
# Append to ratio_cols — deduplicated in case any overlap
_before = len(ratio_cols)
ratio_cols = list(dict.fromkeys(ratio_cols + _new_external_features))
_added = len(ratio_cols) - _before
 
print(f"ratio_cols updated: {_before} → {len(ratio_cols)} features (+{_added})")
print(f"  New GT features:        {len(embu_google_trends)}")
print(f"  New proximity features: {len(embu_proximity)}")
print(f"\n  Step 4 will now include all {len(ratio_cols)} features in embu_target.")
print(f"  KNN will use all {len(ratio_cols)} features for similarity scoring.")
 
# Confirm facility_profiles has all new columns (should already be zero-filled)
_missing_in_profiles = [f for f in _new_external_features if f not in facility_profiles.columns]
if _missing_in_profiles:
    print(f"\n  WARNING: {len(_missing_in_profiles)} features missing from facility_profiles — zero-filling now")
    for feat in _missing_in_profiles:
        facility_profiles[feat] = 0.0
else:
    print(f"\n  ✓ All new features present in facility_profiles")
 
 
# Step 3f:── Data-driven category ↔ burden mapping ─────────────────────────────────────
# Replaces both CATEGORY_CONTEXT and BURDEN_SIGNALS
# Derived from actual co-occurrence of diagnosis groups and dispensing categories
# in the same facility in the same month

# Step 1 — Monthly diagnosis groups per facility
diag_monthly = (
    diag_df.groupby(['facility_id', 'monthly'])['diagnosis_disease_group']
    .apply(list).reset_index()
)

# Step 2 — Monthly dispensing categories per facility
disp_monthly = (
    disp_df.groupby(['facility_id', 'months'])['correct_therapeutic_class']
    .apply(lambda x: list(x.dropna())).reset_index()
    .rename(columns={'months': 'monthly'})
)

# Step 3 — Join on facility + month
paired = diag_monthly.merge(disp_monthly, on=['facility_id', 'monthly'], how='inner')

# Step 4 — For each category, find top co-occurring diagnosis groups
from collections import Counter

CATEGORY_CONTEXT = {}
BURDEN_SIGNALS   = {}

all_categories = disp_df['correct_therapeutic_class'].dropna().unique()
all_burdens    = list(BURDEN_MAP.keys())  # your existing burden dimensions

for category in all_categories:
    # Months where this category was dispensed
    months_with_cat = paired[
        paired['correct_therapeutic_class'].apply(lambda cats: category in cats)
    ]
    if months_with_cat.empty:
        CATEGORY_CONTEXT[category] = 'Insufficient co-occurrence data'
        continue

    # Flatten all diagnosis groups from those months
    all_diags = [
        d for row in months_with_cat['diagnosis_disease_group']
        for d in row if pd.notna(d)
    ]

    # Count and take top 3
    top_diags = Counter(all_diags).most_common(3)
    top_labels = [d[0] for d in top_diags]

    # Map to burden dimensions using BURDEN_MAP keywords
    matched_burdens = []
    for burden_name, keywords in BURDEN_MAP.items():
        if any(any(kw.lower() in diag.lower() for kw in keywords)
               for diag in top_labels):
            matched_burdens.append(burden_name.replace('_',' ').title())

    CATEGORY_CONTEXT[category] = (
        ' / '.join(matched_burdens) if matched_burdens
        else top_labels[0] if top_labels
        else 'Review therapeutic class data'
    )

# Step 5 — Reverse: for each burden, find top co-occurring categories
for burden_name, keywords in BURDEN_MAP.items():
    # Months where this burden was present in diagnoses
    months_with_burden = paired[
        paired['diagnosis_disease_group'].apply(
            lambda diags: any(
                any(kw.lower() in d.lower() for kw in keywords)
                for d in diags if pd.notna(d)
            )
        )
    ]
    if months_with_burden.empty:
        BURDEN_SIGNALS[burden_name] = []
        continue

    # Flatten all categories dispensed in those months
    all_cats = [
        c for row in months_with_burden['correct_therapeutic_class']
        for c in row if pd.notna(c)
    ]

    # Top 4 categories
    top_cats = [c for c, _ in Counter(all_cats).most_common(4)]
    BURDEN_SIGNALS[burden_name] = top_cats

print("=== Data-derived CATEGORY_CONTEXT ===")
for cat, context in sorted(CATEGORY_CONTEXT.items()):
    print(f"  {cat:<35} {context}")

print("\n=== Data-derived BURDEN_SIGNALS ===")
for burden, cats in sorted(BURDEN_SIGNALS.items()):
    print(f"  {burden:<20} {cats}")
---
## 4. Build the Embu target profile

Branch #106 is a **new branch** — it has no data in the system.  
We can't query it. We can't compare it to anything.

Instead, we build the **Embu catchment profile** from what we already know:  
all existing Tendri branches are in Embu, so the average of their profiles  
is our best estimate of what the Embu patient population looks like.

This is the target vector we use in KNN — not Branch #106 itself.

# ── Step 1: Add external DHS features to facility_profiles (existing = 0) ────
external_feature_names = list(embu_external.keys())
for feat in external_feature_names:
    facility_profiles[feat] = 0

# ── FIX: Scale DHS % values to 0-1 to match ratio scale ─────────────────────
# NEW — without this, values like 75.0 dominate KNN distances
for feat in external_feature_names:
    val = embu_external[feat]
    if val is not None and val > 1:
        embu_external[feat] = val / 100

# ── Step 2: Add external + cat_ratio cols to ratio_cols ──────────────────────
ratio_cols = ratio_cols + external_feature_names
ratio_cols = list(dict.fromkeys(ratio_cols))  # remove any accidental duplicates

# ── FIX: Weighted mean — larger branches contribute more ─────────────────────
# CHANGED from facility_profiles[ratio_cols].mean()
weights = facility_profiles['avg_monthly_consultations'].fillna(1)
embu_target = facility_profiles[ratio_cols].apply(
    lambda col: (col * weights).sum() / weights.sum()
)

# ── Step 3: Override with actual Embu values for Branch 106 ──────────────────
for feat, val in embu_external.items():
    embu_target[feat] = val if val is not None else 0

# Set Branch 106 proximity from nearby_all results
embu_target['n_hospitals_25km']    = (nearby_all['amenity'] == 'hospital').sum()
embu_target['n_clinics_25km']      = nearby_all['amenity'].isin(['clinic','doctors']).sum()
embu_target['n_pharmacies_25km']   = (nearby_all['amenity'] == 'pharmacy').sum()
embu_target['n_facilities_25km']   = len(nearby_all)
embu_target['has_hospital_nearby'] = 1

# Set category mix for Branch 106 = mean of existing branches
for col in cat_ratio_cols:
    embu_target[col] = facility_profiles[col].mean()

print("Embu catchment profile — top disease burden dimensions:")
print()
burden_ratios = {
    k.replace('burden_', '').replace('_ratio', ''): round(v, 3)
    for k, v in embu_target.items()
    if 'burden_' in k and v > 0
}
for name, score in sorted(burden_ratios.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(score * 40)
    print(f"  {name:<20} {score:.3f}  {bar}")
facility_profiles.head()
---
## 5. Compute steady-state dispensing

We want to know what a mature Tendri branch dispenses per month on average —  
not what it dispensed in its early months when it was still finding its feet.

So we:
1. Rank each facility's months from 1 (first month) to N (latest month)
2. Exclude the first `RAMP_UP_MONTHS` months
3. Average what remains → this is the **steady-state** dispensing

We also separately pull Month 1 data for each branch, which we'll use  
in the next step to compute the penetration factor.

# Step 5a: Rank months per facility chronologically
# This tells us which month was the "1st month", "2nd month", etc. for each facility

disp_work = disp_df.copy()

# .groupby('facility_id')['months'] groups months by facility
# .rank(method='dense') assigns rank 1 to the earliest month, 2 to next, etc.
# 'dense' means no gaps in ranking (unlike 'average' which handles ties differently)
disp_work['month_rank'] = (
    disp_work
    .groupby('facility_id')['months']
    .rank(method='dense')
    .astype(int)
)

print("Month ranks assigned. Sample:")
disp_work[['facility_id', 'months', 'month_rank', 
            'correct_therapeutic_class', 'total_qty_dispensed']].head(10)

# Step 5b: Pull Month 1 data separately
# We need this to calculate the penetration factor later
# (how does Month 1 volume compare to steady-state?)

month1 = (
    disp_work[disp_work['month_rank'] == 1]    # filter to first month only
    .groupby(['facility_id', 'correct_therapeutic_class'])
    .agg(month1_qty=('total_qty_dispensed', 'sum'))
    .reset_index()
)

print(f"Month 1 data: {len(month1)} facility-category combinations")
month1.head(8)

# Step 5c: Compute steady-state (exclude ramp-up months)

steady_df = disp_work[disp_work['month_rank'] > RAMP_UP_MONTHS]

if steady_df.empty:
    print(f"WARNING: No data beyond month {RAMP_UP_MONTHS}.")
    print("Not enough history to exclude ramp-up months.")
    print("Using all months as steady state. Penetration factor will use default.")
    steady_df = disp_work.copy()
else:
    print(f"Using months {RAMP_UP_MONTHS+1}+ as steady state.")

# Average dispensing per facility per category across all steady-state months
steady_agg = (
    steady_df
    .groupby(['facility_id', 'correct_therapeutic_class'])
    .agg(
        avg_monthly_qty     = ('total_qty_dispensed', 'mean'),   # avg quantity per month
        avg_unique_patients = ('unique_patients',     'mean'),   # avg patients per month
        months_of_data      = ('months',              'nunique') # how many months went in
    )
    .reset_index()
)

print(f"Steady-state table: {len(steady_agg)} rows")
print(f"  {steady_agg['facility_id'].nunique()} facilities")
print(f"  {steady_agg['correct_therapeutic_class'].nunique()} product categories")
steady_agg.head(10)


---
## 6. Compute the penetration factor

A new branch doesn't immediately sell at full capacity.  
In Month 1, patients are still discovering the branch, staff are settling in,  
referral patterns haven't been established yet.

The **penetration factor** answers: *what fraction of mature monthly volume  
does a branch typically sell in its very first month?*

We compute it from existing branches that have both Month 1 data and  
steady-state data (4+ months of history).

> If we don't have enough history, we fall back to **0.35** —  
> meaning we assume Branch #106 will sell about 35% of its  
> eventual mature monthly volume in Month 1. This is conservative  
> and protects against dead stock.

# Join Month 1 qty to steady-state avg for the same facility-category
merged = steady_agg.merge(
    month1, 
    on=['facility_id', 'correct_therapeutic_class'], 
    how='inner'   # only keep rows that exist in BOTH tables
)

# Penetration ratio = Month 1 qty / steady-state avg
# If a branch sold 30 units in Month 1 and averages 100 units in steady state,
# the penetration ratio is 0.30 (30%)
merged['penetration_ratio'] = (
    merged['month1_qty'] / merged['avg_monthly_qty'].replace(0, np.nan)
)

# Use the median ratio across all facility-category combinations
# Median is more robust than mean — less affected by outliers
penetration_factor = merged['penetration_ratio'].dropna().median()
pf_is_default = False

# Sanity check: if factor > 2, something is wrong with the data
# (usually means we only have 1 month, so month1 == steady state)
if pd.isna(penetration_factor) or penetration_factor <= 0 or penetration_factor > 2:
    print("WARNING: Cannot compute a reliable penetration factor.")
    print(f"  Computed value: {penetration_factor}")
    print("  Reason: insufficient history (need 4+ months per facility)")
    print("  Falling back to 0.35 (conservative default)")
    penetration_factor = 0.35
    pf_is_default = True
else:
    print(f"Penetration factor: {penetration_factor:.2f}")
    print(f"  → Branch #106 is expected to sell {penetration_factor*100:.0f}% "
          f"of mature monthly volume in Month 1")
    pf_is_default = False

---
## 7. Rank facilities by similarity to the Embu profile (KNN)

We have the Embu target profile (what Branch #106's catchment looks like)  
and we have a profile for each existing Tendri branch.

**KNN (K-Nearest Neighbours)** finds which existing branches are most similar  
to the Embu target. We then weight their dispensing patterns accordingly —  
a branch whose patient mix closely matches the Embu profile gets more weight  
than a branch that's quite different.

**Steps:**
1. Standardise all feature values (so a large number like "500 consultations"  
   doesn't dominate over a small ratio like "0.17 hypertension rate")
2. Find the K branches with the smallest cosine distance to the Embu target
3. Weight them by inverse distance (closer = higher weight)
4. Check if the spread is meaningful — if all branches score almost the same,  
   equal weights are just as good as KNN weights

facility_profiles.info()
# Step 7a: Prepare the feature matrix
# Rows = facilities, Columns = ratio features

profiles_indexed = facility_profiles.set_index('facility_id')
X = profiles_indexed[ratio_cols].fillna(0)   # fill NaN with 0 (missing = no signal)

print(f"Feature matrix: {X.shape[0]} facilities × {X.shape[1]} features")
print()
print("Sample (first 5 features):")
X.iloc[:, :5].round(3)

# Step 7b: Standardise the features
# StandardScaler transforms each feature so it has mean=0 and std=1
# This prevents features with large absolute values from dominating the distance calculation
# e.g. "avg_monthly_consultations = 500" vs "burden_hypertension_ratio = 0.17"
# Without scaling, the consultation count would dominate every distance calculation

scaler = StandardScaler()

# fit_transform: learn the mean and std of each column, then apply the transformation
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

print("Features standardised (mean=0, std=1 per column)")
print()
print("Before scaling — consultation count range:")
print(f"  min: {X['avg_monthly_consultations'].min():.1f}, "
      f"  max: {X['avg_monthly_consultations'].max():.1f}")
print()
print("After scaling — consultation count range:")
print(f"  min: {X_scaled['avg_monthly_consultations'].min():.2f}, "
      f"  max: {X_scaled['avg_monthly_consultations'].max():.2f}")

# Step 7c: Standardise the Embu target vector using the SAME scaler
# Important: we use transform() not fit_transform() here
# fit_transform() would learn new mean/std from the target — wrong
# transform() applies the mean/std already learned from the facility data — correct

embu_vector = embu_target.reindex(ratio_cols).fillna(0).values.reshape(1, -1)
embu_scaled = scaler.transform(embu_vector)[0]

print(f"Embu target vector standardised ({len(embu_scaled)} dimensions)")

# Step 7d: Fit KNN and find most similar facilities

n_facilities = len(X_scaled)
k = min(KNN_K, n_facilities)   # can't have more neighbours than facilities

if n_facilities < 2:
    print("Only 1 facility in the data — KNN not applicable, using it with full weight")
    weight_map = {X_scaled.index[0]: 1.0}
    used_fallback = False
else:
    # NearestNeighbors finds the k closest points in the feature space
    # metric='cosine' measures the angle between vectors (good for ratio data)
    # Alternative: metric='euclidean' measures straight-line distance
    knn = NearestNeighbors(n_neighbors=k, metric='cosine')
    knn.fit(X_scaled)   # learn the positions of all facilities in feature space

    # kneighbors returns:
    #   distances — how far each neighbour is from the target (0 = identical, 1 = opposite)
    #   indices   — which row numbers in X_scaled those neighbours are
    distances, indices = knn.kneighbors([embu_scaled])
    distances = distances[0]   # unwrap from array-of-arrays
    indices   = indices[0]

    # Convert row indices back to facility IDs
    top_facility_ids = [X_scaled.index[i] for i in indices]

    print(f"Top {k} facilities most similar to the Embu profile:")
    print(f"{'Facility ID':<15} {'Cosine Distance':<20} {'Interpretation'}")
    print("-" * 55)
    for fid, dist in zip(top_facility_ids, distances):
        interp = "Very similar" if dist < 0.1 else "Similar" if dist < 0.3 else "Moderately similar"
        print(f"{fid:<15} {dist:<20.4f} {interp}")

# Step 7e: Assign weights — closer to Embu profile = higher weight
# Check first whether the spread is meaningful

if n_facilities >= 2:
    spread = distances.max() - distances.min()
    print(f"Similarity spread: {spread:.4f}  (threshold: {KNN_FALLBACK_THRESHOLD})")
    print()

    if spread < KNN_FALLBACK_THRESHOLD:
        # All facilities are almost equally similar to Embu
        # KNN weighting adds no value here — use equal weights instead
        print("Spread too narrow → all facilities equally representative of Embu.")
        print("Using equal weights (fallback).")
        weight_map    = {fid: 1.0 / n_facilities for fid in X_scaled.index}
        used_fallback = True
    else:
        # Inverse distance weighting:
        # weight = 1 / distance
        # A facility with distance 0.1 gets weight 10, one with distance 0.5 gets weight 2
        # We then normalise so all weights sum to 1
        weights = 1 / (distances + 1e-6)   # 1e-6 avoids division by zero
        weights = weights / weights.sum()   # normalise to sum=1

        weight_map    = dict(zip(top_facility_ids, weights))
        used_fallback = False

        print("Weights assigned:")
        for fid, w in weight_map.items():
            print(f"  facility_id={fid}  weight={w:.3f}  ({w*100:.1f}% influence)")


---
## 8. Predict opening stock for Branch #106

Now we combine everything:
1. Take the weighted average of steady-state dispensing from the most similar facilities
2. Multiply by the penetration factor to size the opening stock conservatively
3. Flag any categories where we're recommending more than 1.5x monthly velocity (dead stock risk)
4. Assign confidence levels based on data quality

steady_agg.info()
# Step 8a: Weighted average of steady-state dispensing

# Filter to only the facilities that have weights assigned
# (if fallback was used, this is all facilities)
df_pred = steady_agg[
    steady_agg['facility_id'].isin(weight_map.keys())
].copy()

# Map each facility's weight onto the rows
df_pred['weight'] = df_pred['facility_id'].map(weight_map).fillna(0)

# Weighted quantity = each facility's avg qty × its weight
# Summing these per category gives us the weighted average
df_pred['weighted_qty'] = df_pred['avg_monthly_qty'] * df_pred['weight']

print("Weighted quantities per facility per category:")
df_pred[['facility_id', 'correct_therapeutic_class', 
          'avg_monthly_qty', 'weight', 'weighted_qty']].head(10).round(3)

# Step 8b: Aggregate to category level

prediction = (
    df_pred
    .groupby('correct_therapeutic_class')
    .agg(
        # Sum of weighted qtys = weighted average across facilities
        predicted_steady_state    = ('weighted_qty',        'sum'),
        avg_unique_patients       = ('avg_unique_patients', 'mean'),
        n_facilities_contributing = ('facility_id',         'nunique')
    )
    .reset_index()
)

print("Predicted steady-state monthly volume per category:")
prediction[['correct_therapeutic_class', 'predicted_steady_state',
             'n_facilities_contributing']].round(1)

# Step 8c: Apply penetration factor → opening stock quantity

# Opening stock = what a mature branch would sell in a month × how much a new branch sells
# We round to whole units (you can't stock half a tablet box)
prediction['opening_stock_qty'] = (
    prediction['predicted_steady_state'] * penetration_factor
).round().astype(int)

print(f"Penetration factor applied: {penetration_factor:.2f}")
print()
prediction[['correct_therapeutic_class', 'predicted_steady_state', 
             'opening_stock_qty']].round(1)

# Step 8d: Dead stock risk flag

# If opening stock qty > DEAD_STOCK_MULTIPLIER × monthly velocity,
# it means you'd need more than 1.5 months to sell through just the opening stock
# That's a dead stock risk — you might be over-ordering
prediction['dead_stock_risk'] = (
    prediction['opening_stock_qty'] >
    prediction['predicted_steady_state'] * DEAD_STOCK_MULTIPLIER
)

risky = prediction[prediction['dead_stock_risk']]
if len(risky) > 0:
    print(f"⚠️  {len(risky)} categories flagged as dead stock risk:")
    print(risky[['correct_therapeutic_class', 'opening_stock_qty', 
                  'predicted_steady_state']].round(1).to_string(index=False))
else:
    print("✓ No dead stock risk flags")

# Step 8e: Confidence levels + diagnosis context + model reasoning
# ─────────────────────────────────────────────────────────────────
# Confidence logic (updated):
#   High   — 2+ facilities, real penetration factor, 6+ months of data, KNN not fallback
#   Medium — KNN fallback used OR fewer than 6 months of data
#   Low    — penetration factor is default 0.35 OR fewer than 2 facilities


# Merge average months of data per category
months_avg = (
    steady_agg.groupby('correct_therapeutic_class')['months_of_data']
    .mean().reset_index()
    .rename(columns={'months_of_data': 'months_avg'})
)
prediction = prediction.merge(months_avg, on='correct_therapeutic_class', how='left')

prediction['confidence'] = prediction.apply(
    lambda row: (
        'Low'    if pf_is_default or row['n_facilities_contributing'] < 2
        else 'High'   if not used_fallback and row.get('months_avg', 0) >= 6
        else 'Medium'
    ), axis=1
)

# Diagnosis context — which burden drives demand for each category
# CATEGORY_CONTEXT = {
#     'Oral Solid Forms':         'Mixed — review therapeutic class breakdown',
#     'Injectables':              'Communicable / Maternal / Cardiovascular',
#     'Oral Liquid Forms':        'Paediatric / Respiratory / GI',
#     'Eye Medications':          'Communicable / NCD',
#     'Topical Preparations':     'Dermatological / MSK',
#     'IV Fluids & Infusions':    'Communicable / Maternal / Surgical',
#     'Vaccines & Biologicals':   'Immunisation / Maternal',
#     'Ear & Nasal Preparations': 'Respiratory / ENT',
#     'Inhalation Products':      'Respiratory / Asthma',
#     'Wound Care':               'Surgical / Communicable',
#     'Infection Control':        'Facility operations',
# }

prediction['diagnosis_driver'] = (
    prediction['correct_therapeutic_class']
    .map(CATEGORY_CONTEXT)
    .fillna('Review therapeutic class data')
)

# Model reasoning — human-readable explanation per category
# BURDEN_SIGNALS = {
#     'malaria':      ['Injectables', 'Oral Solid Forms', 'Oral Liquid Forms'],
#     'maternal':     ['Injectables', 'IV Fluids & Infusions', 'Oral Solid Forms'],
#     'hypertension': ['Oral Solid Forms'],
#     'diabetes':     ['Oral Solid Forms'],
#     'respiratory':  ['Oral Liquid Forms', 'Inhalation Products', 'Oral Solid Forms'],
#     'communicable': ['Injectables', 'Oral Solid Forms', 'Oral Liquid Forms'],
# }

top_burden = sorted(
    [(k.replace('burden_','').replace('_ratio',''), v)
     for k, v in embu_target.items() if 'burden_' in k and v > 0],
    key=lambda x: x[1], reverse=True
)[:3]

def build_reasoning(row):
    reasons = []
    category = row['correct_therapeutic_class']

    # Signal 1 — Proximity
    drivers = category_proximity_driver.get(category, [])
    if drivers:
        n = len(drivers)
        nearest = drivers[0]
        reasons.append(f"{n} nearby {'facility' if n==1 else 'facilities'} — {nearest}")

    # Signal 2 — Disease burden
    for burden_name, burden_score in top_burden:
        if category in BURDEN_SIGNALS.get(burden_name, []) and burden_score > 0.1:
            reasons.append(f"High {burden_name} catchment burden ({burden_score:.0%})")
            break

    # Signal 3 — Age mix
    if embu_target.get('pct_chronic_age', 0) > 0.25 and category == 'Oral Solid Forms':
        reasons.append(f"High adult 45+ population ({embu_target['pct_chronic_age']:.0%})")
    if embu_target.get('pct_paediatric', 0) > 0.20 and category in ['Oral Liquid Forms', 'Vaccines & Biologicals']:
        reasons.append(f"High paediatric population ({embu_target['pct_paediatric']:.0%})")

    # Signal 4 — Comparable branch
    if not used_fallback and row['n_facilities_contributing'] >= 2:
        top_fac = max(weight_map, key=weight_map.get)
        reasons.append(f"Comparable to Branch {top_fac} Month 1 pattern")

    # Signal 5 — Conservative flag
    if row['dead_stock_risk']:
        return "Low cross-branch demand — order conservatively"
    if row['confidence'] == 'Low':
        reasons.append("Insufficient history — conservative estimate")

    if not reasons:
        return "Based on Embu catchment profile"
    return ' — '.join(reasons[:2])

prediction['model_reasoning'] = prediction.apply(build_reasoning, axis=1)

print("Confidence levels, diagnosis context and model reasoning added")
print(prediction[['correct_therapeutic_class','confidence',
                   'months_avg','model_reasoning']].to_string())
##  9.Use Bayesian Confidence Intervals
# Map from new_category_name to GT category index key
# These must match the 'category' values in embu_trends_category_index
GT_CATEGORY_MAP = {
    "Beauty Products":         "beauty",
    "Vitamins & Supplements":  "supplements",
    "Body Building":           "bodybuilding",
}
 
def compute_gt_weights(category_name, steady_df_subset):
    """
    Returns a weight array aligned to steady_df_subset rows.
    For GT-linked categories: weight = GT index for that month (floored at 0.15).
    For all other categories: uniform weights (all 1.0).
    """
    if embu_trends_category_index.empty:
        return np.ones(len(steady_df_subset))
 
    gt_key = GT_CATEGORY_MAP.get(category_name)
    if gt_key is None:
        return np.ones(len(steady_df_subset))
 
    # Get GT index for this category
    gt_cat = embu_trends_category_index[
        embu_trends_category_index["category"] == gt_key
    ][["month", "category_index"]].copy()
 
    if gt_cat.empty:
        return np.ones(len(steady_df_subset))
 
    gt_cat = gt_cat.set_index("month")["category_index"]
 
    # Align weights to steady_df months
    weights = []
    for _, row in steady_df_subset.iterrows():
        month = pd.to_datetime(row["months"]).to_period("M").to_timestamp()
        w = gt_cat.get(month, None)
        if w is None:
            # Try nearest month fallback
            w = gt_cat.iloc[-1] if len(gt_cat) > 0 else 1.0
        weights.append(max(float(w), 0.15))  # floor at 15%
 
    return np.array(weights)
 
 
def compute_credible_intervals_gt(prediction, steady_df, penetration_factor,
                                   confidence_level=0.90):
    """
    GT-weighted Bayesian credible intervals.
    Drop-in replacement for the original Bayesian CI block in Step 9.
 
    For new categories (Beauty/Supplements/BB):
      - Weights observations by GT monthly index
      - Fits a weighted normal distribution
      - Computes CI from weighted mean/std
    For pharma categories:
      - Identical to the original unweighted logic
    """
    credible_intervals = []
 
    for _, row in prediction.iterrows():
        category = row["correct_therapeutic_class"]
        point    = row["opening_stock_qty"]
 
        # Get observations from steady_df for this category
        obs_df = steady_df[
            steady_df["correct_therapeutic_class"] == category
        ][["months", "total_qty_dispensed"]].dropna()
 
        obs = obs_df["total_qty_dispensed"].values
 
        if len(obs) < 2:
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            credible_intervals.append({
                "correct_therapeutic_class": category,
                "ci_lower":       lo,
                "ci_upper":       hi,
                "ci_method":      "fallback ±40%",
                "n_observations": len(obs),
            })
            continue
 
        # ── GT weighting for new categories ──────────────────────────────────
        weights = compute_gt_weights(category, obs_df)
 
        # Weighted mean and std
        w_sum    = weights.sum()
        w_mean   = (weights * obs).sum() / w_sum
        w_var    = (weights * (obs - w_mean) ** 2).sum() / w_sum
        w_std    = np.sqrt(w_var)
 
        # Guard against degenerate cases
        if w_std <= 0 or np.isnan(w_std) or np.isnan(w_mean):
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            credible_intervals.append({
                "correct_therapeutic_class": category,
                "ci_lower":       lo,
                "ci_upper":       hi,
                "ci_method":      "fallback ±40% (zero variance)",
                "n_observations": len(obs),
            })
            continue
 
        # Apply penetration factor
        mu_m1  = w_mean * penetration_factor
        std_m1 = w_std  * penetration_factor
 
        alpha  = (1 - confidence_level) / 2
        lo_f   = stats.norm.ppf(alpha,     loc=mu_m1, scale=std_m1)
        hi_f   = stats.norm.ppf(1 - alpha, loc=mu_m1, scale=std_m1)
 
        if np.isnan(lo_f) or np.isnan(hi_f):
            lo = max(0, round(point * 0.60))
            hi = round(point * 1.40)
            method = "fallback ±40% (ppf NaN)"
        else:
            lo = max(0, round(lo_f))
            hi = round(hi_f)
            gt_key = GT_CATEGORY_MAP.get(category)
            method = (
                f"GT-weighted normal (n={len(obs)})" if gt_key
                else f"normal fit (n={len(obs)} months)"
            )
 
        credible_intervals.append({
            "correct_therapeutic_class": category,
            "ci_lower":       lo,
            "ci_upper":       hi,
            "ci_method":      method,
            "n_observations": len(obs),
        })
 
    return pd.DataFrame(credible_intervals)
 
 
print("\n✓ compute_credible_intervals_gt() defined")
print("  Use this in Step 9 instead of the original CI block:")
print("  ci_df = compute_credible_intervals_gt(prediction, steady_df, penetration_factor)")
 
def external_multiplier(product_name, embu_target):
    """
    Adjusts product share within its category based on external signals.
    Higher multiplier = boost this product's share in the opening stock allocation.
 
    Sources:
      Original:  hospital proximity, malaria/maternal/paediatric/NCD burden
      New:       beauty/BB/supplement demand proximity, GT momentum per category
    """
    n = str(product_name).lower()
    multiplier = 1.0
 
    # ── ORIGINAL SIGNALS (pharma — unchanged) ────────────────────────────────
 
    # Hospital proximity boosts injectables and IV products
    has_hospital = embu_target.get("has_hospital_nearby", 0)
    if has_hospital and any(k in n for k in ["injection","injectable","iv ","infusion","vial","ampoule"]):
        multiplier *= 1.3
 
    # Malaria burden boosts ACTs and antimalarials
    malaria_burden = embu_target.get("burden_malaria_ratio", 0)
    if malaria_burden > 0.1 and any(k in n for k in ["artemether","coartem","lumefantrine","quinine","fansidar","sp "]):
        multiplier *= (1 + malaria_burden * 2)
 
    # Maternal burden boosts ANC products
    maternal_burden = embu_target.get("burden_maternal_ratio", 0)
    antenatal_pct   = embu_target.get("antenatal_4plus_visits_pct", 0)
    if (maternal_burden > 0.05 or antenatal_pct > 0.5) and \
       any(k in n for k in ["folic","ferrous","antenatal","oxytocin","magnesium","prenatal"]):
        multiplier *= 1.25
 
    # Paediatric mix boosts liquid formulations
    pct_paediatric = embu_target.get("pct_paediatric", 0)
    if pct_paediatric > 0.2 and any(k in n for k in ["syrup","suspension","paediatric","paed","infant","drops","125mg","250mg/5"]):
        multiplier *= (1 + pct_paediatric)
 
    # NCD burden boosts chronic disease products
    hypert_burden = embu_target.get("burden_hypertension_ratio", 0)
    diab_burden   = embu_target.get("burden_diabetes_ratio", 0)
    if hypert_burden > 0.05 and any(k in n for k in ["amlodipine","atenolol","losartan","lisinopril","ramipril","nifedipine"]):
        multiplier *= (1 + hypert_burden * 1.5)
    if diab_burden > 0.05 and any(k in n for k in ["metformin","glibenclamide","insulin","glucophage"]):
        multiplier *= (1 + diab_burden * 1.5)
 
    # ── NEW SIGNALS — BEAUTY PRODUCTS ────────────────────────────────────────
 
    # Is this a beauty product? Check via product name keywords
    _is_beauty = any(k in n for k in [
        "serum","moisturiser","moisturizer","sunscreen","spf","cleanser","toner",
        "mask","scrub","lotion","cream","gel","lip balm","lip gloss","body lotion",
        "body butter","face wash","micellar","retinol","vitamin c","niacinamide",
        "hyaluronic","collagen","brightening","whitening","glow","anti-aging",
        "anti-ageing","eye cream","body wash","shower gel","body mist","perfume",
        "cologne","foundation","concealer","mascara","eyeliner","blush","highlighter"
    ])
 
    if _is_beauty:
        # Proximity: nearby beauty salons and shops drive demand
        beauty_prox = embu_target.get("beauty_demand_proximity", 0)
        if beauty_prox > 0.3:
            multiplier *= (1 + beauty_prox * 0.5)   # up to +50% boost
 
        # GT momentum: if beauty category is trending upward, boost
        beauty_momentum = embu_target.get("beauty_momentum", 0)
        if beauty_momentum > 0.6:                    # accelerating demand
            multiplier *= (1 + (beauty_momentum - 0.5) * 0.4)
 
        # GT recent signal: high recent interest boosts established products
        beauty_recent = embu_target.get("beauty_recent4w", 0)
        if beauty_recent > 0.7:
            multiplier *= 1.15
 
        # Women 15-49 share — core beauty buyer
        beauty_target_pct = embu_target.get("beauty_target_pct", 0)
        if beauty_target_pct > 0.25:
            multiplier *= (1 + (beauty_target_pct - 0.25) * 0.8)
 
    # ── NEW SIGNALS — BODY BUILDING ───────────────────────────────────────────
 
    _is_bb = any(k in n for k in [
        "whey","protein","creatine","pre-workout","preworkout","amino","bcaa",
        "mass gainer","isolate","concentrate","casein","glutamine","beta-alanine",
        "l-carnitine","caffeine","shaker","gym","workout","muscle","bulk",
        "lean mass","fat burner","thermogenic"
    ])
 
    if _is_bb:
        # Proximity: nearby gyms and BB shops are the primary demand driver
        bb_prox = embu_target.get("bb_demand_proximity", 0)
        if bb_prox > 0.2:
            multiplier *= (1 + bb_prox * 0.8)       # gyms nearby = strong signal
 
        # GT momentum for bodybuilding category
        bb_momentum = embu_target.get("bodybuilding_momentum", 0)
        if bb_momentum > 0.5:
            multiplier *= (1 + (bb_momentum - 0.5) * 0.5)
 
        # Men 15-34 share — core BB buyer
        bb_target_pct = embu_target.get("bb_target_pct", 0)
        if bb_target_pct > 0.15:
            multiplier *= (1 + (bb_target_pct - 0.15) * 1.0)
 
    # ── NEW SIGNALS — VITAMINS & SUPPLEMENTS ──────────────────────────────────
 
    _is_supplement = any(k in n for k in [
        "vitamin","supplement","zinc","magnesium","calcium","iron","omega",
        "fish oil","multivitamin","probiotic","prebiotic","collagen","biotin",
        "folic acid","vitamin d","vitamin c","vitamin b","immune","antioxidant",
        "melatonin","turmeric","curcumin","glucosamine","chondroitin","coq10",
        "evening primrose","spirulina","moringa","ashwagandha","echinacea"
    ])
 
    if _is_supplement and not _is_bb:   # BB already handled above
        # Proximity: supplement stores and gyms nearby
        supp_prox = embu_target.get("supplement_demand_proximity", 0)
        if supp_prox > 0.2:
            multiplier *= (1 + supp_prox * 0.5)
 
        # GT momentum for supplements
        supp_momentum = embu_target.get("supplements_momentum", 0)
        if supp_momentum > 0.5:
            multiplier *= (1 + (supp_momentum - 0.5) * 0.4)
 
        # Adults 25-64 share — core supplement buyer
        supp_target_pct = embu_target.get("supplements_target_pct", 0)
        if supp_target_pct > 0.40:
            multiplier *= (1 + (supp_target_pct - 0.40) * 0.6)
 
        # Aged 65+ — bone/joint/heart supplements specifically
        pct_aged = embu_target.get("pct_aged_65_plus", 0)
        if pct_aged > 0.05 and any(k in n for k in ["calcium","vitamin d","glucosamine","chondroitin","coq10","omega"]):
            multiplier *= (1 + pct_aged * 1.5)
 
    return round(multiplier, 3)
 
 
print("\n✓ external_multiplier() redefined with GT + proximity signals")
print("  New signals added:")
print("    Beauty:      beauty_demand_proximity, beauty_momentum, beauty_recent4w, beauty_target_pct")
print("    BB:          bb_demand_proximity, bodybuilding_momentum, bb_target_pct")
print("    Supplements: supplement_demand_proximity, supplements_momentum, supplements_target_pct, pct_aged_65_plus")
print("    Pharma:      unchanged")
 
---
## 10. Final output — Branch #106 Opening Stock List

# Final formatted output table

output = prediction[[
    'correct_therapeutic_class',
    'opening_stock_qty',
    'stock_range',
    'predicted_steady_state',
    'confidence',
    'dead_stock_risk',
    'model_reasoning',
]].sort_values('opening_stock_qty', ascending=False).copy()

output.columns = [
    'Category',
    'Opening Stock Qty',
    'Stock Range (80% CI)',
    'Predicted Monthly (Steady State)',
    'Confidence',
    'Dead Stock Risk',
    'Model Reasoning',
]

print("=" * 70)
print("BRANCH #106 — RECOMMENDED OPENING STOCK")
method = "KNN-weighted" if not used_fallback else "equal-weight average"
print(f"Method:             {method}")
print(f"Penetration factor: {penetration_factor:.2f}"
      + (" (default)" if pf_is_default else " (data-derived)"))
print(f"Facilities used:    {len(weight_map)}")
print("=" * 70)

output

# ── Step 9b: Product-level allocation ────────────────────────────────────────
# Logic:
# 1. Category totals come from the KNN model (reliable, data-rich)
# 2. Within each category, products are ranked by their actual dispensing share
#    across similar branches (buying patterns)
# 3. Shares are then adjusted by external signals — proximity and DHS burden
# 4. Adjusted shares are normalised and applied to the category total

# ── A. Product dispensing share per category from similar branches ────────────
# Use steady_df (ramp-up excluded) filtered to weighted branches only
prod_steady = (
    disp_work[
        (disp_work["month_rank"] > RAMP_UP_MONTHS) &
        (disp_work["facility_id"].isin(weight_map))
    ]
    .copy()
)
prod_steady["weight"] = prod_steady["facility_id"].map(weight_map)
prod_steady["weighted_qty"] = prod_steady["total_qty_dispensed"] * prod_steady["weight"]

# Need product_name — join from fact_dispensing
prod_names = (fact_dispensing[["product_id","product_name"]]
              .dropna()
              .drop_duplicates("product_id"))

prod_agg = (prod_steady
            .merge(prod_names, on="product_id", how="left")
            .groupby(["correct_therapeutic_class","product_id","product_name"])
            .agg(weighted_qty=("weighted_qty","sum"))
            .reset_index())

# Drop products with no name
prod_agg = prod_agg[prod_agg["product_name"].notna()]

# Share within each category
cat_total_qty = prod_agg.groupby("correct_therapeutic_class")["weighted_qty"].transform("sum")
prod_agg["base_share"] = prod_agg["weighted_qty"] / cat_total_qty.replace(0, 1)

# ── B. External adjustment multipliers ───────────────────────────────────────
# Map product to clinical signal using product name keywords
# Higher multiplier = boost this product's share in the allocation

def external_multiplier(product_name, embu_target):
    """
    Adjust product share based on:
    - Proximity: hospital nearby boosts injectables and IV products
    - DHS burden: high malaria boosts ACTs, high maternal boosts ANC products
    - Age mix: high paediatric boosts liquid forms and paediatric formulations
    """
    n = str(product_name).lower()
    multiplier = 1.0

    # Proximity signal — hospital nearby boosts injectables
    has_hospital = embu_target.get("has_hospital_nearby", 0)
    if has_hospital and any(k in n for k in ["injection","injectable","iv ","infusion","vial","ampoule"]):
        multiplier *= 1.3

    # Malaria burden — boost ACTs and antimalarials
    malaria_burden = embu_target.get("burden_malaria_ratio", 0)
    if malaria_burden > 0.1 and any(k in n for k in ["artemether","coartem","lumefantrine","quinine","fansidar","sp "]):
        multiplier *= (1 + malaria_burden * 2)

    # Maternal burden — boost ANC and reproductive health products
    maternal_burden = embu_target.get("burden_maternal_ratio", 0)
    antenatal_pct   = embu_target.get("antenatal_4plus_visits_pct", 0)
    if (maternal_burden > 0.05 or antenatal_pct > 0.5) and \
       any(k in n for k in ["folic","ferrous","antenatal","oxytocin","magnesium","prenatal"]):
        multiplier *= 1.25

    # Paediatric mix — boost liquid formulations and paediatric products
    pct_paediatric = embu_target.get("pct_paediatric", 0)
    if pct_paediatric > 0.2 and any(k in n for k in ["syrup","suspension","paediatric","paed","infant","drops","125mg","250mg/5"]):
        multiplier *= (1 + pct_paediatric)

    # NCD burden — boost chronic disease products
    hypert_burden = embu_target.get("burden_hypertension_ratio", 0)
    diab_burden   = embu_target.get("burden_diabetes_ratio", 0)
    if hypert_burden > 0.05 and any(k in n for k in ["amlodipine","atenolol","losartan","lisinopril","ramipril","nifedipine"]):
        multiplier *= (1 + hypert_burden * 1.5)
    if diab_burden > 0.05 and any(k in n for k in ["metformin","glibenclamide","insulin","glucophage"]):
        multiplier *= (1 + diab_burden * 1.5)

    return round(multiplier, 3)

prod_agg["ext_multiplier"] = prod_agg["product_name"].apply(
    lambda n: external_multiplier(n, embu_target)
)

# ── C. Adjusted share ─────────────────────────────────────────────────────────
prod_agg["adjusted_weight"] = prod_agg["base_share"] * prod_agg["ext_multiplier"]

# Re-normalise adjusted weights within each category so they sum to 1
adj_cat_total = prod_agg.groupby("correct_therapeutic_class")["adjusted_weight"].transform("sum")
prod_agg["adjusted_share"] = prod_agg["adjusted_weight"] / adj_cat_total.replace(0, 1)

# ── D. Apply category totals to get product-level opening stock ────────────────
# Map category opening stock qty from output
cat_opening = output.set_index("Category")["Opening Stock Qty"]

prod_agg["category_opening_qty"] = prod_agg["correct_therapeutic_class"].map(cat_opening)

prod_agg["product_opening_qty"] = (
    prod_agg["adjusted_share"] * prod_agg["category_opening_qty"]
).round().astype(int)

# Drop zeros — no point recommending 0 units
prod_agg = prod_agg[prod_agg["product_opening_qty"] > 0]

# ── E. Add confidence + dead stock risk inherited from category ───────────────
cat_confidence = output.set_index("Category")["Confidence"]
cat_dsr        = output.set_index("Category")["Dead Stock Risk"]
cat_range      = output.set_index("Category").get("Stock Range (80% CI)", None)

prod_agg["confidence"]      = prod_agg["correct_therapeutic_class"].map(cat_confidence)
prod_agg["dead_stock_risk"] = prod_agg["correct_therapeutic_class"].map(cat_dsr)

# ── F. Final product-level output ─────────────────────────────────────────────
output_products = (prod_agg[[
    "product_name",
    "correct_therapeutic_class",
    "product_opening_qty",
    "base_share",
    "ext_multiplier",
    "confidence",
    "dead_stock_risk",
]].sort_values(["correct_therapeutic_class","product_opening_qty"], ascending=[True,False])
  .reset_index(drop=True))

output_products.columns = [
    "Product",
    "Category",
    "Opening Stock Qty",
    "Historical Share",
    "External Adjustment",
    "Confidence",
    "Dead Stock Risk",
]

output_products["Historical Share"]    = (output_products["Historical Share"] * 100).round(1).astype(str) + "%"
output_products["External Adjustment"] = output_products["External Adjustment"].apply(
    lambda x: f"↑ {x:.2f}x" if x > 1 else f"→ {x:.2f}x"
)
output_products["Dead Stock Risk"] = output_products["Dead Stock Risk"].map(
    {True:"⚠️ Yes", False:"✓ No"}
).fillna(output_products["Dead Stock Risk"])

print(f"Product-level output: {len(output_products)} products across {output_products['Category'].nunique()} categories")
print("\nSample — top 20 by opening stock qty:")
print(output_products.nlargest(20, "Opening Stock Qty")[
    ["Product","Category","Opening Stock Qty","Historical Share","External Adjustment","Confidence"]
].to_string(index=False))

# ── G. Save both outputs to pickle ────────────────────────────────────────────


pickle.dump({
    "disp":         fact_dispensing,
    "inv":          fact_inventory_snapshot,
    "pat":          dim_patient_profile,
    "diag_df":      monthly_diagnoses_aggregate,
    "disp_df":      monthly_dispensing_aggregate,
    "pred":         output,           # category-level (existing dashboard)
    "pred_products": output_products, # product-level (new)
}, open("data_export.pkl", "wb"))

print("\nSaved → data_export.pkl")

# Optional CSV export
output_products.to_csv("branch_106_opening_stock_products.csv", index=False)
print("Saved → branch_106_opening_stock_products.csv")
# # Save to CSV
# output.to_csv('branch_106_opening_stock_v2.csv', index=False)
# print("Saved → branch_106_opening_stock_v2.csv")

# #fact_dispensing.to_csv('fact_dispensing.csv', index=False)
# #fact_inventory_snapshot.to_csv('fact_inventory_snapshot.csv', index=False)
# #dim_patient_profile.to_csv('dim_patient_profile.csv', index=False)
# monthly_diagnoses_aggregate.to_csv('monthly_diagnoses_aggregate.csv', index=False)
# monthly_dispensing_aggregate.to_csv('monthly_dispensing_aggregate.csv', index=False)
# import pickle

pickle.dump({
    "disp"          : fact_dispensing,
    "inv"           : fact_inventory_snapshot,
    "pat"           : dim_patient_profile,
    "diag_df"       : monthly_diagnoses_aggregate,
    "disp_df"       : monthly_dispensing_aggregate,
    "pred"          : output,
    "pred_products" : output_products,
    "product_intel" : product_intel,   # NEW — profitability + generic intelligence
}, open("data_export.pkl", "wb"))
 
print("Pickle updated with simulated data.")
print(f"  Keys: {['disp','inv','pat','diag_df','disp_df','pred','pred_products','product_intel']}")
print(output.columns.tolist())

prod_agg["ext_multiplier"] = prod_agg["product_name"].apply(
    lambda n: external_multiplier(n, embu_target)
)

# ── C. Adjusted share ─────────────────────────────────────────────────────────
prod_agg["adjusted_weight"] = prod_agg["base_share"] * prod_agg["ext_multiplier"]

# Re-normalise adjusted weights within each category so they sum to 1
adj_cat_total = prod_agg.groupby("correct_therapeutic_class")["adjusted_weight"].transform("sum")
prod_agg["adjusted_share"] = prod_agg["adjusted_weight"] / adj_cat_total.replace(0, 1)

# ── D. Apply category totals to get product-level opening stock ────────────────
# Map category opening stock qty from output
cat_opening = output.set_index("Category")["Opening Stock Qty"]

prod_agg["category_opening_qty"] = prod_agg["correct_therapeutic_class"].map(cat_opening)

prod_agg["product_opening_qty"] = (
    prod_agg["adjusted_share"] * prod_agg["category_opening_qty"]
).round().astype(int)

# Drop zeros — no point recommending 0 units
prod_agg = prod_agg[prod_agg["product_opening_qty"] > 0]

# ── E. Add confidence + dead stock risk inherited from category ───────────────
cat_confidence = output.set_index("Category")["Confidence"]
cat_dsr        = output.set_index("Category")["Dead Stock Risk"]
cat_range      = output.set_index("Category").get("Stock Range (80% CI)", None)

prod_agg["confidence"]      = prod_agg["correct_therapeutic_class"].map(cat_confidence)
prod_agg["dead_stock_risk"] = prod_agg["correct_therapeutic_class"].map(cat_dsr)

# ── F. Final product-level output ─────────────────────────────────────────────
output_products = (prod_agg[[
    "product_name",
    "correct_therapeutic_class",
    "product_opening_qty",
    "base_share",
    "ext_multiplier",
    "confidence",
    "dead_stock_risk",
]].sort_values(["correct_therapeutic_class","product_opening_qty"], ascending=[True,False])
  .reset_index(drop=True))

output_products.columns = [
    "Product",
    "Category",
    "Opening Stock Qty",
    "Historical Share",
    "External Adjustment",
    "Confidence",
    "Dead Stock Risk",
]

output_products["Historical Share"]    = (output_products["Historical Share"] * 100).round(1).astype(str) + "%"
output_products["External Adjustment"] = output_products["External Adjustment"].apply(
    lambda x: f"↑ {x:.2f}x" if x > 1 else f"→ {x:.2f}x"
)
output_products["Dead Stock Risk"] = output_products["Dead Stock Risk"].map(
    {True:"⚠️ Yes", False:"✓ No"}
).fillna(output_products["Dead Stock Risk"])

print(f"Product-level output: {len(output_products)} products across {output_products['Category'].nunique()} categories")
print("\nSample — top 20 by opening stock qty:")
print(output_products.nlargest(20, "Opening Stock Qty")[
    ["Product","Category","Opening Stock Qty","Historical Share","External Adjustment","Confidence"]
].to_string(index=False))

# ── G. Save both outputs to pickle ────────────────────────────────────────────


pickle.dump({
    "disp":         fact_dispensing,
    "inv":          fact_inventory_snapshot,
    "pat":          dim_patient_profile,
    "diag_df":      monthly_diagnoses_aggregate,
    "disp_df":      monthly_dispensing_aggregate,
    "pred":         output,           # category-level (existing dashboard)
    "pred_products": output_products, # product-level (new)
}, open("data_export.pkl", "wb"))

print("\nSaved → data_export.pkl")

# Optional CSV export
output_products.to_csv("branch_106_opening_stock_products.csv", index=False)
print("Saved → branch_106_opening_stock_products.csv")

Product-level output: 1002 products across 324 categories

Sample — top 20 by opening stock qty:
                    Product                             Category  Opening Stock Qty Historical Share External Adjustment Confidence
  LASIX 40MG BP(FUROSEMIDE}                      Diuretic — loop               1833            97.4%             → 1.00x       High
  METHYLDOPA TABS 250MG-(G)        Antihypertensive — injectable               1438           100.0%             → 1.00x       High
  FLUCLOXACILLIN 500MG  B/P Antibiotic — penicillinase-resistant                876            77.2%             → 1.00x       High
      PREDNISOLONE 5 MG B/P                Corticosteroid — oral                749            97.8%             → 1.00x       High
   NIFEDIPINE CALCIGARD B/P               Antihypertensive — CCB                732            46.4%             ↑ 1.12x       High
         LOSARTAN H NUSAR H               Antihypertensive — ARB                672            49.9%           

In [ ]:
# # Save to CSV
# output.to_csv('branch_106_opening_stock_v2.csv', index=False)
# print("Saved → branch_106_opening_stock_v2.csv")

# #fact_dispensing.to_csv('fact_dispensing.csv', index=False)
# #fact_inventory_snapshot.to_csv('fact_inventory_snapshot.csv', index=False)
# #dim_patient_profile.to_csv('dim_patient_profile.csv', index=False)
# monthly_diagnoses_aggregate.to_csv('monthly_diagnoses_aggregate.csv', index=False)
# monthly_dispensing_aggregate.to_csv('monthly_dispensing_aggregate.csv', index=False)

In [ ]:
# import pickle

pickle.dump({
    "disp"          : fact_dispensing,
    "inv"           : fact_inventory_snapshot,
    "pat"           : dim_patient_profile,
    "diag_df"       : monthly_diagnoses_aggregate,
    "disp_df"       : monthly_dispensing_aggregate,
    "pred"          : output,
    "pred_products" : output_products,
    "product_intel" : product_intel,   # NEW — profitability + generic intelligence
}, open("data_export.pkl", "wb"))
 
print("Pickle updated with simulated data.")
print(f"  Keys: {['disp','inv','pat','diag_df','disp_df','pred','pred_products','product_intel']}")

Saved → data_export.pkl


In [ ]:
print(output.columns.tolist())

['Category', 'Opening Stock Qty', 'Predicted Monthly (Steady State)', 'Confidence', 'Dead Stock Risk', 'Diagnosis Driver']
